# ✈️ OpenSky Historical Flight Density & Noise Impact Heatmap Analysis
Analyze and visualize real flight trajectories, 4-tier altitude classes, and perceived ground noise footprint from OpenSky Network ADS-B data stored in local bounding box caches.

### Key Features:
- **🗄️ 100% Cache-Driven Analysis & Enclosing Bounding Box Auto-Discovery**: Automatically finds and reuses cache files covering your selected region or sub-regions.
- **⏱️ Date & Time Range Selection & Panning**: Interactive multi-day slider displaying complete UTC dates (`YYYY-MM-DD HH:MM:SS UTC`) with `1h`, `3h`, `6h`, `8h`, `12h`, `24h`, `All` quick presets and sliding time window controls.
- **🛩️ Multi-Select Aircraft Type Groups & Cascading Filters**: Multi-select any combination of operational groups (**Commercial Heavy Jet**, **Commercial Medium Jet / Airliner**, **Business Jet**, **Regional Turboprop**, **Light Aircraft / Leisure GA**, **Helicopter / Rotorcraft**, **Military / Special Ops**, **Glider**) with dynamic cross-filtering to available airframe models.
- **🔊 Ground Noise Level Modeling & Filtering**: Estimates perceived A-weighted sound level $L_\text{ground}$ (in $\text{dB(A)}$) directly beneath each flight path using standard ECAC Doc 29 / FAA AEDT acoustic physics (accounting for altitude divergence, atmospheric absorption, vertical climb/descent thrust, and aircraft airframe/engine type).
- **🎨 6-Tier Altitude Classification**:
  - 🔴 **0 – 500 m** (Red, $0 < \text{alt} \le 500\text{ m}$) &mdash; Surface ops & initial climb
  - 🟧 **500 – 1,000 m** (Orange, $500 < \text{alt} \le 1000\text{ m}$) &mdash; Low climb & final approach alignment
  - 🟠 **1,000 – 2,000 m** (Amber, $1000 < \text{alt} \le 2000\text{ m}$) &mdash; Lower terminal maneuvering airspace
  - 🟡 **2,000 – 3,000 m** (Yellow, $2000 < \text{alt} \le 3000\text{ m}$) &mdash; Intermediate TMA & entry corridors
  - 🩵 **3,000 – 5,000 m** (Teal, $3000 < \text{alt} \le 5000\text{ m}$) &mdash; Regional climb & descent transitions
  - 🟢 **5,000+ m** (Green, $\text{alt} > 5000\text{ m}$) &mdash; En-route high cruising airways
- **🗺️ Multi-Background Map**: `CartoDB Positron` by default, switchable to Dark Matter, Satellite, OSM, and Topo.


In [ ]:
import os
import re
import gc
import math
import json
import time
import requests
from datetime import datetime, timedelta, timezone
from typing import List, Dict, Any, Tuple, Optional, Set

import numpy as np
import pandas as pd
import folium
from folium import LayerControl
from folium.plugins import HeatMap, HeatMapWithTime, Fullscreen, MousePosition
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from ipyleaflet import Map, basemaps, DrawControl, Rectangle, WidgetControl
import ipywidgets as widgets
from ipywidgets import HTML, VBox, HBox, Output, Button, Checkbox, Dropdown, SelectMultiple, SelectionRangeSlider, IntSlider, Layout, Label, DatetimePicker
from IPython.display import display, IFrame

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except ImportError:
    print("⚠️ PyArrow is required for reading parquet cache files.")

WORKSPACE_DIR = os.path.join(os.getcwd(), "flight_heatmap") if os.path.basename(os.getcwd()) != "flight_heatmap" and os.path.exists(os.path.join(os.getcwd(), "flight_heatmap")) else os.getcwd()
CACHE_DIR = os.path.join(WORKSPACE_DIR, "flight_data_cache")
os.makedirs(CACHE_DIR, exist_ok=True)

def ensure_aircraft_database(cache_dir: str = CACHE_DIR) -> pd.DataFrame:
    """Ensures the static OpenSky aircraft metadata database is present and indexed."""
    parquet_path = os.path.join(cache_dir, "aircraftDatabase.parquet")
    csv_path = os.path.join(cache_dir, "aircraftDatabase.csv")
    cols = ['icao24', 'registration', 'model', 'typecode', 'operator', 'categoryDescription']
    if os.path.exists(parquet_path):
        schema_names = pq.read_schema(parquet_path).names
        use_cols = [c for c in cols if c in schema_names]
        table = pq.read_table(parquet_path, columns=use_cols)
        df_meta = table.to_pandas()
        return df_meta
        
    if not os.path.exists(csv_path):
        csv_url = "https://s3.opensky-network.org/data-samples/metadata/aircraftDatabase.csv"
        print(f"🌐 Downloading OpenSky Aircraft Database (~90MB) from {csv_url}...")
        r = requests.get(csv_url, stream=True)
        with open(csv_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=65536):
                if chunk:
                    f.write(chunk)
                    
    print("💾 Converting aircraftDatabase.csv to indexed Parquet...")
    df_meta = pd.read_csv(csv_path, usecols=cols, dtype=str, low_memory=False)
    df_meta = df_meta.dropna(subset=['icao24'])
    df_meta['icao24'] = df_meta['icao24'].str.strip().str.lower()
    df_meta = df_meta[df_meta['icao24'] != '']
    df_meta = df_meta.drop_duplicates(subset=['icao24'])
    df_meta.to_parquet(parquet_path, index=False)
    return df_meta

AIRCRAFT_DB = ensure_aircraft_database(CACHE_DIR)

def classify_aircraft_type(typecode: str = '', model: str = '', category_desc: str = '', cat_id: int = 0) -> str:
    """Classifies airframe into meaningful operational groups (Heavy Jet, Medium Jet, Bizjet, Turboprop, Light/Leisure, Helicopter, Military)."""
    tc = str(typecode).upper().strip()
    m = str(model).upper().strip()
    cat = str(category_desc).lower().strip()
    
    # 1. Helicopter / Rotorcraft
    helis = {'EC35', 'EC45', 'H145', 'H135', 'A109', 'A139', 'AW139', 'AW109', 'B06', 'B429', 'AS50', 'AS55', 'EC20', 'R44', 'R22', 'R66', 'S76', 'S92', 'H125', 'BK117', 'EH10', 'NH90', 'CH47', 'UH60', 'EC55', 'SA33'}
    if tc in helis or 'rotorcraft' in cat or 'helicopter' in m or cat_id == 7:
        return 'Helicopter / Rotorcraft'
        
    # 2. Military / Fast Jet / Special Ops
    military = {'F18', 'F18H', 'PC21', 'PC7', 'PC9', 'EUFI', 'F35', 'TORN', 'HAWK', 'M346', 'C130', 'A400', 'C17', 'K35R', 'A330_MRTT'}
    if tc in military or 'fighter' in m or 'swiss air force' in m or cat_id == 6:
        return 'Military / Special Ops'
        
    # 3. Commercial Heavy Jet (Widebody)
    heavies = {'B77W', 'B772', 'B773', 'B77L', 'B77F', 'B788', 'B789', 'B78X', 'A332', 'A333', 'A339', 'A343', 'A346', 'A359', 'A35K', 'B744', 'B748', 'A388', 'MD11', 'A306', 'B763', 'B764'}
    if tc in heavies or 'heavy' in cat or cat_id == 5:
        return 'Commercial Heavy Jet'
        
    # 4. Business Jet
    bizjets = {'CL60', 'CL30', 'CL35', 'GL5T', 'GLEX', 'GL7T', 'GL6T', 'FA7X', 'FA8X', 'FA50', 'FA20', 'F900', 'F2TH', 'C56X', 'C680', 'C700', 'C750', 'C510', 'C525', 'C25A', 'C25B', 'C25C', 'C550', 'C560', 'E55P', 'E50P', 'E545', 'E550', 'LJ35', 'LJ45', 'LJ60', 'LJ75', 'HA4T', 'PC24', 'PRM1', 'H25B', 'ASTR', 'G150', 'G200', 'G280', 'GLF4', 'GLF5', 'GLF6', 'GA5C', 'GA6C'}
    if tc in bizjets or 'business jet' in m or 'gulfstream' in m or 'citation' in m or 'falcon' in m or 'learjet' in m:
        return 'Business Jet'
        
    # 5. Commercial Medium Jet (Airliner & Regional Jet)
    medium_jets = {'A320', 'A321', 'A319', 'A318', 'A20N', 'A21N', 'A19N', 'BCS3', 'BCS1', 'B738', 'B737', 'B739', 'B733', 'B734', 'B735', 'B38M', 'B39M', 'E190', 'E195', 'E170', 'E175', 'E290', 'E295', 'CRJ9', 'CRJ7', 'CRJ2', 'CRJX', 'F100', 'F70', 'RJ1H', 'RJ85', 'B712', 'SU95', 'B752', 'B753'}
    if tc in medium_jets or ('airbus' in m and ('A32' in m or 'A31' in m)) or cat_id in [3, 4] or 'large' in cat:
        return 'Commercial Medium Jet'
        
    # 6. Regional Turboprop & Utility
    turboprops = {'PC12', 'PC6T', 'DH8D', 'DH8C', 'DH8B', 'DH8A', 'AT72', 'AT76', 'AT75', 'AT45', 'AT43', 'B350', 'BE20', 'BE9L', 'BE99', 'BE19', 'C208', 'D228', 'D328', 'SB20', 'JS41', 'JS31', 'JS32', 'SW4', 'E120', 'SF34', 'L410', 'PAY3', 'PA42', 'PA31'}
    if tc in turboprops or 'turboprop' in cat or 'king air' in m or 'pilatus pc-12' in m or 'dash 8' in m or 'atr' in m or cat_id == 2:
        return 'Regional Turboprop'
        
    # 7. Light Aircraft / Leisure & GA
    light_ga = {'C172', 'C152', 'C150', 'C182', 'C177', 'C210', 'PA28', 'P28A', 'PA18', 'PA24', 'PA32', 'PA34', 'PA44', 'DR40', 'DR30', 'DV20', 'DA40', 'DA20', 'DA42', 'DA62', 'SR20', 'SR22', 'AA5', 'AA5A', 'AA5B', 'TB10', 'TB20', 'TB9', 'M20P', 'M20T', 'RV7', 'RV8', 'RV6', 'RV10', 'RV12', 'ULAC', 'G109', 'S10S', 'AT3', 'AP22', 'B121', 'FDCT', 'EVE9', 'P208', 'P202', 'P06T'}
    if tc in light_ga or 'light (< 15500' in cat or 'cessna' in m or 'piper' in m or 'cirrus' in m or 'diamond' in m or 'robin' in m or cat_id in [1, 11]:
        return 'Light Aircraft / Leisure GA'
        
    # 8. Glider / Sailplane
    if 'glider' in cat or 'sailplane' in cat or cat_id == 8:
        return 'Glider / Sailplane'
        
    return 'Other / General'

def estimate_ground_noise_level(
    altitude_m: float,
    vertical_rate_ms: float = 0.0,
    velocity_ms: float = 0.0,
    typecode: str = 'Unknown',
    category_desc: str = ''
) -> float:
    """Estimates perceived A-weighted sound pressure level on the ground in dB(A) via ECAC Doc 29 / FAA AEDT."""
    group = classify_aircraft_type(typecode, '', category_desc)
    
    if group == 'Commercial Heavy Jet':
        L0 = 104.0
    elif group == 'Commercial Medium Jet':
        tc = str(typecode).upper().strip()
        L0 = 92.0 if tc in {'A20N', 'A21N', 'BCS3', 'BCS1', 'E290', 'B38M'} else 97.0
    elif group == 'Business Jet':
        L0 = 91.0
    elif group == 'Regional Turboprop' or group == 'Helicopter / Rotorcraft':
        L0 = 87.0
    elif group == 'Light Aircraft / Leisure GA' or group == 'Glider / Sailplane':
        L0 = 76.0
    elif group == 'Military / Special Ops':
        L0 = 106.0
    else:
        L0 = 93.0
        
    if vertical_rate_ms > 1.5:
        thrust_corr = min(7.0, 2.0 + (vertical_rate_ms - 1.5) * 1.2)
    elif vertical_rate_ms < -1.5:
        thrust_corr = -5.0
    else:
        thrust_corr = 0.0
        
    speed_corr = min(3.0, (velocity_ms - 140.0) * 0.04) if velocity_ms > 140.0 else 0.0
        
    d = max(50.0, float(altitude_m))
    d0 = 300.0
    geom_loss = 20.0 * math.log10(d / d0)
    atmo_loss = 0.005 * d
    
    noise_db = L0 + thrust_corr + speed_corr - geom_loss - atmo_loss
    return max(30.0, min(115.0, round(noise_db, 1)))

def estimate_ground_noise_level_vectorized(
    altitudes: np.ndarray,
    vertical_rates: np.ndarray,
    velocities: np.ndarray,
    typecode: str = 'Unknown',
    category_desc: str = ''
) -> np.ndarray:
    """Vectorized calculation of ground noise level in dB(A) across arrays (10x faster)."""
    group = classify_aircraft_type(typecode, '', category_desc)
    if group == 'Commercial Heavy Jet':
        L0 = 104.0
    elif group == 'Commercial Medium Jet':
        tc = str(typecode).upper().strip()
        L0 = 92.0 if tc in {'A20N', 'A21N', 'BCS3', 'BCS1', 'E290', 'B38M'} else 97.0
    elif group == 'Business Jet':
        L0 = 91.0
    elif group == 'Regional Turboprop' or group == 'Helicopter / Rotorcraft':
        L0 = 87.0
    elif group == 'Light Aircraft / Leisure GA' or group == 'Glider / Sailplane':
        L0 = 76.0
    elif group == 'Military / Special Ops':
        L0 = 106.0
    else:
        L0 = 93.0

    d = np.maximum(50.0, altitudes.astype(np.float32))
    geom_loss = 20.0 * np.log10(d / 300.0)
    atmo_loss = 0.005 * d

    thrust_corr = np.zeros_like(vertical_rates, dtype=np.float32)
    climb_mask = vertical_rates > 1.5
    desc_mask = vertical_rates < -1.5
    thrust_corr[climb_mask] = np.minimum(7.0, 2.0 + (vertical_rates[climb_mask] - 1.5) * 1.2)
    thrust_corr[desc_mask] = -5.0

    speed_corr = np.zeros_like(velocities, dtype=np.float32)
    fast_mask = velocities > 140.0
    speed_corr[fast_mask] = np.minimum(3.0, (velocities[fast_mask] - 140.0) * 0.04)

    noise_db = L0 + thrust_corr + speed_corr - geom_loss - atmo_loss
    return np.clip(np.round(noise_db, 1), 30.0, 115.0).astype(np.float32)

print("✅ Dependencies, Aircraft DB, Group Classifier & Acoustic Model ready.")
print(f"📁 Cache directory: {CACHE_DIR} ({len(AIRCRAFT_DB):,} aircraft indexed)")


## 1. 🗺️ Region Selection & Enclosing Cache Discovery
Select your geographic bounding box. The engine automatically discovers existing cache files that **fully enclose** your target area.

In [ ]:
from typing import Any, Dict, List, Tuple, Optional, Set
DEFAULT_LAT_MIN = 47.1102
DEFAULT_LAT_MAX = 47.4855
DEFAULT_LON_MIN = 8.3720
DEFAULT_LON_MAX = 9.0478

SELECTED_BBOX = {
    "lat_min": DEFAULT_LAT_MIN,
    "lat_max": DEFAULT_LAT_MAX,
    "lon_min": DEFAULT_LON_MIN,
    "lon_max": DEFAULT_LON_MAX
}

def parse_bbox_from_filename(filename: str) -> Optional[Dict[str, float]]:
    pattern = r"bbox-lat-([0-9]+_[0-9]+)-([0-9]+_[0-9]+)-lon-([0-9]+_[0-9]+)-([0-9]+_[0-9]+)\.parquet"
    m = re.match(pattern, filename)
    if not m:
        return None
    return {
        "lat_min": float(m.group(1).replace("_", ".")),
        "lat_max": float(m.group(2).replace("_", ".")),
        "lon_min": float(m.group(3).replace("_", ".")),
        "lon_max": float(m.group(4).replace("_", "."))
    }

def find_enclosing_cache(lat_min: float, lat_max: float, lon_min: float, lon_max: float, cache_dir: str = CACHE_DIR) -> Optional[Dict[str, Any]]:
    if not os.path.exists(cache_dir):
        return None
    candidates = []
    for fname in os.listdir(cache_dir):
        if fname.startswith("bbox-") and fname.endswith(".parquet"):
            b = parse_bbox_from_filename(fname)
            if b:
                eps = 1e-4
                if b["lat_min"] <= lat_min + eps and b["lat_max"] >= lat_max - eps and                    b["lon_min"] <= lon_min + eps and b["lon_max"] >= lon_max - eps:
                    area = (b["lat_max"] - b["lat_min"]) * (b["lon_max"] - b["lon_min"])
                    candidates.append((area, {
                        "filename": fname,
                        "path": os.path.join(cache_dir, fname),
                        **b
                    }))
    if not candidates:
        return None
    candidates.sort(key=lambda x: x[0])
    return candidates[0][1]

center_lat = (DEFAULT_LAT_MIN + DEFAULT_LAT_MAX) / 2.0
center_lon = (DEFAULT_LON_MIN + DEFAULT_LON_MAX) / 2.0

map_widget = Map(center=[center_lat, center_lon], zoom=10, basemap=basemaps.OpenTopoMap, scroll_wheel_zoom=True)

default_rect = Rectangle(
    bounds=((DEFAULT_LAT_MIN, DEFAULT_LON_MIN), (DEFAULT_LAT_MAX, DEFAULT_LON_MAX)),
    color="#ff3366",
    weight=2,
    fill_opacity=0.18
)
map_widget.add_layer(default_rect)

def generate_bbox_info_html(lat_min: float, lat_max: float, lon_min: float, lon_max: float) -> str:
    cache_match = find_enclosing_cache(lat_min, lat_max, lon_min, lon_max)
    if cache_match:
        cache_status = f'''
        <span style="color: #10b981; font-weight: bold;">✅ Enclosing Cache Found:</span> <code>{cache_match['filename']}</code><br>
        • Cache Bounding Box : <strong>[{cache_match['lat_min']:.4f}, {cache_match['lat_max']:.4f}] &times; [{cache_match['lon_min']:.4f}, {cache_match['lon_max']:.4f}]</strong>
        '''
    else:
        cache_status = f'''
        <span style="color: #f59e0b; font-weight: bold;">⚠️ No enclosing cache found</span> for this exact region.<br>
        <span style="font-size: 11px; color: #94a3b8;">(Run <code>python fetch_flight_data.py</code> or adjust the bounding box)</span>
        '''
        
    return f'''
    <div style="padding: 12px 16px; background: #0f172a; color: #f8fafc; border-radius: 8px; font-family: monospace; font-size: 12px; border: 1px solid #334155; margin-top: 8px; line-height: 1.6;">
        <strong style="color: #38bdf8; font-size: 13px;">📍 Active Analysis Bounding Box:</strong><br>
        • Latitude Range : <strong>{lat_min:.4f}° N</strong> &rarr; <strong>{lat_max:.4f}° N</strong><br>
        • Longitude Range: <strong>{lon_min:.4f}° E</strong> &rarr; <strong>{lon_max:.4f}° E</strong><br>
        {cache_status}
    </div>
    '''

info_out = HTML(value=generate_bbox_info_html(DEFAULT_LAT_MIN, DEFAULT_LAT_MAX, DEFAULT_LON_MIN, DEFAULT_LON_MAX))
draw_control = DrawControl(
    rectangle={'shapeOptions': {'color': '#ff3366', 'weight': 2, 'fillOpacity': 0.18}},
    polyline={}, polygon={}, circlemarker={}
)

def handle_draw(target, action, geo_json):
    if action == 'created' and geo_json['geometry']['type'] == 'Polygon':
        coords = geo_json['geometry']['coordinates'][0]
        lats = [c[1] for c in coords]
        lons = [c[0] for c in coords]
        
        SELECTED_BBOX["lat_min"] = round(min(lats), 4)
        SELECTED_BBOX["lat_max"] = round(max(lats), 4)
        SELECTED_BBOX["lon_min"] = round(min(lons), 4)
        SELECTED_BBOX["lon_max"] = round(max(lons), 4)
        
        info_out.value = generate_bbox_info_html(
            SELECTED_BBOX["lat_min"], SELECTED_BBOX["lat_max"],
            SELECTED_BBOX["lon_min"], SELECTED_BBOX["lon_max"]
        )

draw_control.on_draw(handle_draw)
map_widget.add_control(draw_control)

display(map_widget)
display(info_out)


## 2. ⚙️ Interactive Date/Time Range, Multi-Select Groups, Models & Noise Filtering
Interactively configure the temporal scope with full UTC dates (`YYYY-MM-DD HH:MM:SS UTC`), multi-select operational aircraft groups, filter specific models (with automatic cascading cross-filtering), and adjust ground noise thresholds.

In [ ]:
# 1. Active Bounding Box & Matched Cache
LAT_MIN = SELECTED_BBOX["lat_min"]
LAT_MAX = SELECTED_BBOX["lat_max"]
LON_MIN = SELECTED_BBOX["lon_min"]
LON_MAX = SELECTED_BBOX["lon_max"]
REGION_NAME = f"Map Region [{LAT_MIN:.2f}° to {LAT_MAX:.2f}° N, {LON_MIN:.2f}° to {LON_MAX:.2f}° E]"

MATCHED_CACHE = find_enclosing_cache(LAT_MIN, LAT_MAX, LON_MIN, LON_MAX)

cached_timestamps = []
ALL_AIRCRAFT_GROUPS = [
    'Commercial Heavy Jet',
    'Commercial Medium Jet',
    'Business Jet',
    'Regional Turboprop',
    'Light Aircraft / Leisure GA',
    'Helicopter / Rotorcraft',
    'Military / Special Ops',
    'Glider / Sailplane',
    'Other / General'
]

preview_models_by_group = {g: set() for g in ALL_AIRCRAFT_GROUPS}
preview_models_all = set()

def load_cache_timestamps():
    global MATCHED_CACHE, preview_models_all, preview_models_by_group
    ts_list = []
    if MATCHED_CACHE and os.path.exists(MATCHED_CACHE['path']):
        try:
            t = pq.read_table(MATCHED_CACHE['path'], columns=['timestamp', 'icao24'])
            df_prev = t.to_pandas()
            ts_list = sorted(df_prev['timestamp'].astype(int).unique().tolist())
            
            df_prev['icao24'] = df_prev['icao24'].astype(str).str.strip().str.lower()
            merged_prev = df_prev[['icao24']].drop_duplicates().merge(AIRCRAFT_DB, on='icao24', how='left')
            
            for r in merged_prev[['typecode', 'model', 'categoryDescription']].itertuples(index=False):
                tc = str(r[0]).strip() if pd.notna(r[0]) else 'Unknown'
                m = str(r[1]).strip() if pd.notna(r[1]) else 'Unknown'
                cd = str(r[2]).strip() if pd.notna(r[2]) else 'Unknown'
                grp = classify_aircraft_type(tc, m, cd)
                if tc and tc != 'Unknown':
                    preview_models_all.add(tc)
                    if grp in preview_models_by_group:
                        preview_models_by_group[grp].add(tc)
        except Exception as e:
            print(f"⚠️ Could not read cache metadata preview: {e}")
    return ts_list

def build_slider_options(ts_list, step_sec=1800):
    if not ts_list:
        _now = datetime.now(timezone.utc)
        min_dt = _now - timedelta(hours=1)
        max_dt = _now
        return [(min_dt.strftime('%Y-%m-%d %H:%M:%S UTC'), int(min_dt.timestamp())),
                (max_dt.strftime('%Y-%m-%d %H:%M:%S UTC'), int(max_dt.timestamp()))]
    min_ts = ts_list[0]
    max_ts = ts_list[-1]
    slider_ts = [min_ts]
    cur = min_ts + (step_sec - (min_ts % step_sec))
    while cur < max_ts:
        slider_ts.append(cur)
        cur += step_sec
    if slider_ts[-1] != max_ts:
        slider_ts.append(max_ts)
    return [
        (datetime.fromtimestamp(ts, tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC'), ts)
        for ts in slider_ts
    ]

cached_timestamps = load_cache_timestamps()

if cached_timestamps:
    min_cached_dt = datetime.fromtimestamp(cached_timestamps[0], tz=timezone.utc)
    max_cached_dt = datetime.fromtimestamp(cached_timestamps[-1], tz=timezone.utc)
else:
    min_cached_dt = datetime.now(timezone.utc) - timedelta(hours=1)
    max_cached_dt = datetime.now(timezone.utc)

time_options = build_slider_options(cached_timestamps, step_sec=1800)

# 2. Interactive Time Range Widgets with Dual Selectors & Full Associated Dates
use_full_history_chk = Checkbox(
    value=True,
    description='Use Full Cache History (Analyze All Recorded Data)',
    style={'description_width': 'initial'},
    layout=Layout(width='420px')
)

btn_refresh_cache = Button(
    description='🔄 Refresh Cache',
    tooltip='Re-scan disk cache to include newly ingested flight data',
    button_style='info',
    layout=Layout(width='140px', height='28px', margin='2px 0 0 10px')
)

start_badge = HTML(
    value=f'<div style="background: #064e3b; color: #6ee7b7; padding: 4px 10px; border-radius: 4px; font-family: monospace; font-size: 11px; border: 1px solid #059669;"><strong>🟢 Start:</strong> {min_cached_dt.strftime("%Y-%m-%d %H:%M:%S UTC")}</div>'
)

end_badge = HTML(
    value=f'<div style="background: #881337; color: #fda4af; padding: 4px 10px; border-radius: 4px; font-family: monospace; font-size: 11px; border: 1px solid #e11d48;"><strong>🔴 End:</strong> {max_cached_dt.strftime("%Y-%m-%d %H:%M:%S UTC")}</div>'
)

span_badge = HTML(
    value='<div style="background: #1e293b; color: #94a3b8; padding: 4px 10px; border-radius: 4px; font-family: monospace; font-size: 11px; border: 1px solid #334155;"><strong>⏱️ Window:</strong> Full Cache</div>'
)

selectors_badge_row = HBox([start_badge, end_badge, span_badge], layout=Layout(margin='4px 0 2px 0'))

time_slider = SelectionRangeSlider(
    options=[opt[0] for opt in time_options],
    index=(0, len(time_options) - 1),
    description='Time Range:',
    disabled=True,
    continuous_update=False,
    style={'description_width': '95px'},
    layout=Layout(width='720px')
)

start_dt_picker = DatetimePicker(
    value=min_cached_dt,
    description='Start UTC:',
    disabled=True,
    layout=Layout(width='340px')
)

end_dt_picker = DatetimePicker(
    value=max_cached_dt,
    description='End UTC:',
    disabled=True,
    layout=Layout(width='340px')
)

pickers_row = HBox([
    start_dt_picker,
    end_dt_picker
], layout=Layout(margin='2px 0 4px 0'))

status_label = HTML(
    value=f'<div style="font-family: monospace; color: #38bdf8; margin-top: 4px;">📅 Active Window: <strong>{min_cached_dt.strftime("%Y-%m-%d %H:%M:%S UTC")}</strong> &rarr; <strong>{max_cached_dt.strftime("%Y-%m-%d %H:%M:%S UTC")}</strong></div>'
)

_updating_ui = False

def update_ui_status_and_cascades():
    global _updating_ui
    if _updating_ui:
        return
    _updating_ui = True
    try:
        if use_full_history_chk.value:
            s_ts = time_options[0][1]
            e_ts = time_options[-1][1]
            dt_s = min_cached_dt
            dt_e = max_cached_dt
            s_str = dt_s.strftime('%Y-%m-%d %H:%M:%S UTC')
            e_str = dt_e.strftime('%Y-%m-%d %H:%M:%S UTC')
            span_hrs = max(0.01, (e_ts - s_ts) / 3600.0)
            span_str = f"{span_hrs:.1f} hrs ({span_hrs/24.0:.1f} days)" if span_hrs >= 24 else f"{span_hrs:.1f} hrs"
            t_info = f'<span style="color: #10b981;">Full History [{s_str} &rarr; {e_str}]</span> ({len(cached_timestamps):,} time slices, {span_str})'
            time_slider.disabled = True
            start_dt_picker.disabled = True
            end_dt_picker.disabled = True
            start_dt_picker.value = dt_s
            end_dt_picker.value = dt_e
        else:
            time_slider.disabled = False
            start_dt_picker.disabled = False
            end_dt_picker.disabled = False
            s_idx, e_idx = time_slider.index
            s_ts = time_options[s_idx][1]
            e_ts = time_options[e_idx][1]
            dt_s = datetime.fromtimestamp(s_ts, tz=timezone.utc)
            dt_e = datetime.fromtimestamp(e_ts, tz=timezone.utc)
            s_str = dt_s.strftime('%Y-%m-%d %H:%M:%S UTC')
            e_str = dt_e.strftime('%Y-%m-%d %H:%M:%S UTC')
            span_min = max(0.1, (e_ts - s_ts) / 60.0)
            span_hrs = span_min / 60.0
            span_str = f"{span_hrs/24.0:.1f} days ({span_hrs:.1f} hrs)" if span_hrs >= 24 else (f"{span_hrs:.2f} hrs" if span_hrs >= 1.0 else f"{span_min:.1f} mins")
            t_info = f'<span style="color: #38bdf8;">Sub-Range: {s_str} &rarr; {e_str} ({span_str})</span>'
            start_dt_picker.value = dt_s
            end_dt_picker.value = dt_e
            
        start_badge.value = f'<div style="background: #064e3b; color: #6ee7b7; padding: 4px 10px; border-radius: 4px; font-family: monospace; font-size: 11px; border: 1px solid #059669;"><strong>🟢 Start:</strong> {s_str}</div>'
        end_badge.value = f'<div style="background: #881337; color: #fda4af; padding: 4px 10px; border-radius: 4px; font-family: monospace; font-size: 11px; border: 1px solid #e11d48;"><strong>🔴 End:</strong> {e_str}</div>'
        span_badge.value = f'<div style="background: #1e293b; color: #94a3b8; padding: 4px 10px; border-radius: 4px; font-family: monospace; font-size: 11px; border: 1px solid #334155;"><strong>⏱️ Window:</strong> {span_str}</div>'

        sel_groups = set(group_filter_widget.value)
        if not sel_groups or len(sel_groups) == len(ALL_AIRCRAFT_GROUPS):
            available_models = sorted(list(preview_models_all))
        else:
            allowed = set()
            for g in sel_groups:
                allowed.update(preview_models_by_group.get(g, set()))
            available_models = sorted(list(allowed))
            
        current_model = model_filter_dropdown.value
        new_model_opts = ['All Specific Models'] + available_models
        model_filter_dropdown.options = new_model_opts
        if current_model in new_model_opts:
            model_filter_dropdown.value = current_model
        else:
            model_filter_dropdown.value = 'All Specific Models'
            
        grp_text = f"{len(sel_groups)} of {len(ALL_AIRCRAFT_GROUPS)} Groups" if len(sel_groups) < len(ALL_AIRCRAFT_GROUPS) else "All Groups"
        mod_text = model_filter_dropdown.value
        noise_text = f"≥ {noise_threshold_slider.value} dB(A)" if noise_threshold_slider.value > 30 else "All Noise"
        
        status_label.value = f'<div style="font-family: monospace; font-size: 11px; padding: 6px 10px; background: #0f172a; border-radius: 6px; border: 1px solid #334155; margin-top: 4px; line-height: 1.5;">📅 <strong>Time:</strong> {t_info}<br>🛩️ <strong>Filters:</strong> <span style="color: #38bdf8;">{grp_text}</span> &bull; Model: <span style="color: #38bdf8;">{mod_text}</span> &bull; Min Noise: <span style="color: #f59e0b;">{noise_text}</span></div>'
    finally:
        _updating_ui = False

def on_history_toggle(change):
    time_slider.disabled = change['new']
    start_dt_picker.disabled = change['new']
    end_dt_picker.disabled = change['new']
    update_ui_status_and_cascades()

def on_slider_change(change):
    update_ui_status_and_cascades()

def on_picker_change(_change):
    global _updating_ui
    if _updating_ui or use_full_history_chk.value:
        return
    _updating_ui = True
    try:
        s_dt = start_dt_picker.value
        e_dt = end_dt_picker.value
        if s_dt is None or e_dt is None:
            return
        if s_dt.tzinfo is None:
            s_dt = s_dt.replace(tzinfo=timezone.utc)
        if e_dt.tzinfo is None:
            e_dt = e_dt.replace(tzinfo=timezone.utc)
        s_ts = int(s_dt.timestamp())
        e_ts = int(e_dt.timestamp())
        if s_ts > e_ts:
            s_ts, e_ts = e_ts, s_ts
            
        s_idx = min(range(len(time_options)), key=lambda i: abs(time_options[i][1] - s_ts))
        e_idx = min(range(len(time_options)), key=lambda i: abs(time_options[i][1] - e_ts))
        if s_idx >= e_idx:
            if s_idx > 0:
                s_idx = e_idx - 1
            else:
                e_idx = s_idx + 1
        time_slider.index = (s_idx, e_idx)
    finally:
        _updating_ui = False
    update_ui_status_and_cascades()

use_full_history_chk.observe(on_history_toggle, names='value')
time_slider.observe(on_slider_change, names='value')
start_dt_picker.observe(on_picker_change, names='value')
end_dt_picker.observe(on_picker_change, names='value')

def on_refresh_cache_click(_b):
    global cached_timestamps, min_cached_dt, max_cached_dt, time_options
    cached_timestamps = load_cache_timestamps()
    if cached_timestamps:
        min_cached_dt = datetime.fromtimestamp(cached_timestamps[0], tz=timezone.utc)
        max_cached_dt = datetime.fromtimestamp(cached_timestamps[-1], tz=timezone.utc)
        time_options = build_slider_options(cached_timestamps, step_sec=1800)
        time_slider.options = [opt[0] for opt in time_options]
        time_slider.index = (0, len(time_options) - 1)
        update_ui_status_and_cascades()
        print(f"✅ Live cache refreshed: {len(cached_timestamps):,} waypoints up to {max_cached_dt.strftime('%Y-%m-%d %H:%M:%S UTC')}")

btn_refresh_cache.on_click(on_refresh_cache_click)

def create_preset_btn(label: str, span_hours: Optional[float] = None) -> Button:
    btn = Button(
        description=label,
        layout=Layout(width='48px', height='28px', margin='2px'),
        button_style='primary' if label == 'All' else ''
    )
    def on_click(_b):
        if label == 'All' or span_hours is None:
            use_full_history_chk.value = True
            time_slider.index = (0, len(time_options) - 1)
        else:
            use_full_history_chk.value = False
            time_slider.disabled = False
            start_dt_picker.disabled = False
            end_dt_picker.disabled = False
            max_ts = time_options[-1][1]
            target_min_ts = max(time_options[0][1], max_ts - int(span_hours * 3600))
            s_idx = 0
            for i, opt in enumerate(time_options):
                if opt[1] >= target_min_ts:
                    s_idx = i
                    break
            time_slider.index = (s_idx, len(time_options) - 1)
        update_ui_status_and_cascades()
    btn.on_click(on_click)
    return btn

preset_btns = [
    create_preset_btn('1h', 1.0),
    create_preset_btn('3h', 3.0),
    create_preset_btn('6h', 6.0),
    create_preset_btn('12h', 12.0),
    create_preset_btn('24h', 24.0),
    create_preset_btn('3d', 72.0),
    create_preset_btn('7d', 168.0),
    create_preset_btn('All', None)
]

btn_pan_left = Button(
    description='◂ Pan',
    tooltip='Shift time window earlier',
    layout=Layout(width='58px', height='28px', margin='2px')
)
btn_pan_right = Button(
    description='Pan ▸',
    tooltip='Shift time window later',
    layout=Layout(width='58px', height='28px', margin='2px')
)

def on_pan_left_click(_b):
    if use_full_history_chk.value:
        return
    s_idx, e_idx = time_slider.index
    span = e_idx - s_idx
    shift = max(1, span // 4)
    new_s = max(0, s_idx - shift)
    new_e = new_s + span
    if new_e >= len(time_options):
        new_e = len(time_options) - 1
        new_s = max(0, new_e - span)
    time_slider.index = (new_s, new_e)
    update_ui_status_and_cascades()

def on_pan_right_click(_b):
    if use_full_history_chk.value:
        return
    s_idx, e_idx = time_slider.index
    span = e_idx - s_idx
    shift = max(1, span // 4)
    new_e = min(len(time_options) - 1, e_idx + shift)
    new_s = new_e - span
    if new_s < 0:
        new_s = 0
        new_e = min(len(time_options) - 1, span)
    time_slider.index = (new_s, new_e)
    update_ui_status_and_cascades()

btn_pan_left.on_click(on_pan_left_click)
btn_pan_right.on_click(on_pan_right_click)

top_history_row = HBox([use_full_history_chk, btn_refresh_cache])

presets_row = HBox([
    Label(value='Presets:', layout=Layout(width='58px', margin='4px 0 0 0')),
    btn_pan_left
] + preset_btns + [btn_pan_right])

# 3. Multi-Select Aircraft Type Group Widget & Cascading Controls
group_filter_widget = SelectMultiple(
    options=ALL_AIRCRAFT_GROUPS,
    value=tuple(ALL_AIRCRAFT_GROUPS),
    description='Type Groups:',
    rows=6,
    style={'description_width': '95px'},
    layout=Layout(width='320px')
)

btn_grp_all = Button(description='All', layout=Layout(width='45px', height='24px', margin='1px'))
btn_grp_none = Button(description='None', layout=Layout(width='50px', height='24px', margin='1px'))
btn_grp_airliners = Button(description='Airliners', layout=Layout(width='70px', height='24px', margin='1px'))
btn_grp_ga = Button(description='GA/Leisure', layout=Layout(width='80px', height='24px', margin='1px'))
btn_grp_heli = Button(description='Helis', layout=Layout(width='50px', height='24px', margin='1px'))

def set_groups(val_list):
    group_filter_widget.value = tuple(val_list)
    update_ui_status_and_cascades()

btn_grp_all.on_click(lambda _b: set_groups(ALL_AIRCRAFT_GROUPS))
btn_grp_none.on_click(lambda _b: set_groups([]))
btn_grp_airliners.on_click(lambda _b: set_groups(['Commercial Heavy Jet', 'Commercial Medium Jet']))
btn_grp_ga.on_click(lambda _b: set_groups(['Light Aircraft / Leisure GA', 'Regional Turboprop']))
btn_grp_heli.on_click(lambda _b: set_groups(['Helicopter / Rotorcraft']))

grp_helper_row = HBox([
    Label(value='Quick:', layout=Layout(width='45px')),
    btn_grp_all, btn_grp_none, btn_grp_airliners, btn_grp_ga, btn_grp_heli
])

model_filter_dropdown = Dropdown(
    options=['All Specific Models'] + sorted(list(preview_models_all)),
    value='All Specific Models',
    description='📋 Model:',
    style={'description_width': '75px'},
    layout=Layout(width='260px')
)

noise_threshold_slider = IntSlider(
    value=30,
    min=30,
    max=90,
    step=5,
    description='🔊 Min Noise:',
    style={'description_width': '95px'},
    layout=Layout(width='260px'),
    tooltip='Filter points by minimum estimated ground noise level in dB(A)'
)

def on_group_filter_change(change):
    update_ui_status_and_cascades()

def on_model_filter_change(change):
    update_ui_status_and_cascades()

def on_noise_filter_change(change):
    update_ui_status_and_cascades()

group_filter_widget.observe(on_group_filter_change, names='value')
model_filter_dropdown.observe(on_model_filter_change, names='value')
noise_threshold_slider.observe(on_noise_filter_change, names='value')

filters_col_right = VBox([
    model_filter_dropdown,
    noise_threshold_slider
], layout=Layout(margin='0 0 0 10px'))

filters_main_row = HBox([
    VBox([group_filter_widget, grp_helper_row]),
    filters_col_right
])

display(VBox([
    top_history_row,
    selectors_badge_row,
    time_slider,
    pickers_row,
    presets_row,
    filters_main_row,
    status_label
]))

# Initial update of UI status badges
update_ui_status_and_cascades()

# 4. Global Analysis Constants (Preserved User Configuration)
HISTORICAL_TIME_STEP_SEC = 60

ALTITUDE_TIERS = {
    "0 - 500m (Red)":       {"min": 0,    "max": 500,   "color": "#ff3344", "desc": "Surface ops, approach & initial climb (0 < alt <= 500m)"},
    "500 - 1000m (Orange)": {"min": 500,  "max": 1000,  "color": "#ff6a00", "desc": "Low climb & approach alignment (500 < alt <= 1000m)"},
    "1000 - 2000m (Amber)": {"min": 1000, "max": 2000,  "color": "#ffb300", "desc": "Lower terminal maneuvering airspace (1000 < alt <= 2000m)"},
    "2000 - 3000m (Yellow)":{"min": 2000, "max": 3000,  "color": "#fbc02d", "desc": "Intermediate TMA & entry corridors (2000 < alt <= 3000m)"},
    "3000 - 5000m (Teal)":  {"min": 3000, "max": 5000,  "color": "#00bcd4", "desc": "Regional climb & descent transitions (3000 < alt <= 5000m)"},
    "5000m+ (Green)":       {"min": 5000, "max": 20000, "color": "#00e676", "desc": "En-route high-altitude jet airways (alt > 5000m)"}
}

MIN_ALTITUDE_M    = 0
MAX_ALTITUDE_M    = 10000
EXCLUDE_ON_GROUND = True

INTERPOLATE_PATHS       = True
INTERPOLATION_STEP_KM   = 0.5


## 3. 🗄️ Cache Ingestion, Multi-Select Groups & Acoustic Modeling
Loads state vectors from the enclosing Parquet cache file, joins aircraft metadata, classifies aircraft groups, applies multi-group and cascading model filters, models ground noise, computes coverage metrics, and **persists filtered waypoints to disk** (`stage3_filtered_waypoints.parquet`) while freeing in-memory DataFrames to maximize available RAM for subsequent stages.

In [ ]:
from typing import Any, Dict, List, Tuple, Optional, Set
class CacheDataReader:
    """Loads state vectors from enclosing Parquet cache files with predicate pushdown, downcasting, and selective metadata joins."""
    def __init__(self, cache_dir: str = CACHE_DIR):
        self.cache_dir = cache_dir
        self.df_meta = ensure_aircraft_database(cache_dir)
        
    def load_cached_dataset(
        self,
        lat_min: float, lat_max: float, lon_min: float, lon_max: float,
        use_full_history: bool = True,
        time_slider_widget: Optional[Any] = None,
        time_options_list: Optional[List[Tuple[str, int]]] = None,
        start_dt_picker: Optional[Any] = None,
        end_dt_picker: Optional[Any] = None,
        selected_groups: Optional[Tuple[str, ...]] = None,
        model_filter: str = 'All Specific Models',
        min_noise_dba: float = 30.0
    ) -> Tuple[pd.DataFrame, Dict[str, Any]]:
        cache_match = find_enclosing_cache(lat_min, lat_max, lon_min, lon_max, self.cache_dir)
        
        if not cache_match or not os.path.exists(cache_match['path']):
            print(f"⚠️ No enclosing cache file found in {self.cache_dir} covering [{lat_min:.4f}, {lat_max:.4f}] x [{lon_min:.4f}, {lon_max:.4f}].")
            return pd.DataFrame(), {}
            
        cache_path = cache_match['path']
        
        # 1. Fast cache bounds discovery without reading the whole 7.75M rows table into Pandas
        if time_options_list:
            cache_min_ts = time_options_list[0][1]
            cache_max_ts = time_options_list[-1][1]
        else:
            ts_tbl = pq.read_table(cache_path, columns=['timestamp'])
            cache_min_ts = int(ts_tbl.column('timestamp').to_numpy().min())
            cache_max_ts = int(ts_tbl.column('timestamp').to_numpy().max())
            del ts_tbl
            
        if use_full_history:
            slice_start_ts = cache_min_ts
            slice_end_ts   = cache_max_ts
        elif start_dt_picker is not None and end_dt_picker is not None and start_dt_picker.value and end_dt_picker.value:
            s_dt = start_dt_picker.value
            e_dt = end_dt_picker.value
            if s_dt.tzinfo is None:
                s_dt = s_dt.replace(tzinfo=timezone.utc)
            if e_dt.tzinfo is None:
                e_dt = e_dt.replace(tzinfo=timezone.utc)
            slice_start_ts = int(s_dt.timestamp())
            slice_end_ts   = int(e_dt.timestamp())
        elif time_slider_widget is not None and time_options_list:
            s_idx, e_idx = time_slider_widget.index
            slice_start_ts = time_options_list[s_idx][1]
            slice_end_ts   = time_options_list[e_idx][1]
        else:
            slice_start_ts = cache_min_ts
            slice_end_ts   = cache_max_ts
            
        if slice_start_ts > slice_end_ts:
            slice_start_ts, slice_end_ts = slice_end_ts, slice_start_ts
            
        req_start_str = datetime.fromtimestamp(slice_start_ts, tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')
        req_end_str   = datetime.fromtimestamp(slice_end_ts, tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')
        
        # 2. PyArrow Predicate & Projection Pushdown directly during file read (100x faster, ~99% less RAM)
        margin = 0.02
        filters = [
            ('timestamp', '>=', slice_start_ts),
            ('timestamp', '<=', slice_end_ts),
            ('latitude', '>=', lat_min - margin),
            ('latitude', '<=', lat_max + margin),
            ('longitude', '>=', lon_min - margin),
            ('longitude', '<=', lon_max + margin)
        ]
        
        schema_names = set(pq.read_schema(cache_path).names)
        desired_cols = ['icao24', 'callsign', 'timestamp', 'latitude', 'longitude', 'altitude', 'velocity', 'vertical_rate', 'on_ground', 'category']
        load_cols = [c for c in desired_cols if c in schema_names]
        
        table_slice = pq.read_table(cache_path, columns=load_cols, filters=filters)
        df_slice = table_slice.to_pandas()
        del table_slice
        
        if df_slice.empty:
            print("⚠️ No waypoints found in the selected time and spatial window.")
            return pd.DataFrame(), {}
            
        # Downcast numeric columns to float32 to cut memory in half
        df_slice['latitude'] = df_slice['latitude'].astype(np.float32)
        df_slice['longitude'] = df_slice['longitude'].astype(np.float32)
        df_slice['altitude'] = df_slice['altitude'].astype(np.float32)
        if 'vertical_rate' in df_slice.columns:
            df_slice['vertical_rate'] = df_slice['vertical_rate'].astype(np.float32)
        if 'velocity' in df_slice.columns:
            df_slice['velocity'] = df_slice['velocity'].astype(np.float32)
            
        # 3. Merge static aircraft metadata only for active sliced records (selective join)
        df_slice['icao24'] = df_slice['icao24'].astype(str).str.strip().str.lower()
        active_icaos = df_slice['icao24'].dropna().unique()
        meta_cols = [c for c in ['icao24', 'registration', 'model', 'typecode', 'operator', 'categoryDescription'] if c in self.df_meta.columns]
        df_meta_active = self.df_meta[self.df_meta['icao24'].isin(active_icaos)][meta_cols]
        df_slice = df_slice.merge(df_meta_active, on='icao24', how='left')
        
        df_slice['typecode'] = df_slice['typecode'].fillna('Unknown').astype(str).str.strip()
        df_slice['model'] = df_slice['model'].fillna('Unknown').astype(str).str.strip()
        df_slice['operator'] = df_slice['operator'].fillna('Unknown').astype(str).str.strip()
        df_slice['registration'] = df_slice['registration'].fillna('Unknown').astype(str).str.strip()
        df_slice['categoryDescription'] = df_slice['categoryDescription'].fillna('Unknown').astype(str).str.strip()
        df_slice.loc[df_slice['typecode'] == '', 'typecode'] = 'Unknown'
        df_slice.loc[df_slice['categoryDescription'] == '', 'categoryDescription'] = 'Unknown'
        
        # High-performance classified mapping on unique aircraft combinations:
        cat_key = 'category' if 'category' in df_slice.columns else 'icao24'
        unique_planes = df_slice[['typecode', 'model', 'categoryDescription', cat_key]].drop_duplicates()
        group_lookup = {}
        for row in unique_planes.itertuples(index=False):
            tc = str(row[0])
            m = str(row[1])
            cd = str(row[2])
            cid = int(row[3]) if (cat_key == 'category' and pd.notna(row[3])) else 0
            group_lookup[(tc, m, cd, cid if cat_key == 'category' else 0)] = classify_aircraft_type(tc, m, cd, cid)
            
        if 'category' in df_slice.columns:
            df_slice['aircraft_type_group'] = [
                group_lookup.get((tc, m, cd, int(cid) if pd.notna(cid) else 0), 'Other / General')
                for tc, m, cd, cid in zip(df_slice['typecode'], df_slice['model'], df_slice['categoryDescription'], df_slice['category'])
            ]
        else:
            df_slice['aircraft_type_group'] = [
                group_lookup.get((tc, m, cd, 0), 'Other / General')
                for tc, m, cd in zip(df_slice['typecode'], df_slice['model'], df_slice['categoryDescription'])
            ]
        
        # Calculate perceived ground noise level for each waypoint using vectorized helper
        if 'ground_noise_dba' not in df_slice.columns:
            vr_col = df_slice['vertical_rate'].to_numpy(dtype=np.float32) if 'vertical_rate' in df_slice.columns else np.zeros(len(df_slice), dtype=np.float32)
            vel_col = df_slice['velocity'].to_numpy(dtype=np.float32) if 'velocity' in df_slice.columns else np.zeros(len(df_slice), dtype=np.float32)
            alts = df_slice['altitude'].to_numpy(dtype=np.float32)
            
            noise_arr = np.zeros(len(df_slice), dtype=np.float32)
            for (tc, cat), grp_indices in df_slice.groupby(['typecode', 'categoryDescription']).groups.items():
                idx_arr = grp_indices.to_numpy()
                noise_arr[idx_arr] = estimate_ground_noise_level_vectorized(
                    alts[idx_arr], vr_col[idx_arr], vel_col[idx_arr], str(tc), str(cat)
                )
            df_slice['ground_noise_dba'] = noise_arr
            
        # Apply Multi-Group Filter
        if selected_groups is not None and len(selected_groups) > 0:
            df_slice = df_slice[df_slice['aircraft_type_group'].isin(selected_groups)]
        elif selected_groups is not None and len(selected_groups) == 0:
            df_slice = df_slice.iloc[0:0]
            
        # Apply Specific Model Filter
        if model_filter and model_filter != 'All Specific Models':
            df_slice = df_slice[df_slice['typecode'] == model_filter]
            
        # Apply Noise Filter
        if min_noise_dba > 30.0:
            df_slice = df_slice[df_slice['ground_noise_dba'] >= min_noise_dba]
            
        # Convert repeated string columns to categories to save RAM
        df_slice['typecode'] = df_slice['typecode'].astype('category')
        df_slice['aircraft_type_group'] = df_slice['aircraft_type_group'].astype('category')
        
        if not df_slice.empty:
            actual_min_ts = int(df_slice['timestamp'].min())
            actual_max_ts = int(df_slice['timestamp'].max())
            actual_start_str = datetime.fromtimestamp(actual_min_ts, tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')
            actual_end_str   = datetime.fromtimestamp(actual_max_ts, tz=timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')
            actual_span_hrs  = max(0.01, (actual_max_ts - actual_min_ts) / 3600.0)
            
            timestamps_sorted = sorted(df_slice['timestamp'].unique())
            segments = []
            cur_seg = [timestamps_sorted[0]]
            for i in range(1, len(timestamps_sorted)):
                if timestamps_sorted[i] - timestamps_sorted[i - 1] > 300:
                    segments.append((cur_seg[0], cur_seg[-1]))
                    cur_seg = [timestamps_sorted[i]]
                else:
                    cur_seg.append(timestamps_sorted[i])
            segments.append((cur_seg[0], cur_seg[-1]))
        else:
            actual_start_str = "N/A"
            actual_end_str   = "N/A"
            actual_span_hrs  = 0.0
            segments         = []
            
        req_duration_hrs = max(0.01, (slice_end_ts - slice_start_ts) / 3600.0)
        coverage_pct = min(100.0, (actual_span_hrs / req_duration_hrs * 100.0)) if req_duration_hrs > 0 else 100.0
        
        coverage_report = {
            "cache_file": cache_match['filename'],
            "cache_bbox": f"[{cache_match['lat_min']:.4f}, {cache_match['lat_max']:.4f}] x [{cache_match['lon_min']:.4f}, {cache_match['lon_max']:.4f}]",
            "analysis_bbox": f"[{lat_min:.4f}, {lat_max:.4f}] x [{lon_min:.4f}, {lon_max:.4f}]",
            "file_size_kb": os.path.getsize(cache_path) / 1024.0,
            "req_start": req_start_str,
            "req_end": req_end_str,
            "req_duration_hrs": req_duration_hrs,
            "actual_start": actual_start_str,
            "actual_end": actual_end_str,
            "actual_span_hrs": actual_span_hrs,
            "coverage_pct": coverage_pct,
            "num_segments": len(segments),
            "segments": segments,
            "total_waypoints": len(df_slice),
            "unique_aircraft": df_slice['icao24'].nunique() if not df_slice.empty else 0,
            "unique_slices": df_slice['timestamp'].nunique() if not df_slice.empty else 0,
            "avg_noise_dba": df_slice['ground_noise_dba'].mean() if not df_slice.empty else 0.0,
            "max_noise_dba": df_slice['ground_noise_dba'].max() if not df_slice.empty else 0.0
        }
        
        print("=" * 60)
        print("📊 CACHE INGESTION, MULTI-GROUP & ACOUSTIC REPORT")
        print("=" * 60)
        print(f"  • Source Cache File          : {coverage_report['cache_file']} ({coverage_report['file_size_kb']:.1f} KB)")
        print(f"  • Cache Bounding Box Bounds  : {coverage_report['cache_bbox']}")
        print(f"  • Target Analysis Bounding Box: {coverage_report['analysis_bbox']}")
        print(f"  • Requested Time Window      : {coverage_report['req_start']} -> {coverage_report['req_end']} ({coverage_report['req_duration_hrs']:.2f} hrs)")
        print(f"  • Available in Cache         : {coverage_report['actual_start']} -> {coverage_report['actual_end']} ({coverage_report['actual_span_hrs']:.2f} hrs)")
        print(f"  • Temporal Data Coverage     : {coverage_report['coverage_pct']:.1f}%")
        print(f"  • Disjoint Time Segments     : {coverage_report['num_segments']} segment(s)")
        print(f"  • Active Time Slices (Epoch) : {coverage_report['unique_slices']:,}")
        print(f"  • Total Valid Waypoints      : {coverage_report['total_waypoints']:,} ({df_slice.memory_usage(deep=True).sum()/1024/1024:.2f} MB in RAM)")
        print(f"  • Unique Aircraft Captured   : {coverage_report['unique_aircraft']:,}")
        if not df_slice.empty:
            print(f"  • Ground Noise Level (dB(A)) : Avg {coverage_report['avg_noise_dba']:.1f} dB(A), Max {coverage_report['max_noise_dba']:.1f} dB(A)")
            print("  • Active Aircraft Type Groups in Filter:")
            for grp, cnt in df_slice['aircraft_type_group'].value_counts().items():
                print(f"    - {grp:<28}: {cnt:>8,} waypoints ({cnt/len(df_slice)*100:>5.1f}%)")
        print("=" * 60)
        
        return df_slice, coverage_report

reader = CacheDataReader(CACHE_DIR)
df_raw, cache_coverage = reader.load_cached_dataset(
    lat_min=LAT_MIN, lat_max=LAT_MAX,
    lon_min=LON_MIN, lon_max=LON_MAX,
    use_full_history=use_full_history_chk.value,
    time_slider_widget=time_slider,
    time_options_list=time_options,
    start_dt_picker=start_dt_picker,
    end_dt_picker=end_dt_picker,
    selected_groups=group_filter_widget.value,
    model_filter=model_filter_dropdown.value,
    min_noise_dba=noise_threshold_slider.value
)

# ==============================================================================
# 💾 STAGE 3 DISK PERSISTENCE & MEMORY CLEARANCE
# ==============================================================================
stage3_intermediate_path = os.path.join(WORKSPACE_DIR, "stage3_filtered_waypoints.parquet")
raw_waypoints_count = len(df_raw) if not df_raw.empty else 0

if not df_raw.empty:
    df_raw.to_parquet(stage3_intermediate_path, index=False)
    file_size_mb = os.path.getsize(stage3_intermediate_path) / (1024 * 1024)
    print(f"\n💾 Stage 3 Disk Staging Complete:")
    print(f"  • Persisted File   : {stage3_intermediate_path} ({file_size_mb:.2f} MB)")
    print(f"  • Raw Waypoints    : {raw_waypoints_count:,}")
    
    # Explicitly delete df_raw and reader to free memory before Step 4 starts
    del df_raw
    del reader
    gc.collect()
    print("🧹 Stage 3 in-memory DataFrame cleared. RAM is 100% free for Step 4 processing!")
else:
    print("\n⚠️ No waypoints matched filters. Staging file was not created.")

## 4. 🧹 Batched Data Preprocessing & Trajectory Interpolation (Streaming to Disk)
Categorize flight records into 6 altitude tiers, densify flight paths via linear interpolation, and stream results in batches directly to Parquet on disk (`processed_flight_paths.parquet`).
- **🛡️ Constant Low-Memory Footprint**: Processes aircraft in chunks (~150-200 aircraft at a time) and writes directly to disk via `pyarrow.parquet.ParquetWriter`, eliminating Python list explosions and `np.concatenate` memory spikes.
- **⚡ Retains Lightweight Preview in RAM**: Previews up to 60,000 sampled points in `df_processed` for instant browser rendering and statistical plotting without high heap usage.


In [ ]:
from typing import Any, Dict, List, Tuple, Optional, Set
def stream_and_interpolate_trajectories(
    input_source: Any,
    output_parquet_path: str,
    lat_min: float, lat_max: float, lon_min: float, lon_max: float,
    min_alt: float = 0, max_alt: float = 20000,
    exclude_ground: bool = True,
    interpolate: bool = True,
    step_km: float = 0.5,
    batch_size_aircraft: int = 150,
    max_preview_points: int = 60000
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """
    Batched, low-memory trajectory interpolation engine.
    Streams dense interpolated points directly to Parquet on disk to avoid OOM,
    while returning a lightweight sampled DataFrame for interactive browser maps.
    """
    # 1. Ingest raw data from disk staging file or DataFrame
    if isinstance(input_source, str):
        if not os.path.exists(input_source):
            print(f"⚠️ Staging file not found: {input_source}")
            return pd.DataFrame(), {}
        print(f"📂 Reading Stage 3 intermediate data from: {input_source}")
        df_clean = pd.read_parquet(input_source)
    elif isinstance(input_source, pd.DataFrame):
        df_clean = input_source.copy()
    else:
        print("⚠️ Invalid input source provided.")
        return pd.DataFrame(), {}
        
    if df_clean.empty:
        return pd.DataFrame(), {}
        
    raw_point_count = len(df_clean)
    
    # 2. Spatial & Altitude Filtering
    if exclude_ground and 'on_ground' in df_clean.columns:
        df_clean = df_clean[~df_clean['on_ground']]
        
    df_clean = df_clean[
        (df_clean['altitude'] > min_alt) &
        (df_clean['altitude'] <= max_alt)
    ]
    
    buffer = 0.02
    df_clean = df_clean[
        (df_clean['latitude'] >= lat_min - buffer) &
        (df_clean['latitude'] <= lat_max + buffer) &
        (df_clean['longitude'] >= lon_min - buffer) &
        (df_clean['longitude'] <= lon_max + buffer)
    ]
    
    def categorize_altitude_arr(alts_arr: np.ndarray) -> np.ndarray:
        tiers = np.empty(len(alts_arr), dtype=object)
        tiers[alts_arr <= 500] = "0 - 500m (Red)"
        tiers[(alts_arr > 500) & (alts_arr <= 1000)] = "500 - 1000m (Orange)"
        tiers[(alts_arr > 1000) & (alts_arr <= 2000)] = "1000 - 2000m (Amber)"
        tiers[(alts_arr > 2000) & (alts_arr <= 3000)] = "2000 - 3000m (Yellow)"
        tiers[(alts_arr > 3000) & (alts_arr <= 5000)] = "3000 - 5000m (Teal)"
        tiers[alts_arr > 5000] = "5000m+ (Green)"
        return tiers

    if not interpolate:
        df_clean['altitude_tier'] = pd.Categorical(categorize_altitude_arr(df_clean['altitude'].to_numpy(dtype=np.float32)))
        df_clean.to_parquet(output_parquet_path, index=False)
        return df_clean.iloc[::max(1, len(df_clean) // max_preview_points)], {
            "raw_points": raw_point_count,
            "interpolated_points": len(df_clean),
            "parquet_size_mb": os.path.getsize(output_parquet_path) / (1024 * 1024),
            "tier_counts": df_clean['altitude_tier'].value_counts().to_dict(),
            "avg_noise": df_clean['ground_noise_dba'].mean() if 'ground_noise_dba' in df_clean.columns else 0.0,
            "max_noise": df_clean['ground_noise_dba'].max() if 'ground_noise_dba' in df_clean.columns else 0.0,
            "preview_points": min(len(df_clean), max_preview_points)
        }

    step_deg = step_km / 111.0
    unique_icaos = df_clean['icao24'].dropna().unique()
    num_aircraft = len(unique_icaos)
    
    print(f"🚀 Starting Batched Trajectory Interpolation across {num_aircraft:,} aircraft...")
    print(f"  • Interpolation step : {step_km} km ({step_deg:.5f}°)")
    print(f"  • Batch size         : {batch_size_aircraft} aircraft/chunk")
    print(f"  • Direct output file : {output_parquet_path}")

    # Index static metadata per icao24 to eliminate redundant string allocations during point densification
    meta_cols = ['callsign', 'typecode', 'model', 'operator', 'registration', 'categoryDescription', 'aircraft_type_group']
    existing_meta = [c for c in meta_cols if c in df_clean.columns]
    meta_lookup = df_clean.groupby('icao24')[existing_meta].first().to_dict(orient='index')

    # Fixed Arrow Schema to ensure strict schema consistency across all batches
    # (prevents ValueError when Pandas dynamically changes dictionary index bit-width like int8 vs int16)
    FIXED_SCHEMA = pa.schema([
        ('icao24', pa.string()),
        ('callsign', pa.string()),
        ('timestamp', pa.int64()),
        ('latitude', pa.float32()),
        ('longitude', pa.float32()),
        ('altitude', pa.float32()),
        ('vertical_rate', pa.float32()),
        ('velocity', pa.float32()),
        ('ground_noise_dba', pa.float32()),
        ('typecode', pa.string()),
        ('model', pa.string()),
        ('operator', pa.string()),
        ('registration', pa.string()),
        ('categoryDescription', pa.string()),
        ('aircraft_type_group', pa.string()),
        ('altitude_tier', pa.string())
    ])
    schema_cols = [f.name for f in FIXED_SCHEMA]

    temp_parquet_path = output_parquet_path + ".tmp"
    if os.path.exists(temp_parquet_path):
        try:
            os.remove(temp_parquet_path)
        except Exception:
            pass
    num_batches = max(1, math.ceil(num_aircraft / batch_size_aircraft))
    writer = None
    total_interpolated_points = 0
    tier_counts_total = {
        "0 - 500m (Red)": 0,
        "500 - 1000m (Orange)": 0,
        "1000 - 2000m (Amber)": 0,
        "2000 - 3000m (Yellow)": 0,
        "3000 - 5000m (Teal)": 0,
        "5000m+ (Green)": 0
    }
    noise_vals_sum = 0.0
    noise_vals_max = 30.0
    preview_chunks = []
    
    samples_per_batch = max(50, max_preview_points // num_batches)

    try:
        for b_idx in range(num_batches):
            chunk_icaos = unique_icaos[b_idx * batch_size_aircraft : (b_idx + 1) * batch_size_aircraft]
            batch_raw = df_clean[df_clean['icao24'].isin(chunk_icaos)]
            
            b_icaos, b_callsigns, b_times = [], [], []
            b_lats, b_lons, b_alts = [], [], []
            b_vrs, b_vels, b_noises = [], [], []
            b_typecodes, b_models, b_operators = [], [], []
            b_regs, b_cat_descs, b_type_grps = [], [], []
            b_tiers = []
            
            for icao, group in batch_raw.groupby('icao24'):
                group = group.sort_values('timestamp')
                lats = group['latitude'].to_numpy(dtype=np.float32)
                lons = group['longitude'].to_numpy(dtype=np.float32)
                alts = group['altitude'].to_numpy(dtype=np.float32)
                times = group['timestamp'].to_numpy(dtype=np.int64)
                v_rates = group['vertical_rate'].to_numpy(dtype=np.float32) if 'vertical_rate' in group.columns else np.zeros(len(lats), dtype=np.float32)
                vels = group['velocity'].to_numpy(dtype=np.float32) if 'velocity' in group.columns else np.zeros(len(lats), dtype=np.float32)
                
                m_info = meta_lookup.get(icao, {})
                callsign = str(m_info.get('callsign', 'N/A'))
                typecode = str(m_info.get('typecode', 'Unknown'))
                model = str(m_info.get('model', 'Unknown'))
                operator = str(m_info.get('operator', 'Unknown'))
                reg = str(m_info.get('registration', 'Unknown'))
                cat_desc = str(m_info.get('categoryDescription', 'Unknown'))
                type_grp = str(m_info.get('aircraft_type_group', classify_aircraft_type(typecode, model, cat_desc)))
                
                if len(lats) == 1:
                    a0 = alts[0:1]
                    vr0 = v_rates[0:1]
                    v0 = vels[0:1]
                    noise0 = estimate_ground_noise_level_vectorized(a0, vr0, v0, typecode, cat_desc)
                    tier0 = categorize_altitude_arr(a0)
                    b_icaos.append(np.array([icao], dtype=object))
                    b_callsigns.append(np.array([callsign], dtype=object))
                    b_times.append(times[0:1])
                    b_lats.append(lats[0:1])
                    b_lons.append(lons[0:1])
                    b_alts.append(a0)
                    b_vrs.append(vr0)
                    b_vels.append(v0)
                    b_noises.append(noise0)
                    b_typecodes.append(np.array([typecode], dtype=object))
                    b_models.append(np.array([model], dtype=object))
                    b_operators.append(np.array([operator], dtype=object))
                    b_regs.append(np.array([reg], dtype=object))
                    b_cat_descs.append(np.array([cat_desc], dtype=object))
                    b_type_grps.append(np.array([type_grp], dtype=object))
                    b_tiers.append(tier0)
                    continue
                    
                for i in range(len(lats) - 1):
                    lat1, lon1, alt1, t1 = lats[i], lons[i], alts[i], times[i]
                    lat2, lon2, alt2, t2 = lats[i+1], lons[i+1], alts[i+1], times[i+1]
                    vr1, vr2 = v_rates[i], v_rates[i+1]
                    v1, v2 = vels[i], vels[i+1]
                    
                    time_gap = t2 - t1
                    if time_gap > 900 or time_gap < 0:
                        continue
                        
                    dist_deg = math.hypot(lat2 - lat1, lon2 - lon1)
                    num_steps = max(1, int(math.ceil(dist_deg / step_deg)))
                    
                    interp_lats = np.linspace(lat1, lat2, num_steps, endpoint=False, dtype=np.float32)
                    interp_lons = np.linspace(lon1, lon2, num_steps, endpoint=False, dtype=np.float32)
                    interp_alts = np.linspace(alt1, alt2, num_steps, endpoint=False, dtype=np.float32)
                    interp_times = np.linspace(t1, t2, num_steps, endpoint=False, dtype=np.int64)
                    interp_vrs = np.linspace(vr1, vr2, num_steps, endpoint=False, dtype=np.float32)
                    interp_vels = np.linspace(v1, v2, num_steps, endpoint=False, dtype=np.float32)
                    interp_noises = estimate_ground_noise_level_vectorized(interp_alts, interp_vrs, interp_vels, typecode, cat_desc)
                    interp_tiers = categorize_altitude_arr(interp_alts)
                    
                    b_lats.append(interp_lats)
                    b_lons.append(interp_lons)
                    b_alts.append(interp_alts)
                    b_times.append(interp_times)
                    b_vrs.append(interp_vrs)
                    b_vels.append(interp_vels)
                    b_noises.append(interp_noises)
                    b_tiers.append(interp_tiers)
                    
                    b_icaos.append(np.full(num_steps, icao, dtype=object))
                    b_callsigns.append(np.full(num_steps, callsign, dtype=object))
                    b_typecodes.append(np.full(num_steps, typecode, dtype=object))
                    b_models.append(np.full(num_steps, model, dtype=object))
                    b_operators.append(np.full(num_steps, operator, dtype=object))
                    b_regs.append(np.full(num_steps, reg, dtype=object))
                    b_cat_descs.append(np.full(num_steps, cat_desc, dtype=object))
                    b_type_grps.append(np.full(num_steps, type_grp, dtype=object))
                    
                last_alt = alts[-1:]
                last_vr = v_rates[-1:]
                last_vel = vels[-1:]
                last_noise = estimate_ground_noise_level_vectorized(last_alt, last_vr, last_vel, typecode, cat_desc)
                last_tier = categorize_altitude_arr(last_alt)
                
                b_lats.append(lats[-1:])
                b_lons.append(lons[-1:])
                b_alts.append(last_alt)
                b_times.append(times[-1:])
                b_vrs.append(last_vr)
                b_vels.append(last_vel)
                b_noises.append(last_noise)
                b_tiers.append(last_tier)
                
                b_icaos.append(np.array([icao], dtype=object))
                b_callsigns.append(np.array([callsign], dtype=object))
                b_typecodes.append(np.array([typecode], dtype=object))
                b_models.append(np.array([model], dtype=object))
                b_operators.append(np.array([operator], dtype=object))
                b_regs.append(np.array([reg], dtype=object))
                b_cat_descs.append(np.array([cat_desc], dtype=object))
                b_type_grps.append(np.array([type_grp], dtype=object))
                
            if not b_lats:
                del batch_raw
                continue
                
            batch_df = pd.DataFrame({
                'icao24': pd.Categorical(np.concatenate(b_icaos)),
                'callsign': pd.Categorical(np.concatenate(b_callsigns)),
                'timestamp': np.concatenate(b_times),
                'latitude': np.concatenate(b_lats),
                'longitude': np.concatenate(b_lons),
                'altitude': np.concatenate(b_alts),
                'vertical_rate': np.concatenate(b_vrs),
                'velocity': np.concatenate(b_vels),
                'ground_noise_dba': np.concatenate(b_noises),
                'typecode': pd.Categorical(np.concatenate(b_typecodes)),
                'model': pd.Categorical(np.concatenate(b_models)),
                'operator': pd.Categorical(np.concatenate(b_operators)),
                'registration': pd.Categorical(np.concatenate(b_regs)),
                'categoryDescription': pd.Categorical(np.concatenate(b_cat_descs)),
                'aircraft_type_group': pd.Categorical(np.concatenate(b_type_grps)),
                'altitude_tier': pd.Categorical(
                    np.concatenate(b_tiers),
                    categories=["0 - 500m (Red)", "500 - 1000m (Orange)", "1000 - 2000m (Amber)", "2000 - 3000m (Yellow)", "3000 - 5000m (Teal)", "5000m+ (Green)"]
                )
            })
            
            del b_icaos, b_callsigns, b_times, b_lats, b_lons, b_alts, b_vrs, b_vels, b_noises
            del b_typecodes, b_models, b_operators, b_regs, b_cat_descs, b_type_grps, b_tiers
            
            batch_df = batch_df.drop_duplicates(subset=['icao24', 'timestamp', 'latitude', 'longitude'])
            
            # Accumulate overall summary stats across full dataset
            total_interpolated_points += len(batch_df)
            t_counts = batch_df['altitude_tier'].value_counts()
            for t_k, c in t_counts.items():
                tier_counts_total[t_k] = tier_counts_total.get(t_k, 0) + int(c)
            noise_vals_sum += float(batch_df['ground_noise_dba'].sum())
            noise_vals_max = max(noise_vals_max, float(batch_df['ground_noise_dba'].max()))
            
            # Stream to Parquet file on disk with strict schema enforcement (writing to temp file for atomic swap)
            batch_tbl = pa.Table.from_pandas(batch_df[schema_cols], schema=FIXED_SCHEMA, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(temp_parquet_path, FIXED_SCHEMA, compression='snappy')
            writer.write_table(batch_tbl)
            
            # Sample for interactive in-memory preview
            stride = max(1, len(batch_df) // samples_per_batch)
            preview_chunks.append(batch_df.iloc[::stride].copy())
            
            del batch_df, batch_tbl, batch_raw
            gc.collect()
            
            pct = ((b_idx + 1) / num_batches) * 100
            print(f"  • Progress: Batch {b_idx + 1}/{num_batches} ({pct:.0f}%) -> {total_interpolated_points:,} points written...", end='\r')
            
    finally:
        if writer is not None:
            writer.close()
            # Atomically replace final file only when writing and footer are 100% complete
            if os.path.exists(temp_parquet_path):
                os.replace(temp_parquet_path, output_parquet_path)
            
    del df_clean
    gc.collect()
    
    df_preview = pd.concat(preview_chunks, ignore_index=True) if preview_chunks else pd.DataFrame()
    del preview_chunks
    gc.collect()
    
    file_size_mb = os.path.getsize(output_parquet_path) / (1024 * 1024) if os.path.exists(output_parquet_path) else 0
    avg_noise = (noise_vals_sum / total_interpolated_points) if total_interpolated_points > 0 else 0.0
    
    summary_stats = {
        "raw_points": raw_point_count,
        "interpolated_points": total_interpolated_points,
        "parquet_size_mb": file_size_mb,
        "tier_counts": tier_counts_total,
        "avg_noise": avg_noise,
        "max_noise": noise_vals_max,
        "preview_points": len(df_preview)
    }
    
    return df_preview, summary_stats

# Run batched processing with disk streaming
parquet_export_path = os.path.join(WORKSPACE_DIR, "processed_flight_paths.parquet")
stage3_input = stage3_intermediate_path if (
    'stage3_intermediate_path' in globals() and os.path.exists(stage3_intermediate_path)
) else (df_raw if 'df_raw' in globals() else parquet_export_path)

df_processed, interp_stats = stream_and_interpolate_trajectories(
    input_source=stage3_input,
    output_parquet_path=parquet_export_path,
    lat_min=LAT_MIN, lat_max=LAT_MAX,
    lon_min=LON_MIN, lon_max=LON_MAX,
    min_alt=MIN_ALTITUDE_M,
    max_alt=MAX_ALTITUDE_M,
    exclude_ground=EXCLUDE_ON_GROUND,
    interpolate=INTERPOLATE_PATHS,
    step_km=INTERPOLATION_STEP_KM,
    batch_size_aircraft=150,
    max_preview_points=60000
)

print("\n" + "=" * 60)
print("✅ Batched Trajectory Preprocessing & Disk Streaming Complete!")
print("=" * 60)
print(f"  • Points before interpolation   : {interp_stats['raw_points']:,}")
print(f"  • Total Interpolated on Disk    : {interp_stats['interpolated_points']:,} ({interp_stats['parquet_size_mb']:.2f} MB Parquet)")
print(f"  • In-Memory Preview DataFrame   : {len(df_processed):,} points ({df_processed.memory_usage(deep=True).sum()/1024/1024:.2f} MB in RAM)")
if interp_stats['interpolated_points'] > 0:
    print("  • Breakdown by 6 Altitude Tiers (Full Dataset):")
    for tier_name in ["0 - 500m (Red)", "500 - 1000m (Orange)", "1000 - 2000m (Amber)", "2000 - 3000m (Yellow)", "3000 - 5000m (Teal)", "5000m+ (Green)"]:
        cnt = interp_stats['tier_counts'].get(tier_name, 0)
        pct = (cnt / interp_stats['interpolated_points'] * 100) if interp_stats['interpolated_points'] > 0 else 0
        print(f"    - {tier_name:<24}: {cnt:>9,} points ({pct:>5.1f}%)")
    print(f"  • Perceived Ground Noise (dB(A)): Mean {interp_stats['avg_noise']:.1f} dB(A), Max {interp_stats['max_noise']:.1f} dB(A)")
print("=" * 60)


## 5. 🗺️ Interactive Folium Heatmap with Full Dates, Multi-Select Groups & Cascading Filters
Interactive Leaflet/Folium visualization featuring:
- **☀️ CartoDB Positron as Default Base Map** (with Dark Matter, Satellite, OSM, Topo).
- **Client-Side Interactive Date/Time Window & Presets** (`YYYY-MM-DD HH:MM:SS UTC`, `1h`, `3h`, `6h`, `8h`, `12h`, `24h`, `All`, Pan).
- **🛩️ Multi-Select Aircraft Type Group Filter Chips**: Multi-select any combination of operational groups with quick select buttons (`All`, `None`, `Airliners`, `GA/Leisure`, `Helis`).
- **📋 Cascading Airframe Model Dropdown**: Dynamically filters models in real time to match selected groups.
- **🔊 Live Ground Noise Threshold Filter**: Filter flight footprints by minimum estimated ground noise in $\text{dB(A)}$ ($30$ to $85+\text{ dB(A)}$).
- **6 Dedicated Altitude Heatmap Tiers** (Red $0-500$m, Orange $500-1000$m, Amber $1000-2000$m, Yellow $2000-3000$m, Teal $3000-5000$m, Green $5000$+m).


In [ ]:
from typing import Any, Dict, List, Tuple, Optional, Set
def rdp_simplify(coords: np.ndarray, eps: float = 0.0004) -> np.ndarray:
    """Vectorized Ramer-Douglas-Peucker trajectory simplification."""
    if len(coords) <= 3:
        return coords
    p0 = coords[0]
    p1 = coords[-1]
    dx = p1[0] - p0[0]
    dy = p1[1] - p0[1]
    denom = math.hypot(dx, dy)
    if denom == 0:
        return coords[[0, -1]]
    dists = np.abs(dx * (p0[1] - coords[:, 1]) - (p0[0] - coords[:, 0]) * dy) / denom
    max_idx = int(np.argmax(dists))
    if dists[max_idx] > eps:
        left = rdp_simplify(coords[:max_idx + 1], eps)
        right = rdp_simplify(coords[max_idx:], eps)
        return np.vstack([left[:-1], right])
    else:
        return coords[[0, -1]]

def create_altitude_tiered_heatmap_map(
    df: pd.DataFrame,
    lat_min: float, lat_max: float, lon_min: float, lon_max: float,
    region_name: str,
    coverage: Dict[str, Any]
) -> folium.Map:
    center_lat = (lat_min + lat_max) / 2.0
    center_lon = (lon_min + lon_max) / 2.0
    
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=10,
        tiles=None,
        control_scale=True
    )
    
    # 1. Base Map Tile Layers (Zero-Key Low-Contrast Alternatives)
    folium.TileLayer(
        tiles='https://tile.openstreetmap.org/{z}/{x}/{y}.png',
        attr='&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors',
        name='☀️ Muted Grayscale (OSM Mono)',
        className='tiles-muted-mono',
        max_zoom=19,
        show=True
    ).add_to(m)
    folium.TileLayer(
        tiles='https://tile.openstreetmap.org/{z}/{x}/{y}.png',
        attr='&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors',
        name='🌑 Midnight Slate (Dark Canvas)',
        className='tiles-midnight-dark',
        max_zoom=19,
        show=False
    ).add_to(m)
    folium.TileLayer(
        tiles='https://{s}.tile.openstreetmap.fr/hot/{z}/{x}/{y}.png',
        attr='&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors, HOT',
        name='🌸 Soft Pastel (OSM Humanitarian)',
        max_zoom=19,
        show=False
    ).add_to(m)
    folium.TileLayer(
        tiles='https://wmts.geo.admin.ch/1.0.0/ch.swisstopo.pixelkarte-grau/default/current/3857/{z}/{x}/{y}.jpeg',
        attr='&copy; <a href="https://www.swisstopo.admin.ch/">swisstopo</a>',
        name='🇨🇭 Swisstopo Grau (Federal Grey)',
        max_zoom=18,
        show=False
    ).add_to(m)
    folium.TileLayer(
        tiles='https://{s}.tile.opentopomap.org/{z}/{x}/{y}.png',
        attr='Map data: &copy; OpenStreetMap contributors, SRTM | Map style: &copy; OpenTopoMap (CC-BY-SA)',
        name='⛰️ OpenTopoMap (Topographic)',
        max_zoom=17,
        show=False
    ).add_to(m)
    
    # 2. Bounding Box Outline
    folium.Rectangle(
        bounds=[[lat_min, lon_min], [lat_max, lon_max]],
        color="#38bdf8",
        weight=2,
        dash_array="5, 5",
        fill=False,
        tooltip=f"<b>Analysis Region:</b> {region_name}"
    ).add_to(m)

    # 3. Ingest Trajectory Data
    parquet_src = os.path.join(WORKSPACE_DIR, "processed_flight_paths.parquet")
    cols = ['icao24', 'callsign', 'timestamp', 'latitude', 'longitude', 'ground_noise_dba', 'typecode', 'model', 'aircraft_type_group', 'altitude_tier']
    if os.path.exists(parquet_src):
        src_df = pd.read_parquet(parquet_src, columns=cols)
    else:
        src_df = df.copy()
        
    src_df = src_df.sort_values(['icao24', 'timestamp']).reset_index(drop=True)
    src_df['time_gap'] = src_df.groupby('icao24')['timestamp'].diff()
    src_df['flight_id'] = ((src_df['time_gap'] > 300) | (src_df['time_gap'].isna())).cumsum()
    
    flight_stats = src_df.groupby('flight_id').agg(
        n_pts=('timestamp', 'count'),
        t_start=('timestamp', 'min'),
        t_end=('timestamp', 'max'),
        tier=('altitude_tier', 'first'),
        grp=('aircraft_type_group', 'first'),
        tc=('typecode', 'first'),
        noise=('ground_noise_dba', 'max'),
        cs=('callsign', 'first')
    )
    
    valid_mask = flight_stats['n_pts'] >= 4
    valid_flights = flight_stats[valid_mask]
    total_valid = len(valid_flights)
    target_flights = 12000
    sample_step = max(1, total_valid // target_flights)
    sampled_fids = set(valid_flights.index[::sample_step][:target_flights])
    sub_df = src_df[src_df['flight_id'].isin(sampled_fids)]
    
    all_groups = [
        'Commercial Heavy Jet',
        'Commercial Medium Jet',
        'Business Jet',
        'Regional Turboprop',
        'Light Aircraft / Leisure GA',
        'Helicopter / Rotorcraft',
        'Military / Special Ops',
        'Glider / Sailplane',
        'Other / General'
    ]
    group_to_idx = {g: i for i, g in enumerate(all_groups)}
    all_models = ['All Specific Models'] + sorted([t for t in sub_df['typecode'].unique() if t and t != 'Unknown']) + (['Unknown'] if 'Unknown' in sub_df['typecode'].values else [])
    model_to_idx = {m: i for i, m in enumerate(all_models)}
    
    tier_idx_map = {
        "0 - 500m (Red)": 0,
        "500 - 1000m (Orange)": 1,
        "1000 - 2000m (Amber)": 2,
        "2000 - 3000m (Yellow)": 3,
        "3000 - 5000m (Teal)": 4,
        "5000m+ (Green)": 5
    }
    
    model_group_map = {}
    for r in sub_df[['typecode', 'aircraft_type_group']].drop_duplicates().itertuples(index=False):
        tc_name, grp_name = r[0], r[1]
        if tc_name in model_to_idx and grp_name in group_to_idx:
            model_group_map[model_to_idx[tc_name]] = group_to_idx[grp_name]
            
    # Segment by contiguous altitude tiers so flights show their real altitude profiles
    sub_df['seg_id'] = ((sub_df['flight_id'] != sub_df['flight_id'].shift()) | (sub_df['altitude_tier'] != sub_df['altitude_tier'].shift())).cumsum()

    compact_tracks = []
    t_min_total = int(sub_df['timestamp'].min()) if not sub_df.empty else 0
    t_max_total = int(sub_df['timestamp'].max()) if not sub_df.empty else 0
    
    seg_grouped = sub_df.groupby('seg_id')
    for _, grp in seg_grouped:
        if len(grp) < 2:
            continue
        pts = grp[['latitude', 'longitude']].values.round(4)
        sim = rdp_simplify(pts, eps=0.0004)
        
        t_start = int(grp['timestamp'].min())
        t_end = int(grp['timestamp'].max())
        tier_str = str(grp['altitude_tier'].iloc[0])
        tier_idx = tier_idx_map.get(tier_str, 0)
        grp_str = str(grp['aircraft_type_group'].iloc[0])
        grp_idx = group_to_idx.get(grp_str, 8)
        tc = str(grp['typecode'].iloc[0])
        model_idx = model_to_idx.get(tc, 0)
        max_noise = float(grp['ground_noise_dba'].max()) if grp['ground_noise_dba'].notna().any() else 30.0
        cs = str(grp['callsign'].iloc[0]) if grp['callsign'].notna().any() and grp['callsign'].iloc[0] != 'N/A' else ''
        f_id = int(grp['flight_id'].iloc[0]) if ('flight_id' in grp.columns and grp['flight_id'].notna().any()) else 0
        
        compact_tracks.append([
            sim.tolist(),
            t_start,
            t_end,
            tier_idx,
            grp_idx,
            model_idx,
            f_id,
            round(max_noise, 1),
            cs
        ])
        
    actual_start = datetime.fromtimestamp(t_min_total, tz=timezone.utc).strftime("%Y-%m-%d %H:%M UTC") if t_min_total else "N/A"
    actual_end = datetime.fromtimestamp(t_max_total, tz=timezone.utc).strftime("%Y-%m-%d %H:%M UTC") if t_max_total else "N/A"
    
    tracks_json = json.dumps(compact_tracks)
    all_groups_json = json.dumps(all_groups)
    all_models_json = json.dumps(all_models)
    model_group_map_json = json.dumps(model_group_map)

    control_and_banner_html = f"""
    <style>
    /* Clean Low-Contrast Map Tile Styling (Zero API Key) */
    .leaflet-tile-pane .tiles-muted-mono {{
        filter: grayscale(100%) contrast(72%) brightness(108%) !important;
    }}
    .leaflet-tile-pane .tiles-midnight-dark {{
        filter: grayscale(100%) invert(92%) contrast(85%) brightness(95%) !important;
    }}
    .dual-slider-wrap {{
        position: relative;
        width: 100%;
        height: 28px;
        margin: 6px 0;
        user-select: none;
    }}
    .dual-slider-track {{
        position: absolute;
        top: 11px;
        left: 0;
        width: 100%;
        height: 6px;
        background: #334155;
        border-radius: 3px;
    }}
    .dual-slider-progress {{
        position: absolute;
        top: 9px;
        height: 10px;
        background: linear-gradient(90deg, #0284c7, #38bdf8);
        border-radius: 4px;
        left: 0%;
        width: 100%;
        cursor: grab;
        pointer-events: auto;
        z-index: 10;
        box-shadow: 0 0 6px rgba(56, 189, 248, 0.5);
    }}
    .dual-slider-progress:active {{
        cursor: grabbing;
    }}
    .dual-slider-input {{
        position: absolute;
        top: 0;
        left: 0;
        width: 100%;
        height: 28px;
        background: transparent;
        pointer-events: none;
        -webkit-appearance: none;
        appearance: none;
        margin: 0;
        z-index: 20;
    }}
    .dual-slider-input::-webkit-slider-thumb {{
        pointer-events: auto;
        cursor: ew-resize;
        width: 18px;
        height: 18px;
        border-radius: 50%;
        background: #38bdf8;
        border: 2px solid #0f172a;
        box-shadow: 0 0 6px rgba(0,0,0,0.6);
        -webkit-appearance: none;
    }}
    .dual-slider-input::-moz-range-thumb {{
        pointer-events: auto;
        cursor: ew-resize;
        width: 18px;
        height: 18px;
        border-radius: 50%;
        background: #38bdf8;
        border: 2px solid #0f172a;
        box-shadow: 0 0 6px rgba(0,0,0,0.6);
    }}
    .grp-chip {{
        display: inline-flex;
        align-items: center;
        gap: 3px;
        font-size: 9px;
        background: #1e293b;
        color: #94a3b8;
        border: 1px solid #334155;
        border-radius: 4px;
        padding: 2px 6px;
        cursor: pointer;
        transition: all 0.15s ease;
        user-select: none;
    }}
    .grp-chip:hover {{
        background: #334155;
        color: #f8fafc;
    }}
    .grp-chip.active {{
        background: rgba(14, 165, 233, 0.25);
        color: #38bdf8;
        border-color: #38bdf8;
        font-weight: 600;
    }}
    .view-btn {{
        background: #1e293b;
        color: #94a3b8;
        border: 1px solid #334155;
        border-radius: 4px;
        padding: 3px 7px;
        font-size: 10px;
        cursor: pointer;
        transition: all 0.15s ease;
    }}
    .view-btn:hover {{
        background: #334155;
        color: #f8fafc;
    }}
    .view-btn.active {{
        background: #0284c7;
        color: #ffffff;
        border-color: #38bdf8;
        font-weight: bold;
        box-shadow: 0 0 8px rgba(2, 132, 199, 0.5);
    }}
    </style>
    
    <!-- Floating Header Banner & Controller Card -->
    <div style="
        position: fixed;
        top: 15px;
        left: 60px;
        background: rgba(15, 23, 42, 0.95);
        color: #f8fafc;
        z-index: 9999;
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
        border-radius: 8px;
        padding: 12px 16px;
        box-shadow: 0 4px 20px rgba(0,0,0,0.6);
        border: 1px solid rgba(56, 189, 248, 0.3);
        backdrop-filter: blur(8px);
        max-width: 440px;
    ">
        <div style="display: flex; align-items: center; justify-content: space-between; gap: 10px; margin-bottom: 4px;">
            <span style="font-weight: 700; font-size: 13px; color: #38bdf8;">
                ✈️ OpenSky Flight Corridors &mdash; {REGION_NAME}
            </span>
            <span id="pts_counter" style="font-size: 10px; color: #94a3b8; font-family: monospace;">{len(compact_tracks):,} segments</span>
        </div>

        <!-- View Mode Switcher -->
        <div style="margin: 6px 0 8px 0;">
            <div style="font-size: 10px; color: #38bdf8; font-weight: 600; margin-bottom: 3px;">
                🗺️ Raster Density Mode:
            </div>
            <div style="display: flex; flex-wrap: wrap; gap: 3px;">
                <button id="btn_layer_all" class="view-btn active" onclick="setDensityView('all')">🔥 Density Heatmap</button>
                <button id="btn_layer_multi" class="view-btn" onclick="setDensityView('multi')">🌈 Multi-Tier</button>
                <button id="btn_layer_5" class="view-btn" onclick="setDensityView(5)">🟢 5,000+ m</button>
                <button id="btn_layer_4" class="view-btn" onclick="setDensityView(4)">🩵 3,000–5,000 m</button>
                <button id="btn_layer_3" class="view-btn" onclick="setDensityView(3)">🟡 2,000–3,000 m</button>
                <button id="btn_layer_2" class="view-btn" onclick="setDensityView(2)">🟠 1,000–2,000 m</button>
                <button id="btn_layer_1" class="view-btn" onclick="setDensityView(1)">🟧 500–1,000 m</button>
                <button id="btn_layer_0" class="view-btn" onclick="setDensityView(0)">🔴 0–500 m</button>
            </div>
        </div>
        
        <div style="display: flex; flex-direction: column; gap: 4px; font-size: 10px; font-family: monospace; color: #94a3b8;">
            <div style="display: flex; flex-direction: column; gap: 1px;">
                <div>Start: <strong id="lbl_start_t" style="color: #f1f5f9;">{actual_start}</strong></div>
                <div>End: <strong id="lbl_end_t" style="color: #f1f5f9;">{actual_end}</strong></div>
            </div>
            
            <div class="dual-slider-wrap" id="slider_wrap">
                <div class="dual-slider-track"></div>
                <div id="slider_progress" class="dual-slider-progress" title="Drag bar to slide entire time window"></div>
                <input type="range" id="slider_start_t" class="dual-slider-input" min="{t_min_total}" max="{t_max_total}" value="{t_min_total}" step="30" title="Drag Start Time">
                <input type="range" id="slider_end_t" class="dual-slider-input" min="{t_min_total}" max="{t_max_total}" value="{t_max_total}" step="30" title="Drag End Time">
            </div>
            
            <div style="font-size: 9px; color: #64748b; text-align: center; margin: -2px 0 2px 0;">
                💡 Drag thumbs for bounds &bull; Drag center bar to slide window
            </div>
                   
            <div style="display: flex; justify-content: space-between; gap: 3px;">
                <button onclick="shiftTimeWindow('left')" title="Pan window left" style="flex: 0.8; font-size: 9px; padding: 4px 0; background: #1e293b; color: #38bdf8; border: 1px solid #334155; border-radius: 4px; cursor: pointer;">&ltrif; Pan</button>
                <button onclick="setTimePreset('1h')" style="flex: 1; font-size: 9px; padding: 4px 0; background: #1e293b; color: #f8fafc; border: 1px solid #334155; border-radius: 4px; cursor: pointer;">1h</button>
                <button onclick="setTimePreset('3h')" style="flex: 1; font-size: 9px; padding: 4px 0; background: #1e293b; color: #f8fafc; border: 1px solid #334155; border-radius: 4px; cursor: pointer;">3h</button>
                <button onclick="setTimePreset('6h')" style="flex: 1; font-size: 9px; padding: 4px 0; background: #1e293b; color: #f8fafc; border: 1px solid #334155; border-radius: 4px; cursor: pointer;">6h</button>
                <button onclick="setTimePreset('8h')" style="flex: 1; font-size: 9px; padding: 4px 0; background: #1e293b; color: #f8fafc; border: 1px solid #334155; border-radius: 4px; cursor: pointer;">8h</button>
                <button onclick="setTimePreset('12h')" style="flex: 1; font-size: 9px; padding: 4px 0; background: #1e293b; color: #f8fafc; border: 1px solid #334155; border-radius: 4px; cursor: pointer;">12h</button>
                <button onclick="setTimePreset('24h')" style="flex: 1; font-size: 9px; padding: 4px 0; background: #1e293b; color: #f8fafc; border: 1px solid #334155; border-radius: 4px; cursor: pointer;">24h</button>
                <button onclick="setTimePreset('all')" style="flex: 1; font-size: 9px; padding: 4px 0; background: #1e293b; color: #38bdf8; border: 1px solid #334155; border-radius: 4px; cursor: pointer; font-weight: bold;">All</button>
                <button onclick="shiftTimeWindow('right')" title="Pan window right" style="flex: 0.8; font-size: 9px; padding: 4px 0; background: #1e293b; color: #38bdf8; border: 1px solid #334155; border-radius: 4px; cursor: pointer;">Pan &rtrif;</button>
            </div>

            <!-- Corridor Glow & Intensity Slider -->
            <div style="margin-top: 6px; padding-top: 4px; border-top: 1px solid #334155; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
                <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 2px;">
                    <span style="font-size: 10px; color: #ec4899; font-weight: 600;">💥 Corridor Glow &amp; Intensity:</span>
                    <span id="lbl_heat_strength_val" style="font-size: 10px; color: #f8fafc; font-family: monospace; font-weight: bold;">100%</span>
                </div>
                <input type="range" id="slider_heat_strength" min="40" max="250" value="100" step="10" oninput="onHeatmapStrengthInput()" style="width: 100%; accent-color: #ec4899; cursor: pointer; margin: 2px 0;" title="Enhances busy flight corridor width and brilliance relative to background traffic">
                <div style="display: flex; justify-content: space-between; font-size: 8px; color: #64748b; margin-top: -1px;">
                    <span>Subtle (40%)</span>
                    <span>Standard (100%)</span>
                    <span>Vivid (250%)</span>
                </div>
            </div>

            <!-- Ground Noise Threshold Slider -->
            <div style="margin-top: 6px; padding-top: 4px; border-top: 1px solid #334155; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
                <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 2px;">
                    <span style="font-size: 10px; color: #f59e0b; font-weight: 600;">🔊 Min Ground Noise:</span>
                    <span id="lbl_noise_val" style="font-size: 10px; color: #f8fafc; font-family: monospace;">≥ 30 dB(A) (All)</span>
                </div>
                <input type="range" id="filter_noise" min="30" max="85" value="30" step="5" oninput="onNoiseSliderInput()" style="width: 100%; accent-color: #f59e0b; cursor: pointer; margin: 2px 0;">
            </div>

            <!-- Cursor Inspection Radius Slider -->
            <div style="margin-top: 6px; padding-top: 4px; border-top: 1px solid #334155; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
                <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 2px;">
                    <span style="font-size: 10px; color: #38bdf8; font-weight: 600;">🎯 Cursor Inspect Radius:</span>
                    <span id="lbl_hover_radius_val" style="font-size: 10px; color: #f8fafc; font-family: monospace; font-weight: bold;">1.0 km</span>
                </div>
                <input type="range" id="slider_hover_radius" min="0.1" max="10.0" value="1.0" step="0.1" oninput="onHoverRadiusInput(this.value)" style="width: 100%; accent-color: #38bdf8; cursor: pointer; margin: 2px 0;" title="Hover inspection sample radius (0.1 km – 10.0 km)">
                <div style="display: flex; justify-content: space-between; font-size: 8px; color: #64748b; margin-top: -1px;">
                    <span>0.1 km</span>
                    <span>1.0 km (Default)</span>
                    <span>10.0 km</span>
                </div>
            </div>

            <!-- Multi-Select Aircraft Type Groups -->
            <div style="margin-top: 6px; padding-top: 4px; border-top: 1px solid #334155; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
                <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 4px;">
                    <span style="font-size: 10px; color: #38bdf8; font-weight: 600;">🛫 Multi-Select Groups:</span>
                    <div style="display: flex; gap: 3px;">
                        <button onclick="setAllGroups(true)" style="font-size: 8px; padding: 1px 4px; background: #1e293b; color: #38bdf8; border: 1px solid #334155; border-radius: 3px; cursor: pointer;">All</button>
                        <button onclick="setAllGroups(false)" style="font-size: 8px; padding: 1px 4px; background: #1e293b; color: #94a3b8; border: 1px solid #334155; border-radius: 3px; cursor: pointer;">None</button>
                        <button onclick="setGroupPreset('airliners')" style="font-size: 8px; padding: 1px 4px; background: #1e293b; color: #f8fafc; border: 1px solid #334155; border-radius: 3px; cursor: pointer;">Airliners</button>
                        <button onclick="setGroupPreset('ga')" style="font-size: 8px; padding: 1px 4px; background: #1e293b; color: #f8fafc; border: 1px solid #334155; border-radius: 3px; cursor: pointer;">GA/Leisure</button>
                        <button onclick="setGroupPreset('helis')" style="font-size: 8px; padding: 1px 4px; background: #1e293b; color: #f8fafc; border: 1px solid #334155; border-radius: 3px; cursor: pointer;">Helis</button>
                    </div>
                </div>
                <div id="group_chips_container" style="display: flex; flex-wrap: wrap; gap: 4px; max-height: 95px; overflow-y: auto; padding-right: 2px;"></div>
            </div>

            <!-- Cascading Airframe Model Selector -->
            <div style="margin-top: 6px; padding-top: 4px; border-top: 1px solid #334155; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
                <div style="font-size: 10px; color: #38bdf8; font-weight: 600; margin-bottom: 2px;">
                    📋 Filter Specific Aircraft Model:
                </div>
                <select id="filter_model" onchange="onFilterChange()" style="width: 100%; font-size: 10px; background: #1e293b; color: #f8fafc; border: 1px solid #334155; border-radius: 4px; padding: 3px 4px; cursor: pointer;">
                    <option value="0">All Specific Models</option>
                </select>
            </div>
        </div>
    </div>

    <!-- Altitude Tier Legend -->
    <div style="
        position: fixed;
        bottom: 30px;
        right: 30px;
        background: rgba(15, 23, 42, 0.95);
        color: #f8fafc;
        z-index: 9999;
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
        border-radius: 8px;
        padding: 12px 16px;
        box-shadow: 0 4px 20px rgba(0,0,0,0.5);
        border: 1px solid rgba(56, 189, 248, 0.25);
        backdrop-filter: blur(8px);
        min-width: 220px;
    ">
        <div style="font-weight: 700; font-size: 12px; color: #38bdf8; margin-bottom: 8px; border-bottom: 1px solid #334155; padding-bottom: 4px;">
            ✈️ Flight Density Corridors
        </div>
        <div style="display: flex; align-items: center; margin-bottom: 5px; font-size: 11px;">
            <span style="display: inline-block; width: 18px; height: 3px; background: #ff2a2a; margin-right: 8px; border-radius: 2px; box-shadow: 0 0 6px rgba(255,42,42,0.8);"></span>
            <span>🔴 <strong>0 – 500 m</strong> (Surface / Climb)</span>
        </div>
        <div style="display: flex; align-items: center; margin-bottom: 5px; font-size: 11px;">
            <span style="display: inline-block; width: 18px; height: 3px; background: #ff6a00; margin-right: 8px; border-radius: 2px; box-shadow: 0 0 6px rgba(255,106,0,0.8);"></span>
            <span>🟧 <strong>500 – 1,000 m</strong> (Low TMA / Approach)</span>
        </div>
        <div style="display: flex; align-items: center; margin-bottom: 5px; font-size: 11px;">
            <span style="display: inline-block; width: 18px; height: 3px; background: #ffb300; margin-right: 8px; border-radius: 2px; box-shadow: 0 0 6px rgba(255,179,0,0.8);"></span>
            <span>🟠 <strong>1,000 – 2,000 m</strong> (Lower TMA)</span>
        </div>
        <div style="display: flex; align-items: center; margin-bottom: 5px; font-size: 11px;">
            <span style="display: inline-block; width: 18px; height: 3px; background: #fcc419; margin-right: 8px; border-radius: 2px; box-shadow: 0 0 6px rgba(252,196,25,0.8);"></span>
            <span>🟡 <strong>2,000 – 3,000 m</strong> (Intermediate TMA)</span>
        </div>
        <div style="display: flex; align-items: center; margin-bottom: 5px; font-size: 11px;">
            <span style="display: inline-block; width: 18px; height: 3px; background: #00d6b2; margin-right: 8px; border-radius: 2px; box-shadow: 0 0 6px rgba(0,214,178,0.8);"></span>
            <span>🩵 <strong>3,000 – 5,000 m</strong> (Transitions)</span>
        </div>
        <div style="display: flex; align-items: center; margin-bottom: 5px; font-size: 11px;">
            <span style="display: inline-block; width: 18px; height: 3px; background: #00e676; margin-right: 8px; border-radius: 2px; box-shadow: 0 0 6px rgba(0,230,118,0.8);"></span>
            <span>🟢 <strong>5,000+ m</strong> (Cruising Airways)</span>
        </div>
        <div style="font-size: 9px; color: #64748b; border-top: 1px solid #334155; padding-top: 5px; margin-top: 5px; line-height: 1.3;">
            🔥 Silky-smooth Gaussian density corridors compound in real time.
        </div>
    </div>

    <script>
    window.FLIGHT_TRACKS = {tracks_json};
    window.ALL_GROUPS = {all_groups_json};
    window.ALL_MODELS = {all_models_json};
    window.MODEL_GROUP_MAP = {model_group_map_json};
    window.T_MIN = {t_min_total};
    window.T_MAX = {t_max_total};
    window.ACTIVE_GROUPS = new Set(window.ALL_GROUPS.map((_, i) => i));

    var LAT_MIN = {LAT_MIN}, LAT_MAX = {LAT_MAX}, LON_MIN = {LON_MIN}, LON_MAX = {LON_MAX};

    // -------------------------------------------------------------------------
    // Precompute 256-entry Uint32 Palettes (Strictly matching generate_density_heatmap.py)
    // -------------------------------------------------------------------------
    function buildPaletteUint32(stops) {{
        var c = document.createElement('canvas');
        c.width = 256; c.height = 1;
        var ctx = c.getContext('2d');
        var grad = ctx.createLinearGradient(0, 0, 256, 0);
        for (var i = 0; i < stops.length; i++) {{
            grad.addColorStop(stops[i][0], stops[i][1]);
        }}
        ctx.fillStyle = grad;
        ctx.fillRect(0, 0, 256, 1);
        var raw = ctx.getImageData(0, 0, 256, 1).data;
        var lut32 = new Uint32Array(256);
        
        for (var i = 0; i < 256; i++) {{
            var frac = i / 255.0;
            var idx = i * 4;
            var r = raw[idx];
            var g = raw[idx + 1];
            var b = raw[idx + 2];
            
            // Alpha curve: starts at ~46% for pale solitary tracks, smoothly ramps up to ~95% for core corridors
            var alpha = Math.min(242, Math.floor((0.46 + 0.49 * Math.pow(frac, 0.75)) * 255));
            lut32[i] = (alpha << 24) | (b << 16) | (g << 8) | r;
        }}
        return lut32;
    }}

    var PALETTES_32 = {{
        // Red tier: 0 - 500m (Runway approaches & climb): Pale rose -> coral -> deep crimson
        0: buildPaletteUint32([[0.0, '#fca5a5'], [0.25, '#f87171'], [0.55, '#ef4444'], [0.85, '#dc2626'], [1.0, '#991b1b']]),
        // Dark Orange tier: 500 - 1000m (Low Climb / Approach): Pale peach -> warm orange -> deep burnt orange
        1: buildPaletteUint32([[0.0, '#fed7aa'], [0.25, '#fb923c'], [0.55, '#f97316'], [0.85, '#ea580c'], [1.0, '#c2410c']]),
        // Amber tier: 1000 - 2000m (Lower TMA): Pale amber -> golden yellow -> rich amber
        2: buildPaletteUint32([[0.0, '#fef08a'], [0.25, '#facc15'], [0.55, '#eab308'], [0.85, '#d97706'], [1.0, '#b45309']]),
        // Yellow tier: 2000 - 3000m (Intermediate TMA): Pale lemon -> bright yellow -> deep golden
        3: buildPaletteUint32([[0.0, '#fef9c3'], [0.25, '#fde047'], [0.55, '#eab308'], [0.85, '#ca8a04'], [1.0, '#a16207']]),
        // Teal tier: 3000 - 5000m (Upper Transitions): Pale aqua -> mint cyan -> deep teal
        4: buildPaletteUint32([[0.0, '#a5f3fc'], [0.25, '#5eead4'], [0.55, '#14b8a6'], [0.85, '#0d9488'], [1.0, '#0f766e']]),
        // Green tier: 5000m+ (Cruising Airways): Pale mint -> light green -> deep emerald
        5: buildPaletteUint32([[0.0, '#bbf7d0'], [0.25, '#86efac'], [0.55, '#22c55e'], [0.85, '#16a34a'], [1.0, '#15803d']]),
        // All-Altitudes Combined: Pale sky blue -> light cyan -> green -> amber -> fiery red corridor
        'all': buildPaletteUint32([[0.0, '#bfdbfe'], [0.20, '#38bdf8'], [0.40, '#34d399'], [0.65, '#fbbf24'], [0.85, '#f97316'], [1.0, '#dc2626']])
    }};


    // -------------------------------------------------------------------------
    // High-Performance Client-Side Real-Time Raster Density Engine
    // -------------------------------------------------------------------------
    var InteractiveRasterDensityLayer = L.Layer.extend({{
        initialize: function(tracks, options) {{
            this._tracks = tracks;
            this._filteredTracks = tracks;
            this._mode = 'all'; // 'all', 'multi', 0, 1, 2, 3
            this._strength = 1.0;
            this._tierPeaks = {{}};
            this._options = options || {{}};
        }},
        onAdd: function(map) {{
            this._map = map;
            this._canvas = L.DomUtil.create('canvas', 'leaflet-interactive-raster-density');
            this._canvas.style.position = 'absolute';
            this._canvas.style.top = '0';
            this._canvas.style.left = '0';
            this._canvas.style.pointerEvents = 'none';
            this._canvas.style.zIndex = '350';
            map.getPanes().overlayPane.appendChild(this._canvas);
            map.on('moveend resize', this.redraw, this);
            this.redraw();
        }},
        onRemove: function(map) {{
            if (this._canvas && this._canvas.parentNode) {{
                this._canvas.parentNode.removeChild(this._canvas);
            }}
            map.off('moveend resize', this.redraw, this);
        }},
        setMode: function(mode) {{
            this._mode = mode;
            this.redraw();
        }},
        setStrength: function(str) {{
            this._strength = str;
            this.redraw();
        }},
        setFilteredTracks: function(filtered) {{
            this._filteredTracks = filtered;
            this._tierPeaks = {{}};
            this.redraw();
        }},
        computeGlobalPeakA: function(tracksToRender) {{
            var tCount = tracksToRender.length;
            if (tCount === 0) return 0;
            // Stroke alpha scaled so solitary flights retain visible intensity after blur
            var alpha = Math.max(0.045, Math.min(0.22, 2.5 / Math.pow(Math.max(1, tCount), 0.38)));

            var refW = 380;
            var refH = 280;
            var refCanvas = document.createElement('canvas');
            refCanvas.width = refW;
            refCanvas.height = refH;
            var refCtx = refCanvas.getContext('2d');

            refCtx.save();
            refCtx.beginPath();
            refCtx.rect(0, 0, refW, refH);
            refCtx.clip();

            refCtx.lineCap = 'round';
            refCtx.lineJoin = 'round';
            refCtx.lineWidth = 2.0;
            refCtx.strokeStyle = 'rgba(255, 255, 255, ' + alpha.toFixed(4) + ')';

            var lonSpan = LON_MAX - LON_MIN;
            var latSpan = LAT_MAX - LAT_MIN;

            for (var i = 0; i < tCount; i++) {{
                var coords = tracksToRender[i][0];
                if (!coords || coords.length < 2) continue;

                refCtx.beginPath();
                var x0 = ((coords[0][1] - LON_MIN) / lonSpan) * refW;
                var y0 = ((LAT_MAX - coords[0][0]) / latSpan) * refH;
                refCtx.moveTo(x0, y0);
                for (var j = 1; j < coords.length; j++) {{
                    var xj = ((coords[j][1] - LON_MIN) / lonSpan) * refW;
                    var yj = ((LAT_MAX - coords[j][0]) / latSpan) * refH;
                    refCtx.lineTo(xj, yj);
                }}
                refCtx.stroke();
            }}
            refCtx.restore();

            var blurCanvas = document.createElement('canvas');
            blurCanvas.width = refW;
            blurCanvas.height = refH;
            var bCtx = blurCanvas.getContext('2d');
            bCtx.filter = 'blur(2.6px)';
            bCtx.drawImage(refCanvas, 0, 0);

            var imgData = bCtx.getImageData(0, 0, refW, refH);
            var src = imgData.data;
            var maxVal = 0;
            var len = refW * refH;
            for (var p = 0; p < len; p++) {{
                var a = src[p * 4 + 3];
                if (a > maxVal) maxVal = a;
            }}
            return maxVal;
        }},
        redraw: function() {{
            if (!this._map || !this._canvas) return;
            var tracks = this._filteredTracks || this._tracks;
            if (!tracks) return;

            var size = this._map.getSize();
            var padFactor = 0.35;
            var padX = Math.round(size.x * padFactor);
            var padY = Math.round(size.y * padFactor);
            var canvasW = size.x + 2 * padX;
            var canvasH = size.y + 2 * padY;

            this._canvas.width = canvasW;
            this._canvas.height = canvasH;
            var topLeft = this._map.containerPointToLayerPoint([-padX, -padY]);
            L.DomUtil.setPosition(this._canvas, topLeft);

            var ctx = this._canvas.getContext('2d');
            ctx.clearRect(0, 0, canvasW, canvasH);

            var count = tracks.length;
            if (count === 0) return;

            var zoom = this._map.getZoom();
            // Gentle multi-zoom blur merges corridors smoothly without dispersing thin solitary lines
            var blurPx = Math.max(1.8, Math.min(5.5, 3.6 - (zoom - 10) * 0.5));
            var lineW = Math.max(1.6, (2.0 + Math.max(0, zoom - 10) * 0.35));

            // Clip to analysis bounding box to prevent boundary bleed
            var pMin = this._map.latLngToContainerPoint([LAT_MAX, LON_MIN]);
            var pMax = this._map.latLngToContainerPoint([LAT_MIN, LON_MAX]);
            var boxW = pMax.x - pMin.x;
            var boxH = pMax.y - pMin.y;

            var self = this;
            function renderTracksToCanvas(tracksToRender, lutToUse, tierKey) {{
                var tCount = tracksToRender.length;
                if (tCount === 0) return null;

                // Scale stroke alpha smoothly so both solitary flights and heavy corridors have optimal contrast
                var alpha = Math.max(0.045, Math.min(0.22, 2.5 / Math.pow(Math.max(1, tCount), 0.38)));

                var offCanvas = document.createElement('canvas');
                offCanvas.width = canvasW;
                offCanvas.height = canvasH;
                var offCtx = offCanvas.getContext('2d');

                offCtx.save();
                offCtx.beginPath();
                offCtx.rect(pMin.x + padX, pMin.y + padY, boxW, boxH);
                offCtx.clip();

                offCtx.lineCap = 'round';
                offCtx.lineJoin = 'round';
                offCtx.lineWidth = lineW;
                offCtx.strokeStyle = 'rgba(255, 255, 255, ' + alpha.toFixed(4) + ')';

                for (var i = 0; i < tCount; i++) {{
                    var coords = tracksToRender[i][0];
                    if (!coords || coords.length < 2) continue;

                    offCtx.beginPath();
                    var p0 = self._map.latLngToContainerPoint(coords[0]);
                    offCtx.moveTo(p0.x + padX, p0.y + padY);
                    for (var j = 1; j < coords.length; j++) {{
                        var pj = self._map.latLngToContainerPoint(coords[j]);
                        offCtx.lineTo(pj.x + padX, pj.y + padY);
                    }}
                    offCtx.stroke();
                }}
                offCtx.restore();

                var blurCanvas = document.createElement('canvas');
                blurCanvas.width = canvasW;
                blurCanvas.height = canvasH;
                var bCtx = blurCanvas.getContext('2d');

                bCtx.save();
                bCtx.beginPath();
                bCtx.rect(pMin.x + padX, pMin.y + padY, boxW, boxH);
                bCtx.clip();
                bCtx.filter = 'blur(' + blurPx.toFixed(1) + 'px)';
                bCtx.drawImage(offCanvas, 0, 0);
                bCtx.restore();

                var imgData = bCtx.getImageData(0, 0, canvasW, canvasH);
                var srcBytes = imgData.data;
                var numPixels = canvasW * canvasH;

                // Stable global peak density calculation (independent of viewport pan position)
                if (!self._tierPeaks) self._tierPeaks = {{}};
                if (self._tierPeaks[tierKey] === undefined) {{
                    self._tierPeaks[tierKey] = self.computeGlobalPeakA(tracksToRender);
                }}
                var peakA = self._tierPeaks[tierKey];
                if (peakA < 1) return null;

                var outCanvas = document.createElement('canvas');
                outCanvas.width = canvasW;
                outCanvas.height = canvasH;
                var outCtx = outCanvas.getContext('2d');
                var outImg = outCtx.createImageData(canvasW, canvasH);
                var outView32 = new Uint32Array(outImg.data.buffer);

                // Effective peak with soft floor: small traffic is pale & visible, corridors are vivid & hot
                var effectivePeak = Math.max(20.0, peakA);
                var str = self._strength;

                for (var p = 0; p < numPixels; p++) {{
                    var a = srcBytes[p * 4 + 3];
                    if (a < 1) continue; // Show all traffic! No destructive cutoff
                    var ratio = Math.min(1.0, a / effectivePeak);
                    // Power curve: lifts low densities smoothly so solitary traffic is pale but clearly visible
                    var boostedNorm = Math.min(1.0, Math.pow(ratio, 0.42) * str);
                    var lutIdx = Math.min(255, Math.floor(boostedNorm * 255));
                    var pixel = lutToUse[lutIdx];
                    if (pixel !== 0) outView32[p] = pixel;
                }}
                outCtx.putImageData(outImg, 0, 0);
                return outCanvas;
            }}

            function renderMultiTierBlend(tracksToRender) {{
                var tCount = tracksToRender.length;
                if (tCount === 0) return null;

                // Stroke alpha scaled identically to Density Heatmap for consistent density representation
                var alpha = Math.max(0.04, Math.min(0.20, 2.2 / Math.pow(Math.max(1, tCount), 0.38)));

                var offCanvas = document.createElement('canvas');
                offCanvas.width = canvasW;
                offCanvas.height = canvasH;
                var offCtx = offCanvas.getContext('2d');

                var blurCanvas = document.createElement('canvas');
                blurCanvas.width = canvasW;
                blurCanvas.height = canvasH;
                var bCtx = blurCanvas.getContext('2d');

                var numPixels = canvasW * canvasH;

                // Partition tracks by altitude tier 0..5
                var tierTracks = [[], [], [], [], [], []];
                for (var i = 0; i < tCount; i++) {{
                    var tr = tracksToRender[i];
                    var t = tr[3];
                    if (t >= 0 && t <= 5) tierTracks[t].push(tr);
                }}

                // Render each tier onto offCanvas, blur with standard heatmap radius, and extract alpha density
                var tierDensities = [];
                for (var ti = 0; ti < 6; ti++) {{
                    var tracksInTier = tierTracks[ti];
                    if (tracksInTier.length === 0) {{
                        tierDensities.push(null);
                        continue;
                    }}

                    offCtx.clearRect(0, 0, canvasW, canvasH);
                    offCtx.save();
                    offCtx.beginPath();
                    offCtx.rect(pMin.x + padX, pMin.y + padY, boxW, boxH);
                    offCtx.clip();
                    offCtx.lineCap = 'round';
                    offCtx.lineJoin = 'round';
                    offCtx.lineWidth = lineW;
                    offCtx.strokeStyle = 'rgba(255, 255, 255, ' + alpha.toFixed(4) + ')';

                    for (var k = 0; k < tracksInTier.length; k++) {{
                        var coords = tracksInTier[k][0];
                        if (!coords || coords.length < 2) continue;
                        offCtx.beginPath();
                        var p0 = self._map.latLngToContainerPoint(coords[0]);
                        offCtx.moveTo(p0.x + padX, p0.y + padY);
                        for (var j = 1; j < coords.length; j++) {{
                            var pj = self._map.latLngToContainerPoint(coords[j]);
                            offCtx.lineTo(pj.x + padX, pj.y + padY);
                        }}
                        offCtx.stroke();
                    }}
                    offCtx.restore();

                    bCtx.clearRect(0, 0, canvasW, canvasH);
                    bCtx.save();
                    bCtx.beginPath();
                    bCtx.rect(pMin.x + padX, pMin.y + padY, boxW, boxH);
                    bCtx.clip();
                    bCtx.filter = 'blur(' + blurPx.toFixed(1) + 'px)';
                    bCtx.drawImage(offCanvas, 0, 0);
                    bCtx.restore();

                    var imgData = bCtx.getImageData(0, 0, canvasW, canvasH);
                    var src = imgData.data;
                    var tArr = new Uint8Array(numPixels);
                    for (var p = 0, s = 3; p < numPixels; p++, s += 4) {{
                        tArr[p] = src[s];
                    }}
                    tierDensities.push(tArr);
                }}

                // Stable global peak density calculation matching Density Heatmap
                if (!self._tierPeaks) self._tierPeaks = {{}};
                if (self._tierPeaks['all'] === undefined) {{
                    self._tierPeaks['all'] = self.computeGlobalPeakA(tracksToRender);
                }}
                var peakA = Math.max(25.0, self._tierPeaks['all'] || 50.0);

                var outCanvas = document.createElement('canvas');
                outCanvas.width = canvasW;
                outCanvas.height = canvasH;
                var outCtx = outCanvas.getContext('2d');
                var outImg = outCtx.createImageData(canvasW, canvasH);
                var outView32 = new Uint32Array(outImg.data.buffer);

                var str = self._strength;
                var d0Arr = tierDensities[0];
                var d1Arr = tierDensities[1];
                var d2Arr = tierDensities[2];
                var d3Arr = tierDensities[3];
                var d4Arr = tierDensities[4];
                var d5Arr = tierDensities[5];

                for (var p = 0; p < numPixels; p++) {{
                    var d0 = d0Arr ? d0Arr[p] : 0;
                    var d1 = d1Arr ? d1Arr[p] : 0;
                    var d2 = d2Arr ? d2Arr[p] : 0;
                    var d3 = d3Arr ? d3Arr[p] : 0;
                    var d4 = d4Arr ? d4Arr[p] : 0;
                    var d5 = d5Arr ? d5Arr[p] : 0;

                    var totalD = d0 + d1 + d2 + d3 + d4 + d5;
                    if (totalD < 1) continue; // Keep empty airspace 100% transparent

                    // Proportional color blend of all altitude tiers present above this area:
                    // 0: Red (#ef4444), 1: Orange (#f97316), 2: Amber (#f59e0b),
                    // 3: Yellow (#eab308), 4: Cyan (#06b6d4), 5: Green (#22c55e)
                    var r = Math.round((d0 * 239 + d1 * 249 + d2 * 245 + d3 * 234 + d4 * 6 + d5 * 34) / totalD);
                    var g = Math.round((d0 * 68 + d1 * 115 + d2 * 158 + d3 * 179 + d4 * 182 + d5 * 197) / totalD);
                    var b = Math.round((d0 * 68 + d1 * 22 + d2 * 11 + d3 * 8 + d4 * 212 + d5 * 94) / totalD);

                    // Intensity reflecting the relative number of flights (total density vs peak)
                    var norm = Math.min(1.0, totalD / peakA);
                    var boosted = Math.min(1.0, Math.pow(norm, 0.52) * str);

                    // Solitary traffic: soft pale tint (~28-35% opacity) clearly visible across all areas
                    // High-density corridors: vivid, saturated color and full solid presence (~95% opacity)
                    var pale = (1.0 - boosted) * 0.22;
                    var rAdj = Math.min(255, Math.round(r + (255 - r) * pale));
                    var gAdj = Math.min(255, Math.round(g + (255 - g) * pale));
                    var bAdj = Math.min(255, Math.round(b + (255 - b) * pale));

                    var alphaVal = Math.min(242, Math.floor((0.26 + 0.70 * boosted) * 255));
                    outView32[p] = (alphaVal << 24) | (bAdj << 16) | (gAdj << 8) | rAdj;
                }}

                outCtx.putImageData(outImg, 0, 0);
                return outCanvas;
            }}

            if (this._mode === 'multi') {{
                var layerCanvas = renderMultiTierBlend(tracks);
                if (layerCanvas) ctx.drawImage(layerCanvas, 0, 0);
            }} else if (this._mode === 'all') {{
                var layerCanvas = renderTracksToCanvas(tracks, PALETTES_32['all'], 'all');
                if (layerCanvas) ctx.drawImage(layerCanvas, 0, 0);
            }} else {{
                var tierNum = parseInt(this._mode);
                var tTracks = tracks.filter(tr => tr[3] === tierNum);
                var layerCanvas = renderTracksToCanvas(tTracks, PALETTES_32[tierNum], tierNum);
                if (layerCanvas) ctx.drawImage(layerCanvas, 0, 0);
            }}
        }}
    }});

    function setDensityView(mode) {{
        window.CURRENT_LAYER_MODE = mode;
        document.querySelectorAll('.view-btn').forEach(b => b.classList.remove('active'));
        var btnId = 'btn_layer_' + mode;
        var btn = document.getElementById(btnId);
        if (btn) btn.classList.add('active');
        if (window.densityLayer) {{
            window.densityLayer.setMode(mode);
        }}
        if (window.hoverCardEl && window.hoverCardEl.style.display !== 'none' && window.CURRENT_MOUSE_LATLNG) {{
            computeAndShowHoverStats(window.CURRENT_MOUSE_LATLNG, window.CURRENT_MOUSE_CLIENT_X, window.CURRENT_MOUSE_CLIENT_Y);
        }}
    }}

    function formatTimestampUTC(ts) {{
        let d = new Date(ts * 1000);
        return d.toISOString().replace("T", " ").substring(0, 19) + " UTC";
    }}

    function updateSliderUI(s_val, e_val) {{
        let totalSpan = window.T_MAX - window.T_MIN;
        let s_pct = ((s_val - window.T_MIN) / totalSpan) * 100;
        let e_pct = ((e_val - window.T_MIN) / totalSpan) * 100;
        
        let prog = document.getElementById("slider_progress");
        if (prog) {{
            prog.style.left = s_pct + "%";
            prog.style.width = (e_pct - s_pct) + "%";
        }}
        
        let lbl_s = document.getElementById("lbl_start_t");
        let lbl_e = document.getElementById("lbl_end_t");
        if (lbl_s) lbl_s.innerText = formatTimestampUTC(s_val);
        if (lbl_e) lbl_e.innerText = formatTimestampUTC(e_val);
    }}

    function filterPoints(min_ts, max_ts) {{
        let sel_model = 0;
        let min_noise = 30.0;
        
        let model_el = document.getElementById("filter_model");
        let noise_el = document.getElementById("filter_noise");
        
        if (model_el) sel_model = parseInt(model_el.value);
        if (noise_el) min_noise = parseFloat(noise_el.value);

        let matchingTracks = [];
        let totalPts = 0;
        let raw = window.FLIGHT_TRACKS;
        let rawLen = raw.length;

        for (let i = 0; i < rawLen; i++) {{
            let tr = raw[i];
            let t_start = tr[1];
            let t_end = tr[2];
            let group_idx = tr[4];
            let model_idx = tr[5];
            let noise_dba = tr[7];

            if (t_end >= min_ts && t_start <= max_ts && noise_dba >= min_noise) {{
                if (window.ACTIVE_GROUPS.has(group_idx)) {{
                    if (sel_model === 0 || model_idx === sel_model) {{
                        matchingTracks.push(tr);
                        totalPts += tr[0].length;
                    }}
                }}
            }}
        }}
        
        if (window.densityLayer) {{
            window.densityLayer.setFilteredTracks(matchingTracks);
        }}
        window.CURRENT_FILTERED_TRACKS = matchingTracks;
        if (window.hoverCardEl && window.hoverCardEl.style.display !== 'none' && window.CURRENT_MOUSE_LATLNG) {{
            computeAndShowHoverStats(window.CURRENT_MOUSE_LATLNG, window.CURRENT_MOUSE_CLIENT_X, window.CURRENT_MOUSE_CLIENT_Y);
        }}
        
        let counter = document.getElementById("pts_counter");
        if (counter) counter.innerText = matchingTracks.length.toLocaleString() + " segments (" + totalPts.toLocaleString() + " pts)";
    }}

    function onNoiseSliderInput() {{
        let noise_el = document.getElementById("filter_noise");
        let lbl = document.getElementById("lbl_noise_val");
        if (noise_el && lbl) {{
            let val = parseInt(noise_el.value);
            if (val <= 30) lbl.innerText = "≥ 30 dB(A) (All)";
            else if (val >= 75) lbl.innerText = "≥ " + val + " dB(A) (Peak)";
            else lbl.innerText = "≥ " + val + " dB(A)";
        }}
        onFilterChange();
    }}

    function onHeatmapStrengthInput() {{
        let el = document.getElementById("slider_heat_strength");
        let lbl = document.getElementById("lbl_heat_strength_val");
        if (!el) return;
        let factor = parseInt(el.value);
        if (lbl) lbl.innerText = factor + "%";
        
        let scale = factor / 100.0;
        if (window.densityLayer) {{
            window.densityLayer.setStrength(scale);
        }}
    }}

    function onFilterChange() {{
        let s_slider = document.getElementById("slider_start_t");
        let e_slider = document.getElementById("slider_end_t");
        let s_val = s_slider ? parseInt(s_slider.value) : window.T_MIN;
        let e_val = e_slider ? parseInt(e_slider.value) : window.T_MAX;
        filterPoints(s_val, e_val);
    }}

    function setTimePreset(preset) {{
        let min_t = window.T_MIN;
        let max_t = window.T_MAX;
        let s_slider = document.getElementById("slider_start_t");
        let e_slider = document.getElementById("slider_end_t");
        
        let s_val = min_t;
        let e_val = max_t;
        
        if (preset === '1h') {{
            s_val = Math.max(min_t, max_t - 3600); e_val = max_t;
        }} else if (preset === '3h') {{
            s_val = Math.max(min_t, max_t - 10800); e_val = max_t;
        }} else if (preset === '6h') {{
            s_val = Math.max(min_t, max_t - 21600); e_val = max_t;
        }} else if (preset === '8h') {{
            s_val = Math.max(min_t, max_t - 28800); e_val = max_t;
        }} else if (preset === '12h') {{
            s_val = Math.max(min_t, max_t - 43200); e_val = max_t;
        }} else if (preset === '24h') {{
            s_val = Math.max(min_t, max_t - 86400); e_val = max_t;
        }} else if (preset === 'all') {{
            s_val = min_t; e_val = max_t;
        }}
        
        if (s_slider && e_slider) {{
            s_slider.value = s_val;
            e_slider.value = e_val;
            updateSliderUI(s_val, e_val);
            filterPoints(s_val, e_val);
        }}
    }}

    function shiftTimeWindow(direction) {{
        let s_slider = document.getElementById("slider_start_t");
        let e_slider = document.getElementById("slider_end_t");
        if (!s_slider || !e_slider) return;
        
        let s = parseInt(s_slider.value);
        let e = parseInt(e_slider.value);
        let span = e - s;
        let step = Math.max(60, Math.floor(span / 4));
        
        if (direction === 'left') {{
            s = Math.max(window.T_MIN, s - step);
            e = s + span;
        }} else {{
            e = Math.min(window.T_MAX, e + step);
            s = e - span;
        }}
        
        s_slider.value = s;
        e_slider.value = e;
        updateSliderUI(s, e);
        filterPoints(s, e);
    }}

    function populateGroupChips() {{
        let container = document.getElementById("group_chips_container");
        if (!container) return;
        container.innerHTML = "";
        
        window.ALL_GROUPS.forEach((name, idx) => {{
            let chip = document.createElement("div");
            let isActive = window.ACTIVE_GROUPS.has(idx);
            chip.className = "grp-chip" + (isActive ? " active" : "");
            chip.id = "chip_grp_" + idx;
            chip.innerHTML = (isActive ? "☑ " : "☐ ") + name;
            chip.onclick = function() {{
                if (window.ACTIVE_GROUPS.has(idx)) {{
                    window.ACTIVE_GROUPS.delete(idx);
                }} else {{
                    window.ACTIVE_GROUPS.add(idx);
                }}
                updateGroupChipsUI();
                updateCascadingModels();
                onFilterChange();
            }};
            container.appendChild(chip);
        }});
    }}

    function updateGroupChipsUI() {{
        window.ALL_GROUPS.forEach((name, idx) => {{
            let chip = document.getElementById("chip_grp_" + idx);
            if (chip) {{
                let isActive = window.ACTIVE_GROUPS.has(idx);
                chip.className = "grp-chip" + (isActive ? " active" : "");
                chip.innerHTML = (isActive ? "☑ " : "☐ ") + name;
            }}
        }});
    }}

    function setAllGroups(enableAll) {{
        if (enableAll) {{
            window.ACTIVE_GROUPS = new Set(window.ALL_GROUPS.map((_, i) => i));
        }} else {{
            window.ACTIVE_GROUPS.clear();
        }}
        updateGroupChipsUI();
        updateCascadingModels();
        onFilterChange();
    }}

    function setGroupPreset(preset) {{
        window.ACTIVE_GROUPS.clear();
        if (preset === 'airliners') {{
            window.ACTIVE_GROUPS.add(0);
            window.ACTIVE_GROUPS.add(1);
        }} else if (preset === 'ga') {{
            window.ACTIVE_GROUPS.add(3);
            window.ACTIVE_GROUPS.add(4);
        }} else if (preset === 'helis') {{
            window.ACTIVE_GROUPS.add(5);
        }}
        updateGroupChipsUI();
        updateCascadingModels();
        onFilterChange();
    }}

    function updateCascadingModels() {{
        let model_el = document.getElementById("filter_model");
        if (!model_el) return;
        
        let prevVal = parseInt(model_el.value) || 0;
        let availableModelIndices = new Set();
        for (let mIdx in window.MODEL_GROUP_MAP) {{
            let grpIdx = window.MODEL_GROUP_MAP[mIdx];
            if (window.ACTIVE_GROUPS.has(grpIdx)) {{
                availableModelIndices.add(parseInt(mIdx));
            }}
        }}
        
        let html = '<option value="0">All Specific Models</option>';
        let isPrevValid = false;
        
        window.ALL_MODELS.forEach((mName, mIdx) => {{
            if (mIdx > 0 && availableModelIndices.has(mIdx)) {{
                if (mIdx === prevVal) isPrevValid = true;
                html += '<option value="' + mIdx + '">' + mName + '</option>';
            }}
        }});
        
        model_el.innerHTML = html;
        model_el.value = isPrevValid ? prevVal : 0;
    }}


    // -------------------------------------------------------------------------
    // Hover Inspection Stats Engine (1s Inactivity Trigger, Dynamic Radius)
    // -------------------------------------------------------------------------
    window.HOVER_RADIUS_KM = 1.0;
    window.CURRENT_LAYER_MODE = 'all';
    window.CURRENT_FILTERED_TRACKS = window.FLIGHT_TRACKS || [];
    window.CURRENT_MOUSE_LATLNG = null;
    window.CURRENT_MOUSE_CLIENT_X = 0;
    window.CURRENT_MOUSE_CLIENT_Y = 0;
    window.hoverTimer = null;
    window.hoverCircle = null;
    window.hoverCardEl = null;

    function precomputeTrackBBoxes() {{
        let tracks = window.FLIGHT_TRACKS;
        if (!tracks) return;
        for (let i = 0; i < tracks.length; i++) {{
            let pts = tracks[i][0];
            if (!pts || pts.length === 0) continue;
            let minL = pts[0][0], maxL = pts[0][0], minO = pts[0][1], maxO = pts[0][1];
            for (let p = 1; p < pts.length; p++) {{
                let lat = pts[p][0], lon = pts[p][1];
                if (lat < minL) minL = lat;
                if (lat > maxL) maxL = lat;
                if (lon < minO) minO = lon;
                if (lon > maxO) maxO = lon;
            }}
            tracks[i]._bbox = [minL, maxL, minO, maxO];
        }}
    }}

    function onHoverRadiusInput(val) {{
        let r = parseFloat(val);
        if (isNaN(r) || r < 0.1) r = 0.1;
        if (r > 10.0) r = 10.0;
        window.HOVER_RADIUS_KM = r;
        let lbl = document.getElementById("lbl_hover_radius_val");
        if (lbl) lbl.innerText = r.toFixed(1) + " km";
        if (window.hoverCircle && window.CURRENT_MOUSE_LATLNG) {{
            window.hoverCircle.setRadius(r * 1000);
        }}
        if (window.hoverCardEl && window.hoverCardEl.style.display !== 'none' && window.CURRENT_MOUSE_LATLNG) {{
            computeAndShowHoverStats(
                window.CURRENT_MOUSE_LATLNG,
                window.CURRENT_MOUSE_CLIENT_X,
                window.CURRENT_MOUSE_CLIENT_Y
            );
        }}
    }}

    function hideHoverCard() {{
        if (window.hoverCardEl) {{
            window.hoverCardEl.style.display = 'none';
        }}
        if (window.hoverCircle) {{
            window.hoverCircle.setStyle({{ opacity: 0, fillOpacity: 0 }});
        }}
    }}

    function positionHoverCard(screenX, screenY) {{
        let card = window.hoverCardEl;
        if (!card) return;
        
        let cardW = card.offsetWidth || 280;
        let cardH = card.offsetHeight || 300;
        
        let posX = screenX + 16;
        let posY = screenY + 16;
        
        if (posX + cardW + 16 > window.innerWidth) {{
            posX = screenX - cardW - 16;
        }}
        if (posX < 8) posX = 8;
        
        if (posY + cardH + 16 > window.innerHeight) {{
            posY = screenY - cardH - 16;
        }}
        if (posY < 8) posY = 8;
        
        card.style.left = posX + 'px';
        card.style.top = posY + 'px';
    }}

    function renderHoverStatsHTML(lat, lon, r_km, activeMode, tierFilter, flightCount, meanMaxNoise, peakNoise, altBreakdown, noiseBreakdown) {{
        const TIER_BADGES = {{
            0: '🔴 0–500m',
            1: '🟧 500–1km',
            2: '🟠 1k–2km',
            3: '🟡 2k–3km',
            4: '🩵 3k–5km',
            5: '🟢 5k+ m',
            'all': 'All Altitudes',
            'multi': 'Multi-Tier'
        }};
        const layerBadge = TIER_BADGES[activeMode] || 'All Altitudes';

        if (flightCount === 0) {{
            return '<div style="display: flex; align-items: center; justify-content: space-between; margin-bottom: 4px; border-bottom: 1px solid #334155; padding-bottom: 4px;">' +
                   '<span style="font-weight: 700; font-size: 11px; color: #38bdf8;">🎯 Local Area Inspection</span>' +
                   '<div style="display: flex; gap: 4px; align-items: center;">' +
                   '<span style="font-size: 9px; color: #38bdf8; background: rgba(56, 189, 248, 0.15); border: 1px solid rgba(56, 189, 248, 0.3); border-radius: 3px; padding: 1px 4px; font-family: monospace;">r = ' + r_km.toFixed(1) + ' km</span>' +
                   '<span style="font-size: 9px; color: #f8fafc; background: rgba(30, 41, 59, 0.9); border: 1px solid #475569; border-radius: 3px; padding: 1px 4px; font-family: monospace;">' + layerBadge + '</span>' +
                   '</div>' +
                   '</div>' +
                   '<div style="font-size: 9px; color: #64748b; margin-bottom: 6px; font-family: monospace;">' +
                   'Lat: ' + lat.toFixed(4) + '° N, Lon: ' + lon.toFixed(4) + '° E' +
                   '</div>' +
                   '<div style="padding: 10px 8px; text-align: center; color: #94a3b8; font-size: 11px;">' +
                   '<div style="font-size: 18px; margin-bottom: 4px;">📭</div>' +
                   '<div><b>No flights in ' + layerBadge + ' within ' + r_km.toFixed(1) + ' km</b></div>' +
                   '<div style="font-size: 9px; color: #64748b; margin-top: 3px;">Try switching to All Altitudes or adjusting filters</div>' +
                   '</div>';
        }}

        let noiseColor = '#22c55e';
        if (meanMaxNoise >= 75) noiseColor = '#ef4444';
        else if (meanMaxNoise >= 65) noiseColor = '#f97316';
        else if (meanMaxNoise >= 55) noiseColor = '#facc15';

        let html = '<div style="display: flex; align-items: center; justify-content: space-between; margin-bottom: 4px; border-bottom: 1px solid #334155; padding-bottom: 4px;">' +
                   '<span style="font-weight: 700; font-size: 11px; color: #38bdf8;">🎯 Local Flight Footprint</span>' +
                   '<div style="display: flex; gap: 4px; align-items: center;">' +
                   '<span style="font-size: 9px; color: #38bdf8; background: rgba(56, 189, 248, 0.15); border: 1px solid rgba(56, 189, 248, 0.3); border-radius: 3px; padding: 1px 4px; font-family: monospace;">r = ' + r_km.toFixed(1) + ' km</span>' +
                   '<span style="font-size: 9px; color: #f8fafc; background: rgba(30, 41, 59, 0.9); border: 1px solid #475569; border-radius: 3px; padding: 1px 4px; font-family: monospace;">' + layerBadge + '</span>' +
                   '</div>' +
                   '</div>' +
                   '<div style="font-size: 9px; color: #64748b; margin-bottom: 6px; font-family: monospace;">' +
                   'Lat: ' + lat.toFixed(4) + '° N, Lon: ' + lon.toFixed(4) + '° E' +
                   '</div>' +
                   '<div style="display: flex; gap: 4px; margin-bottom: 8px;">' +
                   '<div style="flex: 1; background: rgba(30, 41, 59, 0.7); border: 1px solid #334155; border-radius: 4px; padding: 4px 6px; text-align: center;">' +
                   '<div style="font-size: 8px; color: #94a3b8; text-transform: uppercase;">Flights</div>' +
                   '<div style="font-size: 13px; font-weight: 700; color: #38bdf8; font-family: monospace;">' + flightCount.toLocaleString() + '</div>' +
                   '</div>' +
                   '<div style="flex: 1.2; background: rgba(30, 41, 59, 0.7); border: 1px solid #334155; border-radius: 4px; padding: 4px 6px; text-align: center;">' +
                   '<div style="font-size: 8px; color: #94a3b8; text-transform: uppercase;">Mean Max Noise</div>' +
                   '<div style="font-size: 13px; font-weight: 700; color: ' + noiseColor + '; font-family: monospace;">' + meanMaxNoise.toFixed(1) + ' <span style="font-size: 9px; font-weight: normal;">dB</span></div>' +
                   '</div>' +
                   '<div style="flex: 1; background: rgba(30, 41, 59, 0.7); border: 1px solid #334155; border-radius: 4px; padding: 4px 6px; text-align: center;">' +
                   '<div style="font-size: 8px; color: #94a3b8; text-transform: uppercase;">Peak</div>' +
                   '<div style="font-size: 13px; font-weight: 700; color: #f8fafc; font-family: monospace;">' + peakNoise.toFixed(1) + ' <span style="font-size: 9px; font-weight: normal;">dB</span></div>' +
                   '</div>' +
                   '</div>' +
                   '<div style="margin-bottom: 6px;">' +
                   '<div style="font-size: 9.5px; font-weight: 600; color: #cbd5e1; margin-bottom: 3px; display: flex; justify-content: space-between;">' +
                   '<span>🛫 Altitude Distribution' + (tierFilter !== null ? ' (' + layerBadge + ')' : '') + '</span>' +
                   '<span style="font-size: 8.5px; color: #64748b;">Flights (%)</span>' +
                   '</div>' +
                   '<div style="display: flex; flex-direction: column; gap: 2px;">';

        const tierDefs = [
            {{ label: '0–500m', color: '#ef4444' }},
            {{ label: '500–1k', color: '#f97316' }},
            {{ label: '1k–2k',  color: '#eab308' }},
            {{ label: '2k–3k',  color: '#ca8a04' }},
            {{ label: '3k–5k',  color: '#14b8a6' }},
            {{ label: '5k+ m',  color: '#22c55e' }}
        ];

        tierDefs.forEach(function(td, idx) {{
            let isSelected = (tierFilter === idx);
            let cnt = altBreakdown[idx] || 0;
            let pct = flightCount > 0 ? (cnt / flightCount * 100) : 0;
            let barW = Math.max(pct > 0 ? 3 : 0, pct);
            let highlightStyle = isSelected ? 'border: 1px solid ' + td.color + '; background: rgba(56, 189, 248, 0.12); border-radius: 3px; padding: 1px 2px;' : '';
            html += '<div style="display: flex; align-items: center; gap: 4px; font-size: 8.5px; font-family: monospace; ' + highlightStyle + '">' +
                    '<span style="width: 48px; color: ' + (isSelected ? '#38bdf8; font-weight: bold;' : '#94a3b8;') + ' white-space: nowrap;">' + td.label + '</span>' +
                    '<div style="flex: 1; height: 6px; background: #1e293b; border-radius: 3px; overflow: hidden; position: relative;">' +
                    '<div style="width: ' + barW + '%; height: 100%; background: ' + td.color + '; border-radius: 3px;"></div>' +
                    '</div>' +
                    '<span style="width: 54px; text-align: right; color: ' + (cnt > 0 ? '#f1f5f9' : '#64748b') + ';">' + cnt + ' <span style="font-size: 7.5px; color: #64748b;">(' + pct.toFixed(0) + '%)</span></span>' +
                    '</div>';
        }});

        html += '</div></div>' +
                '<div style="border-top: 1px solid #334155; padding-top: 4px;">' +
                '<div style="font-size: 9.5px; font-weight: 600; color: #cbd5e1; margin-bottom: 3px; display: flex; justify-content: space-between;">' +
                '<span>🔊 Flights by Max Noise dB(A)</span>' +
                '<span style="font-size: 8.5px; color: #64748b;">Flights (%)</span>' +
                '</div>' +
                '<div style="display: flex; flex-direction: column; gap: 2px;">';

        const noiseBins = [
            {{ label: '< 50 dB', color: '#22c55e', cnt: noiseBreakdown['<50'] || 0 }},
            {{ label: '50–60 dB', color: '#a3e635', cnt: noiseBreakdown['50-60'] || 0 }},
            {{ label: '60–70 dB', color: '#facc15', cnt: noiseBreakdown['60-70'] || 0 }},
            {{ label: '70–80 dB', color: '#f97316', cnt: noiseBreakdown['70-80'] || 0 }},
            {{ label: '80–90 dB', color: '#ef4444', cnt: noiseBreakdown['80-90'] || 0 }},
            {{ label: '90+ dB',  color: '#b91c1c', cnt: noiseBreakdown['90+'] || 0 }}
        ];

        noiseBins.forEach(function(nb) {{
            let pct = flightCount > 0 ? (nb.cnt / flightCount * 100) : 0;
            let barW = Math.max(pct > 0 ? 3 : 0, pct);
            html += '<div style="display: flex; align-items: center; gap: 4px; font-size: 8.5px; font-family: monospace;">' +
                    '<span style="width: 48px; color: #94a3b8; white-space: nowrap;">' + nb.label + '</span>' +
                    '<div style="flex: 1; height: 6px; background: #1e293b; border-radius: 3px; overflow: hidden;">' +
                    '<div style="width: ' + barW + '%; height: 100%; background: ' + nb.color + '; border-radius: 3px;"></div>' +
                    '</div>' +
                    '<span style="width: 54px; text-align: right; color: ' + (nb.cnt > 0 ? '#f1f5f9' : '#64748b') + ';">' + nb.cnt + ' <span style="font-size: 7.5px; color: #64748b;">(' + pct.toFixed(0) + '%)</span></span>' +
                    '</div>';
        }});

        html += '</div></div>';
        return html;
    }}

    function computeAndShowHoverStats(latlng, clientX, clientY) {{
        if (!latlng || !window.CURRENT_FILTERED_TRACKS || !window.leafletMap) return;

        let activeMode = (window.densityLayer && window.densityLayer._mode !== undefined) ? window.densityLayer._mode : (window.CURRENT_LAYER_MODE || 'all');
        let tierFilter = (activeMode !== 'all' && activeMode !== 'multi') ? parseInt(activeMode) : null;

        let lat = latlng.lat;
        let lon = latlng.lng;
        let r_km = window.HOVER_RADIUS_KM || 1.0;
        let r_sq = r_km * r_km;

        let ky = 110.574;
        let kx = 111.320 * Math.cos(lat * Math.PI / 180.0);
        let dLat = r_km / ky;
        let dLon = r_km / kx;

        let minQ_Lat = lat - dLat;
        let maxQ_Lat = lat + dLat;
        let minQ_Lon = lon - dLon;
        let maxQ_Lon = lon + dLon;

        let tracks = window.CURRENT_FILTERED_TRACKS;
        let nTracks = tracks.length;

        let flightMap = new Map();

        for (let i = 0; i < nTracks; i++) {{
            let tr = tracks[i];
            if (tierFilter !== null && tr[3] !== tierFilter) {{
                continue;
            }}

            let bbox = tr._bbox;
            if (!bbox) {{
                let pts = tr[0];
                let minL = pts[0][0], maxL = pts[0][0], minO = pts[0][1], maxO = pts[0][1];
                for (let p = 1; p < pts.length; p++) {{
                    let pLat = pts[p][0], pLon = pts[p][1];
                    if (pLat < minL) minL = pLat;
                    if (pLat > maxL) maxL = pLat;
                    if (pLon < minO) minO = pLon;
                    if (pLon > maxO) maxO = pLon;
                }}
                bbox = [minL, maxL, minO, maxO];
                tr._bbox = bbox;
            }}

            if (bbox[1] < minQ_Lat || bbox[0] > maxQ_Lat || bbox[3] < minQ_Lon || bbox[2] > maxQ_Lon) {{
                continue;
            }}

            let pts = tr[0];
            let intersects = false;
            let nPts = pts.length;

            for (let j = 0; j < nPts - 1; j++) {{
                let p1 = pts[j];
                let p2 = pts[j + 1];

                let x1 = (p1[1] - lon) * kx;
                let y1 = (p1[0] - lat) * ky;
                let x2 = (p2[1] - lon) * kx;
                let y2 = (p2[0] - lat) * ky;

                let dx = x2 - x1;
                let dy = y2 - y1;
                let lenSq = dx * dx + dy * dy;

                let distSq;
                if (lenSq === 0) {{
                    distSq = x1 * x1 + y1 * y1;
                }} else {{
                    let t = - (x1 * dx + y1 * dy) / lenSq;
                    if (t < 0) t = 0;
                    else if (t > 1) t = 1;
                    let projX = x1 + t * dx;
                    let projY = y1 + t * dy;
                    distSq = projX * projX + projY * projY;
                }}

                if (distSq <= r_sq) {{
                    intersects = true;
                    break;
                }}
            }}

            if (intersects) {{
                let fId = tr[6];
                let cs = tr[8];
                let flightKey;
                if (fId && fId > 0) {{
                    flightKey = 'f_' + fId;
                }} else if (cs && cs.length > 0) {{
                    flightKey = cs + '_' + Math.floor(tr[1] / 3600);
                }} else {{
                    flightKey = 'trk_' + i;
                }}

                let tier = tr[3];
                let noise = tr[7];

                let fInfo = flightMap.get(flightKey);
                if (!fInfo) {{
                    flightMap.set(flightKey, {{
                        maxNoise: noise,
                        tier: tier
                    }});
                }} else {{
                    if (noise > fInfo.maxNoise) {{
                        fInfo.maxNoise = noise;
                        fInfo.tier = tier;
                    }}
                }}
            }}
        }}

        let flightCount = flightMap.size;
        let altBreakdown = {{ 0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0 }};
        let noiseBreakdown = {{ '<50': 0, '50-60': 0, '60-70': 0, '70-80': 0, '80-90': 0, '90+': 0 }};
        let noiseSum = 0;
        let peakNoise = 0;

        flightMap.forEach(function(f) {{
            altBreakdown[f.tier] = (altBreakdown[f.tier] || 0) + 1;
            let n = f.maxNoise;
            noiseSum += n;
            if (n > peakNoise) peakNoise = n;

            if (n < 50) noiseBreakdown['<50']++;
            else if (n < 60) noiseBreakdown['50-60']++;
            else if (n < 70) noiseBreakdown['60-70']++;
            else if (n < 80) noiseBreakdown['70-80']++;
            else if (n < 90) noiseBreakdown['80-90']++;
            else noiseBreakdown['90+']++;
        }});

        let meanMaxNoise = flightCount > 0 ? (noiseSum / flightCount) : 0;

        window.hoverCardEl.innerHTML = renderHoverStatsHTML(
            lat, lon, r_km, activeMode, tierFilter, flightCount, meanMaxNoise, peakNoise, altBreakdown, noiseBreakdown
        );
        window.hoverCardEl.style.display = 'block';

        positionHoverCard(clientX, clientY);

        if (!window.hoverCircle) {{
            window.hoverCircle = L.circle(latlng, {{
                radius: r_km * 1000,
                color: '#38bdf8',
                weight: 1.5,
                dashArray: '4, 4',
                fillColor: '#0284c7',
                fillOpacity: 0.15,
                interactive: false
            }}).addTo(window.leafletMap);
        }} else {{
            window.hoverCircle.setLatLng(latlng);
            window.hoverCircle.setRadius(r_km * 1000);
            window.hoverCircle.setStyle({{ opacity: 1, fillOpacity: 0.15 }});
        }}
    }}

    function initHoverStatsEngine(map) {{
        precomputeTrackBBoxes();
        window.hoverCardEl = document.getElementById("hover_stats_card");
        if (!window.hoverCardEl) {{
            window.hoverCardEl = document.createElement("div");
            window.hoverCardEl.id = "hover_stats_card";
            window.hoverCardEl.style.cssText = "display: none; position: fixed; z-index: 10001; pointer-events: none; background: rgba(15, 23, 42, 0.96); color: #f8fafc; border-radius: 8px; padding: 10px 14px; box-shadow: 0 8px 32px rgba(0, 0, 0, 0.7); border: 1px solid rgba(56, 189, 248, 0.4); backdrop-filter: blur(10px); -webkit-backdrop-filter: blur(10px); width: 280px; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;";
            document.body.appendChild(window.hoverCardEl);
        }}

        map.on('mousemove', function(e) {{
            window.CURRENT_MOUSE_LATLNG = e.latlng;
            window.CURRENT_MOUSE_CLIENT_X = e.originalEvent.clientX;
            window.CURRENT_MOUSE_CLIENT_Y = e.originalEvent.clientY;
            
            if (window.hoverTimer) clearTimeout(window.hoverTimer);
            hideHoverCard();
            
            window.hoverTimer = setTimeout(function() {{
                if (window.CURRENT_MOUSE_LATLNG) {{
                    computeAndShowHoverStats(
                        window.CURRENT_MOUSE_LATLNG,
                        window.CURRENT_MOUSE_CLIENT_X,
                        window.CURRENT_MOUSE_CLIENT_Y
                    );
                }}
            }}, 1000);
        }});

        map.on('mouseout', function() {{
            if (window.hoverTimer) clearTimeout(window.hoverTimer);
            window.CURRENT_MOUSE_LATLNG = null;
            hideHoverCard();
        }});

        map.on('movestart zoomstart dragstart', function() {{
            if (window.hoverTimer) clearTimeout(window.hoverTimer);
            hideHoverCard();
        }});
    }}

    document.addEventListener("DOMContentLoaded", function() {{
        setTimeout(function() {{
            var leafletMap = null;
            for (let k in window) {{
                if (k.startsWith("map_") && window[k] && typeof window[k].getZoom === "function") {{
                    leafletMap = window[k];
                    break;
                }}
            }}

            if (leafletMap) {{
                window.leafletMap = leafletMap;
                window.densityLayer = new InteractiveRasterDensityLayer(window.FLIGHT_TRACKS);
                window.densityLayer.addTo(leafletMap);
                initHoverStatsEngine(leafletMap);
            }}

            populateGroupChips();
            updateCascadingModels();
            
            let s_slider = document.getElementById("slider_start_t");
            let e_slider = document.getElementById("slider_end_t");
            let prog_bar = document.getElementById("slider_progress");
            let wrap = document.getElementById("slider_wrap");
            
            if (s_slider && e_slider) {{
                s_slider.addEventListener("input", function() {{
                    let s_val = parseInt(s_slider.value);
                    let e_val = parseInt(e_slider.value);
                    if (s_val > e_val) {{
                        s_slider.value = e_val;
                        s_val = e_val;
                    }}
                    updateSliderUI(s_val, e_val);
                    filterPoints(s_val, e_val);
                }});
                
                e_slider.addEventListener("input", function() {{
                    let s_val = parseInt(s_slider.value);
                    let e_val = parseInt(e_slider.value);
                    if (e_val < s_val) {{
                        e_slider.value = s_val;
                        e_val = s_val;
                    }}
                    updateSliderUI(s_val, e_val);
                    filterPoints(s_val, e_val);
                }});
            }}
            
            let isDraggingBar = false;
            let startClientX = 0;
            let initS = 0;
            let initE = 0;
            let barSpan = 0;
            
            if (prog_bar && wrap && s_slider && e_slider) {{
                prog_bar.addEventListener("mousedown", function(e) {{
                    isDraggingBar = true;
                    startClientX = e.clientX;
                    initS = parseInt(s_slider.value);
                    initE = parseInt(e_slider.value);
                    barSpan = initE - initS;
                    document.body.style.cursor = "grabbing";
                    prog_bar.style.cursor = "grabbing";
                    e.preventDefault();
                }});
                
                prog_bar.addEventListener("touchstart", function(e) {{
                    if (e.touches && e.touches.length > 0) {{
                        isDraggingBar = true;
                        startClientX = e.touches[0].clientX;
                        initS = parseInt(s_slider.value);
                        initE = parseInt(e_slider.value);
                        barSpan = initE - initS;
                    }}
                }}, {{passive: true}});
                
                window.addEventListener("mousemove", function(e) {{
                    if (!isDraggingBar) return;
                    let dx = e.clientX - startClientX;
                    let rect = wrap.getBoundingClientRect();
                    let dt = Math.round((dx / rect.width) * (window.T_MAX - window.T_MIN));
                    
                    let newS = initS + dt;
                    if (newS < window.T_MIN) newS = window.T_MIN;
                    if (newS + barSpan > window.T_MAX) newS = window.T_MAX - barSpan;
                    let newE = newS + barSpan;
                    
                    s_slider.value = newS;
                    e_slider.value = newE;
                    updateSliderUI(newS, newE);
                    filterPoints(newS, newE);
                }});
                
                window.addEventListener("touchmove", function(e) {{
                    if (!isDraggingBar || !e.touches || e.touches.length === 0) return;
                    let dx = e.touches[0].clientX - startClientX;
                    let rect = wrap.getBoundingClientRect();
                    let dt = Math.round((dx / rect.width) * (window.T_MAX - window.T_MIN));
                    
                    let newS = initS + dt;
                    if (newS < window.T_MIN) newS = window.T_MIN;
                    if (newS + barSpan > window.T_MAX) newS = window.T_MAX - barSpan;
                    let newE = newS + barSpan;
                    
                    s_slider.value = newS;
                    e_slider.value = newE;
                    updateSliderUI(newS, newE);
                    filterPoints(newS, newE);
                }}, {{passive: true}});
                
                window.addEventListener("mouseup", function() {{
                    if (isDraggingBar) {{
                        isDraggingBar = false;
                        document.body.style.cursor = "default";
                        prog_bar.style.cursor = "grab";
                    }}
                }});
                
                window.addEventListener("touchend", function() {{
                    isDraggingBar = false;
                }});
            }}
            
            filterPoints(window.T_MIN, window.T_MAX);
        }}, 400);
    }});
    </script>
    """
    
    m.get_root().html.add_child(folium.Element(control_and_banner_html))

    Fullscreen(position='topright', title='Toggle Fullscreen', title_cancel='Exit Fullscreen', force_separate_button=True).add_to(m)
    MousePosition(position='bottomleft', separator=' | Lat: ', empty_string='NaN', lng_first=False, prefix='Coordinates: Lon ').add_to(m)
    LayerControl(position='topright', collapsed=False).add_to(m)
    
    return m

flight_map = create_altitude_tiered_heatmap_map(
    df=df_processed,
    lat_min=LAT_MIN,
    lat_max=LAT_MAX,
    lon_min=LON_MIN,
    lon_max=LON_MAX,
    region_name=REGION_NAME,
    coverage=cache_coverage
)

html_export_path = os.path.join(WORKSPACE_DIR, "opensky_flight_heatmap.html")
flight_map.save(html_export_path)
print(f"💾 Interactive Heatmap exported to standalone file: {html_export_path}")

## 6. 📊 Statistical Summary & Aircraft Type Group Distribution Plots

In [ ]:
if not df_processed.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5), dpi=100)
    plt.style.use('dark_background')
    
    tier_colors = {
        "0 - 500m (Red)": "#ff3344",
        "500 - 1000m (Orange)": "#ff6a00",
        "1000 - 2000m (Amber)": "#ffb300",
        "2000 - 3000m (Yellow)": "#fbc02d",
        "3000 - 5000m (Teal)": "#00bcd4",
        "5000m+ (Green)": "#00e676"
    }
    
    # 1. Altitude Tier Distribution
    ax1 = axes[0]
    tier_counts = df_processed['altitude_tier'].value_counts()
    tier_order = ["0 - 500m (Red)", "500 - 1000m (Orange)", "1000 - 2000m (Amber)", "2000 - 3000m (Yellow)", "3000 - 5000m (Teal)", "5000m+ (Green)"]
    counts_ordered = [tier_counts.get(t, 0) for t in tier_order]
    colors_ordered = [tier_colors[t] for t in tier_order]
    
    bars = ax1.bar([t.split(' ')[0] + 'm' for t in tier_order], counts_ordered, color=colors_ordered, edgecolor='#334155', width=0.6)
    ax1.set_title("Waypoint Distribution by Altitude Tier", fontsize=11, fontweight='bold', color='#38bdf8')
    ax1.set_ylabel("Point Count")
    ax1.grid(axis='y', alpha=0.2)
    for bar in bars:
        h = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., h + max(counts_ordered)*0.01 if counts_ordered else 0, f'{h:,}', ha='center', va='bottom', fontsize=8, color='#cbd5e1')
        
    # 2. Ground Noise Level Distribution
    ax2 = axes[1]
    n_counts, n_bins, _ = ax2.hist(df_processed['ground_noise_dba'], bins=20, color='#f59e0b', edgecolor='#334155', alpha=0.85)
    ax2.set_title(f"Estimated Ground Noise Footprint (dB(A))\nAvg: {df_processed['ground_noise_dba'].mean():.1f} dB(A), Max: {df_processed['ground_noise_dba'].max():.1f} dB(A)", fontsize=11, fontweight='bold', color='#f59e0b')
    ax2.set_xlabel("Perceived Ground Noise (dB(A))")
    ax2.set_ylabel("Waypoints")
    ax2.grid(axis='y', alpha=0.2)
    ax2.axvline(65, color='#ff3344', linestyle='--', label='Loud Urban Threshold (65 dB)')
    ax2.legend(fontsize=8)
    
    # 3. Aircraft Type Groups Distribution
    ax3 = axes[2]
    top_grps = df_processed['aircraft_type_group'].value_counts()
    ax3.barh(top_grps.index[::-1], top_grps.values[::-1], color='#38bdf8', edgecolor='#334155', height=0.6)
    ax3.set_title("Waypoints by Aircraft Type Group", fontsize=11, fontweight='bold', color='#38bdf8')
    ax3.set_xlabel("Waypoints")
    ax3.grid(axis='x', alpha=0.2)
    
    plt.tight_layout()
    plt.show()


## 7. 📈 Visual Subregion Polygon Selection, In-Depth Statistics & Acoustic Cross-Analysis Dashboard

Interactively select any geographic subregion within the main analysis region directly on the map below:
- **Visual Polygon Controls**: Starts with an initial square centered in the map. Click the **Edit layers (pencil icon)** on the map toolbar to **drag corner bounds**, **drag edge midpoints to add new edges**, or use **Delete** to remove vertices/shapes.
- **Exact Vectorized Polygon Clipping**: Applies 2-stage spatial filtering (PyArrow bounding box pushdown + ray-casting  polygon intersection).
- **Multi-Dimensional Analysis Crossed with Ground Noise**:
  - **Planes & Altitude Tiers**: Unique planes and flights per tier crossed with perceived ground noise dB(A).
  - **Temporal Profiles**: 24-hour diurnal traffic and night sensitivity curve (22:00–06:00) crossed with noise curves.
  - **Days of Week**: Weekly traffic volume and loud exceedance events (>= 65 dB(A)).
  - **Regularity (Days in Month / Week)**: Active calendar days timeline and weekly traffic rates.
  - **Aircraft Fleets**: Dominant aircraft models and categories crossed with mean/peak noise footprints.
  - **2D Acoustic Matrix**: Day of Week x Hour of Day interaction heatmap revealing acoustic peak windows.

In [ ]:
import os, sys, gc
from datetime import datetime, timedelta, timezone
from typing import Any, Optional, Dict, List, Tuple, Set
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow.dataset as ds
import pyarrow.compute as pc
import pyarrow.parquet as pq
from matplotlib.path import Path
import ipywidgets as widgets
from ipywidgets import HTML, VBox, HBox, Button, Checkbox, Dropdown, SelectMultiple, IntSlider, DatetimePicker, Layout, Label
from IPython.display import display, clear_output

# Clear any lingering cell-level output from previous runs
clear_output(wait=False)
from ipyleaflet import Map, basemaps, DrawControl, Rectangle, WidgetControl

# ==============================================================================
# 📍 GEOGRAPHIC SETUP & MAIN REGION BOUNDS
# ==============================================================================
MAIN_LAT_MIN = LAT_MIN if 'LAT_MIN' in globals() else 47.1102
MAIN_LAT_MAX = LAT_MAX if 'LAT_MAX' in globals() else 47.4855
MAIN_LON_MIN = LON_MIN if 'LON_MIN' in globals() else 8.3720
MAIN_LON_MAX = LON_MAX if 'LON_MAX' in globals() else 9.0478
parquet_export_path = parquet_export_path if ('parquet_export_path' in globals() and os.path.exists(parquet_export_path)) else (
    OUTPUT_PARQUET if ('OUTPUT_PARQUET' in globals() and os.path.exists(OUTPUT_PARQUET)) else 'processed_flight_paths.parquet'
)

center_lat = (MAIN_LAT_MIN + MAIN_LAT_MAX) / 2.0
center_lon = (MAIN_LON_MIN + MAIN_LON_MAX) / 2.0
delta_lat = (MAIN_LAT_MAX - MAIN_LAT_MIN) * 0.12
delta_lon = (MAIN_LON_MAX - MAIN_LON_MIN) * 0.12

def get_default_center_square() -> List[Tuple[float, float]]:
    """Generates initial square polygon in the center of the main analysis region."""
    return [
        (round(center_lat - delta_lat, 4), round(center_lon - delta_lon, 4)),
        (round(center_lat + delta_lat, 4), round(center_lon - delta_lon, 4)),
        (round(center_lat + delta_lat, 4), round(center_lon + delta_lon, 4)),
        (round(center_lat - delta_lat, 4), round(center_lon + delta_lon, 4))
    ]

active_polygon_coords: List[Tuple[float, float]] = get_default_center_square()

def coords_to_geojson_feature(lat_lons: List[Tuple[float, float]]) -> Dict[str, Any]:
    """Converts a list of (lat, lon) coordinates to a valid GeoJSON Feature."""
    closed = list(lat_lons)
    if closed[0] != closed[-1]:
        closed.append(closed[0])
    geojson_coords = [[pt[1], pt[0]] for pt in closed]
    return {
        "type": "Feature",
        "properties": {
            "style": {
                "color": "#38bdf8",
                "fillColor": "#0284c7",
                "fillOpacity": 0.35,
                "weight": 3
            }
        },
        "geometry": {
            "type": "Polygon",
            "coordinates": [geojson_coords]
        }
    }

def compute_polygon_area_km2(lat_lons: List[Tuple[float, float]]) -> float:
    """Calculates approximate geographic surface area of polygon in square kilometers."""
    if len(lat_lons) < 3:
        return 0.0
    ref_lat = sum(p[0] for p in lat_lons) / len(lat_lons)
    km_per_lat = 110.574
    km_per_lon = 111.320 * np.cos(np.radians(ref_lat))
    x = [p[1] * km_per_lon for p in lat_lons]
    y = [p[0] * km_per_lat for p in lat_lons]
    return 0.5 * abs(sum(x[i] * y[(i + 1) % len(x)] - x[(i + 1) % len(x)] * y[i] for i in range(len(x))))

def generate_subregion_html(lat_lons: List[Tuple[float, float]]) -> str:
    """Renders formatted HTML metadata card for current polygon shape."""
    area_km2 = compute_polygon_area_km2(lat_lons)
    centroid_lat = sum(p[0] for p in lat_lons) / max(1, len(lat_lons))
    centroid_lon = sum(p[1] for p in lat_lons) / max(1, len(lat_lons))
    pts_summary = ", ".join([f"[{p[0]:.3f}°, {p[1]:.3f}°]" for p in lat_lons[:6]])
    if len(lat_lons) > 6:
        pts_summary += f" ... (+{len(lat_lons)-6} more)"
    
    return f"""
    <div style="background: #0f172a; border: 1px solid #334155; border-radius: 8px; padding: 10px 14px; margin: 6px 0; color: #f8fafc; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 12px; line-height: 1.6;">
        <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 4px;">
            <strong style="color: #38bdf8; font-size: 13px;">📍 Active Subregion Polygon</strong>
            <span style="background: #0284c7; color: white; padding: 2px 8px; border-radius: 10px; font-size: 11px; font-weight: bold;">{len(lat_lons)} Vertices / Edges</span>
        </div>
        • <strong>Vertices</strong>: <code>{pts_summary}</code><br>
        • <strong>Estimated Area</strong>: <strong>~{area_km2:.1f} km²</strong> (Centroid: {centroid_lat:.4f}° N, {centroid_lon:.4f}° E)<br>
        • <strong>Main Analysis Region</strong>: [{MAIN_LAT_MIN:.2f}°, {MAIN_LAT_MAX:.2f}° N] &times; [{MAIN_LON_MIN:.2f}°, {MAIN_LON_MAX:.2f}° E] (dashed amber boundary)<br>
        <span style="font-size: 11px; color: #94a3b8; display: inline-block; margin-top: 3px;">
            💡 <em>Visual Editing: Click <strong>Edit layers (pencil icon)</strong> on the map toolbar to drag corners or drag edge midpoints to add new edges. Click <strong>Delete</strong> to remove. All stats recompute automatically on change.</em>
        </span>
    </div>
    """

# ==============================================================================
# ⚙️ FILTER OPTIONS DEFINITIONS & METADATA BOUNDS
# ==============================================================================
ALL_AIRCRAFT_GROUPS = [
    'Commercial Heavy Jet',
    'Commercial Medium Jet',
    'Business Jet',
    'Regional Turboprop',
    'Light Aircraft / Leisure GA',
    'Helicopter / Rotorcraft',
    'Military / Special Ops',
    'Glider / Sailplane',
    'Other / General'
]

ALTITUDE_TIER_OPTIONS = [
    "0 - 500m (Red)",
    "500 - 1000m (Orange)",
    "1000 - 2000m (Amber)",
    "2000 - 3000m (Yellow)",
    "3000 - 5000m (Teal)",
    "5000m+ (Green)"
]

# Fast timestamp bounds from parquet metadata
try:
    meta_stat = pq.read_metadata(parquet_export_path)
    schema_stat = pq.read_schema(parquet_export_path)
    ts_idx = schema_stat.get_field_index('timestamp')
    min_ts_val = min(meta_stat.row_group(i).column(ts_idx).statistics.min for i in range(meta_stat.num_row_groups))
    max_ts_val = max(meta_stat.row_group(i).column(ts_idx).statistics.max for i in range(meta_stat.num_row_groups))
    dataset_min_dt = datetime.fromtimestamp(min_ts_val, tz=timezone.utc)
    dataset_max_dt = datetime.fromtimestamp(max_ts_val, tz=timezone.utc)
except Exception:
    dataset_min_dt = datetime(2026, 8, 26, 8, 0, tzinfo=timezone.utc)
    dataset_max_dt = datetime(2026, 10, 17, 6, 0, tzinfo=timezone.utc)

# Extract group to model mapping
group_models_dict: Dict[str, List[str]] = {}
try:
    dataset_preview = ds.dataset(parquet_export_path, format='parquet')
    tbl_models = dataset_preview.to_table(columns=['aircraft_type_group', 'typecode']).to_pandas().drop_duplicates()
    for grp, df_grp in tbl_models.groupby('aircraft_type_group'):
        group_models_dict[grp] = sorted([str(m) for m in df_grp['typecode'].unique() if pd.notna(m) and str(m).strip() not in ('', 'Unknown')])
except Exception:
    pass

# ==============================================================================
# 📈 EXACT POLYGON STATISTICAL & ACOUSTIC CROSS-ANALYSIS ENGINE
# ==============================================================================

def analyze_and_plot_polygon_stats(
    lat_lons: List[Tuple[float, float]],
    selected_altitudes: Tuple[str, ...],
    use_full_history: bool,
    start_dt: Optional[datetime],
    end_dt: Optional[datetime],
    min_noise: float,
    selected_groups: Tuple[str, ...],
    model_filter: str
):
    """
    Performs multi-filtered exact polygon clipping on historical trajectories and
    generates 6-panel multi-dimensional analysis crossed with ground noise (dB(A)).
    """
    def _render():
        if len(lat_lons) < 3:
            print("⚠️ Selected polygon requires at least 3 vertices to perform spatial analysis.")
            return

        if not selected_altitudes:
            print("⚠️ No altitude tiers selected. Please select at least one altitude bucket.")
            return

        if not selected_groups:
            print("⚠️ No aircraft groups selected. Please select at least one aircraft category.")
            return

        lats = [p[0] for p in lat_lons]
        lons = [p[1] for p in lat_lons]
        sub_lat_min, sub_lat_max = min(lats), max(lats)
        sub_lon_min, sub_lon_max = min(lons), max(lons)

        # 1. Build PyArrow Pushdown Filters
        parquet_src = parquet_export_path if ('parquet_export_path' in globals() and os.path.exists(parquet_export_path)) else (
            stage3_intermediate_path if ('stage3_intermediate_path' in globals() and os.path.exists(stage3_intermediate_path)) else None
        )

        cols = ['icao24', 'callsign', 'timestamp', 'latitude', 'longitude', 'altitude', 'aircraft_type_group', 'typecode', 'model', 'ground_noise_dba', 'altitude_tier']

        # Bounding Box + Attribute Pushdown Filter
        filter_expr = (
            (pc.field('latitude') >= sub_lat_min) & (pc.field('latitude') <= sub_lat_max) &
            (pc.field('longitude') >= sub_lon_min) & (pc.field('longitude') <= sub_lon_max) &
            (pc.field('altitude_tier').isin(list(selected_altitudes))) &
            (pc.field('aircraft_type_group').isin(list(selected_groups)))
        )

        if min_noise > 30.0:
            filter_expr = filter_expr & (pc.field('ground_noise_dba') >= float(min_noise))

        if model_filter and model_filter != 'All Specific Models':
            filter_expr = filter_expr & (pc.field('typecode') == model_filter)

        if not use_full_history and start_dt and end_dt:
            s_ts = int(start_dt.timestamp())
            e_ts = int(end_dt.timestamp())
            if s_ts > e_ts:
                s_ts, e_ts = e_ts, s_ts
            filter_expr = filter_expr & (pc.field('timestamp') >= s_ts) & (pc.field('timestamp') <= e_ts)

        if parquet_src and os.path.exists(parquet_src):
            dataset = ds.dataset(parquet_src, format='parquet')
            tbl = dataset.to_table(columns=cols, filter=filter_expr)
            df_sub = tbl.to_pandas()
            del tbl
        elif 'df_processed' in globals() and not df_processed.empty:
            df_sub = df_processed[
                (df_processed['latitude'] >= sub_lat_min) & (df_processed['latitude'] <= sub_lat_max) &
                (df_processed['longitude'] >= sub_lon_min) & (df_processed['longitude'] <= sub_lon_max) &
                (df_processed['altitude_tier'].isin(list(selected_altitudes))) &
                (df_processed['aircraft_type_group'].isin(list(selected_groups)))
            ].copy()
            if min_noise > 30.0:
                df_sub = df_sub[df_sub['ground_noise_dba'] >= min_noise]
            if model_filter and model_filter != 'All Specific Models':
                df_sub = df_sub[df_sub['typecode'] == model_filter]
            if not use_full_history and start_dt and end_dt:
                s_ts = int(start_dt.timestamp())
                e_ts = int(end_dt.timestamp())
                df_sub = df_sub[(df_sub['timestamp'] >= s_ts) & (df_sub['timestamp'] <= e_ts)]
        else:
            print("⚠️ No processed flight dataset found on disk or memory.")
            return

        if df_sub.empty:
            print(f"⚠️ No waypoints matched the active filters within the subregion envelope [{sub_lat_min:.4f}, {sub_lat_max:.4f}] x [{sub_lon_min:.4f}, {sub_lon_max:.4f}].")
            return

        # 2. Exact Vectorized Ray-Casting Polygon In/Out Filtering
        poly_path = Path(lat_lons)
        pts = np.column_stack((df_sub['latitude'].to_numpy(), df_sub['longitude'].to_numpy()))
        inside_mask = poly_path.contains_points(pts)
        df_sub = df_sub[inside_mask].copy()
        del pts, inside_mask
        gc.collect()

        if df_sub.empty:
            print("⚠️ No flight waypoints found inside the exact polygon boundary with the active filters.")
            return

        # 3. Discrete Flight Pass Segmentation
        df_sub = df_sub.sort_values(['icao24', 'timestamp']).reset_index(drop=True)
        time_diff = df_sub.groupby('icao24', observed=False)['timestamp'].diff().fillna(99999)
        new_pass = (time_diff > 1200) | (df_sub['icao24'] != df_sub['icao24'].shift(1))
        df_sub['flight_pass_id'] = new_pass.cumsum()

        flights = df_sub.groupby('flight_pass_id').agg(
            icao24=('icao24', 'first'),
            callsign=('callsign', 'first'),
            typecode=('typecode', 'first'),
            model=('model', 'first'),
            aircraft_type_group=('aircraft_type_group', 'first'),
            min_altitude=('altitude', 'min'),
            max_noise=('ground_noise_dba', 'max'),
            avg_noise=('ground_noise_dba', 'mean'),
            start_time=('timestamp', 'min'),
            waypoints=('timestamp', 'count')
        ).reset_index()

        total_pts = len(df_sub)
        total_flights = len(flights)
        unique_planes = flights['icao24'].nunique()
        del df_sub
        gc.collect()

        # 4. Feature Extraction across Dimensions
        dt = pd.to_datetime(flights['start_time'], unit='s', utc=True)
        flights['day_of_week'] = dt.dt.day_name()
        flights['hour_of_day'] = dt.dt.hour
        flights['date'] = dt.dt.date
        flights['month_year'] = dt.dt.strftime('%b %Y')
        flights['week_num'] = dt.dt.isocalendar().week

        tier_order = ["0 - 500m (Red)", "500 - 1000m (Orange)", "1000 - 2000m (Amber)", "2000 - 3000m (Yellow)", "3000 - 5000m (Teal)", "5000m+ (Green)"]
        tier_colors = ["#ff3344", "#ff6a00", "#ffb300", "#fbc02d", "#00bcd4", "#00e676"]

        def _get_tier(alt):
            if alt <= 500: return tier_order[0]
            elif alt <= 1000: return tier_order[1]
            elif alt <= 2000: return tier_order[2]
            elif alt <= 3000: return tier_order[3]
            elif alt <= 5000: return tier_order[4]
            else: return tier_order[5]

        flights['altitude_tier'] = flights['min_altitude'].apply(_get_tier)
        day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
        area_km2 = compute_polygon_area_km2(lat_lons)

                # 5. Formatted Summary Report
        noise_min_val = flights['max_noise'].min()
        noise_max_val = flights['max_noise'].max()
        bin_start = int(np.floor(noise_min_val / 5.0) * 5)
        bin_end = int(np.ceil(noise_max_val / 5.0) * 5)
        if bin_start == bin_end:
            bin_end += 5
        bins_5db = list(range(bin_start, bin_end + 5, 5))
        bin_labels = [f"{bins_5db[i]}-{bins_5db[i+1]} dB(A)" for i in range(len(bins_5db)-1)]

        flight_cut = pd.cut(flights['max_noise'], bins=bins_5db, labels=bin_labels, right=False)
        flight_counts = flight_cut.value_counts(sort=False)

        print("=" * 80)
        print("📊 REGIONAL SUBREGION POLYGON ANALYSIS REPORT (FILTERED)")
        print("=" * 80)
        print(f"  • Shape Geometry             : Polygon with {len(lat_lons)} vertices (~{area_km2:.1f} km²)")
        print(f"  • Active Filter Scope        : Altitudes: {len(selected_altitudes)}/6 tiers | Noise: ≥ {min_noise:.0f} dB(A) | Groups: {len(selected_groups)}/9 | Model: {model_filter}")
        print(f"  • Trajectory Points Inside   : {total_pts:,} waypoints")
        print(f"  • Total Discrete Flight Passes: {total_flights:,}")
        print(f"  • Unique Aircraft Involved   : {unique_planes:,}")
        print(f"  • Active Observation Window  : {dt.min().strftime('%Y-%m-%d %H:%M UTC')} -> {dt.max().strftime('%Y-%m-%d %H:%M UTC')}")
        print(f"  • Active Calendar Days       : {flights['date'].nunique()} days (Avg {total_flights / max(1, flights['date'].nunique()):.1f} flights/active day)")
        print(f"  • Ground Noise Footprint     : Mean Peak {flights['max_noise'].mean():.1f} dB(A), Max Peak {flights['max_noise'].max():.1f} dB(A)")
        
        # Count of flights by noise level with 5 dB step
        print("  • Flights by Noise Level (5 dB steps):")
        for lbl in bin_labels:
            f_cnt = flight_counts.get(lbl, 0)
            f_pct = (f_cnt / total_flights * 100.0) if total_flights > 0 else 0.0
            val_low = int(lbl.split('-')[0])
            tag = ""
            if val_low >= 75:
                tag = " [High Annoyance ≥ 75 dB]"
            elif val_low >= 65:
                tag = " [Urban Threshold ≥ 65 dB]"
            print(f"    - {lbl:>14} : {f_cnt:>6,} flights ({f_pct:>5.1f}%){tag}")
        print(f"    ----------------------------------------------------------------")
        print(f"    - Total Count of Flights: {total_flights:>6,} flights (100.0%)")

        night_mask = (flights['hour_of_day'] >= 22) | (flights['hour_of_day'] < 6)
        night_cnt = night_mask.sum()
        night_avg = flights.loc[night_mask, 'max_noise'].mean() if night_cnt > 0 else 0.0
        print(f"  • Sensitive Night (22h-06h)  : {night_cnt:,} flights ({night_cnt/total_flights*100:.1f}%), Avg Noise: {night_avg:.1f} dB(A)")
        print("=" * 80)

        # 6. Build Multi-Panel Dashboard Visualizations
        plt.style.use('dark_background')
        fig = plt.figure(figsize=(20, 18), dpi=100)
        gs = fig.add_gridspec(4, 2, hspace=0.38, wspace=0.25)

        # --- PANEL 1: Altitude Tier vs Flights & Noise ---
        ax1 = fig.add_subplot(gs[0, 0])
        alt_stats = flights.groupby('altitude_tier', observed=False).agg(
            planes=('icao24', 'nunique'),
            flights=('flight_pass_id', 'count'),
            avg_noise=('max_noise', 'mean'),
            p95_noise=('max_noise', lambda x: np.percentile(x, 95) if len(x)>0 else 0)
        ).reindex(tier_order).fillna(0)

        bars1 = ax1.bar(
            [t.split(' ')[0] + 'm' for t in tier_order],
            alt_stats['flights'],
            color=tier_colors, alpha=0.85, edgecolor='#1e293b', width=0.55, label='Flight Passes'
        )
        ax1.set_title(f'1. Flights by Altitude Tier vs Ground Noise (Total={total_flights:,})', fontsize=11, fontweight='bold', color='#38bdf8', pad=8)
        ax1.set_ylabel('Number of Flights', color='#38bdf8', fontsize=9)
        ax1.tick_params(axis='y', labelcolor='#38bdf8')
        ax1.grid(True, alpha=0.15, linestyle='--')
        for bar in bars1:
            h = bar.get_height()
            if h > 0:
                pct = (h / total_flights * 100.0) if total_flights > 0 else 0.0
                ax1.text(bar.get_x() + bar.get_width()/2., h + max(alt_stats['flights'])*0.015, f'{int(h):,} ({pct:.1f}%)', ha='center', va='bottom', fontsize=7.5, color='#cbd5e1')

        ax1_noise = ax1.twinx()
        ax1_noise.plot([t.split(' ')[0] + 'm' for t in tier_order], alt_stats['avg_noise'], color='#f59e0b', marker='o', linewidth=2.2, label='Mean Noise (dB(A))')
        ax1_noise.plot([t.split(' ')[0] + 'm' for t in tier_order], alt_stats['p95_noise'], color='#ef4444', marker='s', linestyle=':', linewidth=1.8, label='95th %ile Noise (dB(A))')
        ax1_noise.set_ylabel('Ground Noise (dB(A))', color='#f59e0b', fontsize=9)
        ax1_noise.tick_params(axis='y', labelcolor='#f59e0b')
        l1, lab1 = ax1.get_legend_handles_labels()
        l2, lab2 = ax1_noise.get_legend_handles_labels()
        ax1.legend(l1 + l2, lab1 + lab2, loc='upper right', fontsize=8, framealpha=0.6)

        # --- PANEL 2: Diurnal Traffic (Times of Day) & Ground Noise Curve ---
        ax2 = fig.add_subplot(gs[0, 1])
        num_days = max(1, flights['date'].nunique())
        hour_stats = flights.groupby('hour_of_day').agg(
            total_flights=('flight_pass_id', 'count'),
            avg_noise=('max_noise', 'mean'),
            max_noise=('max_noise', 'max')
        ).reindex(range(24)).fillna(0)
        hour_stats['avg_flights_per_hour'] = hour_stats['total_flights'] / num_days

        ax2.axvspan(-0.5, 5.5, color='#ef4444', alpha=0.13, label='Night Sensitive (22-06h)')
        ax2.axvspan(21.5, 23.5, color='#ef4444', alpha=0.13)
        bars2 = ax2.bar(range(24), hour_stats['avg_flights_per_hour'], color='#06b6d4', alpha=0.7, edgecolor='#1e293b', width=0.7, label=f'Avg Flights / Hour (N={num_days}d)')
        title_suffix = f"(Avg Flights / Hour across {num_days} Days)" if num_days > 1 else "(Hourly Volume)"
        ax2.set_title(f'2. Diurnal Traffic Volume {title_suffix} & Noise Curve', fontsize=11, fontweight='bold', color='#38bdf8', pad=8)
        ax2.set_xlabel('Hour of Day (UTC)', fontsize=9)
        ax2.set_ylabel('Avg Flights / Hour (Daily Rate)' if num_days > 1 else 'Flights / Hour', color='#06b6d4', fontsize=9)
        ax2.set_xticks(range(0, 24, 2))
        ax2.tick_params(axis='y', labelcolor='#06b6d4')
        ax2.grid(True, alpha=0.15, linestyle='--')
        
        # Annotate bars with average hourly rate
        max_avg_h = hour_stats['avg_flights_per_hour'].max()
        for h_idx, (bar_h, tot) in enumerate(zip(hour_stats['avg_flights_per_hour'], hour_stats['total_flights'])):
            if tot > 0:
                lbl = f"{bar_h:.1f}" if num_days > 1 else f"{int(tot)}"
                ax2.text(h_idx, bar_h + max_avg_h * 0.02, lbl, ha='center', va='bottom', fontsize=7.5, color='#cbd5e1')

        ax2_noise = ax2.twinx()
        ax2_noise.plot(range(24), hour_stats['avg_noise'], color='#f59e0b', marker='.', linewidth=2, label='Mean Peak Noise (dB(A))')
        ax2_noise.plot(range(24), hour_stats['max_noise'], color='#ef4444', linestyle='--', linewidth=1.5, label='Max Observed Noise (dB(A))')
        ax2_noise.set_ylabel('Ground Noise (dB(A))', color='#f59e0b', fontsize=9)
        ax2_noise.tick_params(axis='y', labelcolor='#f59e0b')
        hl1, hlab1 = ax2.get_legend_handles_labels()
        hl2, hlab2 = ax2_noise.get_legend_handles_labels()
        ax2.legend(hl1 + hl2, hlab1 + hlab2, loc='upper left', fontsize=8, framealpha=0.6)

        # --- PANEL 3: Days of Week vs Noise Level & Loud Events ---
        ax3 = fig.add_subplot(gs[1, 0])
        day_occurrences = flights[['date', 'day_of_week']].drop_duplicates()['day_of_week'].value_counts()
        day_stats = flights.groupby('day_of_week').agg(
            flights=('flight_pass_id', 'count'),
            planes=('icao24', 'nunique'),
            avg_noise=('max_noise', 'mean'),
            loud_flights=('max_noise', lambda x: (x >= 65).sum())
        ).reindex(day_order).fillna(0)
        day_stats['occurrences'] = [day_occurrences.get(d, 1) for d in day_order]
        day_stats['avg_per_day'] = day_stats['flights'] / day_stats['occurrences'].clip(lower=1)

        bars3 = ax3.bar(day_order, day_stats['flights'], color='#8b5cf6', alpha=0.75, edgecolor='#1e293b', width=0.55, label='Total Flights')
        mean_daily_all = flights.groupby('date').size().mean() if len(flights) > 0 else 0
        ax3.set_title(f'3. Traffic by Day of Week (Avg ~{mean_daily_all:.0f} fl/day across {num_days}d) & Loud Events', fontsize=11, fontweight='bold', color='#38bdf8', pad=8)
        ax3.set_ylabel(f'Total Flights (over {num_days} days)', color='#8b5cf6', fontsize=9)
        ax3.tick_params(axis='y', labelcolor='#8b5cf6')
        ax3.tick_params(axis='x', rotation=15)
        ax3.grid(True, alpha=0.15, linestyle='--')
        
        # Annotate bars with total and daily average
        max_f_day = day_stats['flights'].max()
        for b_idx, (tot_f, avg_d) in enumerate(zip(day_stats['flights'], day_stats['avg_per_day'])):
            if tot_f > 0:
                lbl_d = f"{int(tot_f):,} (" + f"{avg_d:.0f}/d)" if num_days > 7 else f"{int(tot_f):,}"
                ax3.text(b_idx, tot_f + max_f_day * 0.02, lbl_d, ha='center', va='bottom', fontsize=7, color='#cbd5e1')

        ax3_noise = ax3.twinx()
        ax3_noise.plot(day_order, day_stats['avg_noise'], color='#10b981', marker='o', linewidth=2.2, label='Mean Noise (dB(A))')
        ax3_noise.bar(day_order, day_stats['loud_flights'], color='#ef4444', alpha=0.4, width=0.25, label='Flights >= 65 dB(A)')
        ax3_noise.set_ylabel('Noise Level (dB(A)) / Exceedance Count', color='#10b981', fontsize=9)
        ax3_noise.tick_params(axis='y', labelcolor='#10b981')
        dl1, dlab1 = ax3.get_legend_handles_labels()
        dl2, dlab2 = ax3_noise.get_legend_handles_labels()
        ax3.legend(dl1 + dl2, dlab1 + dlab2, loc='lower right', fontsize=8, framealpha=0.6)

        # --- PANEL 4: Monthly & Weekly Activity Regularity Timeline ---
        ax4 = fig.add_subplot(gs[1, 1])
        daily_stats = flights.groupby('date').agg(
            flights=('flight_pass_id', 'count'),
            avg_noise=('max_noise', 'mean'),
            max_noise=('max_noise', 'max')
        ).reset_index()

        dates = pd.to_datetime(daily_stats['date'])
        ax4.bar(dates, daily_stats['flights'], color='#3b82f6', alpha=0.7, width=0.8, label='Daily Flights')
        
        months_active = flights.groupby('month_year').agg(days=('date', 'nunique'), fl=('flight_pass_id', 'count'))
        m_str = " | ".join([f"{m}: {d}d ({fl}fl)" for m, d, fl in zip(months_active.index, months_active['days'], months_active['fl'])])
        ax4.set_title(f'4. Activity Regularity: Days in Month / Week Timeline - {m_str}', fontsize=10, fontweight='bold', color='#38bdf8', pad=6)
        ax4.set_ylabel('Flights / Day', color='#3b82f6', fontsize=9)
        ax4.tick_params(axis='y', labelcolor='#3b82f6')
        ax4.tick_params(axis='x', rotation=20)
        ax4.grid(True, alpha=0.15, linestyle='--')

        ax4_noise = ax4.twinx()
        ax4_noise.plot(dates, daily_stats['avg_noise'], color='#f59e0b', linewidth=1.5, label='Daily Avg Noise (dB(A))')
        ax4_noise.plot(dates, daily_stats['max_noise'], color='#ef4444', linestyle=':', alpha=0.7, label='Daily Max Noise (dB(A))')
        ax4_noise.set_ylabel('Ground Noise (dB(A))', color='#f59e0b', fontsize=9)
        ax4_noise.tick_params(axis='y', labelcolor='#f59e0b')
        tl1, tlab1 = ax4.get_legend_handles_labels()
        tl2, tlab2 = ax4_noise.get_legend_handles_labels()
        ax4.legend(tl1 + tl2, tlab1 + tlab2, loc='upper left', fontsize=8, framealpha=0.6)

        # --- PANEL 5: Aircraft Category vs Flights & Ground Noise (NOT exact model) ---
        ax5 = fig.add_subplot(gs[2, 0])
        category_stats = flights.groupby('aircraft_type_group', observed=False).agg(
            flights=('flight_pass_id', 'count'),
            planes=('icao24', 'nunique'),
            avg_noise=('max_noise', 'mean'),
            max_noise=('max_noise', 'max')
        )
        category_stats = category_stats[category_stats['flights'] > 0].sort_values('flights', ascending=False)

        if not category_stats.empty:
            y_pos = np.arange(len(category_stats))
            ax5.barh(y_pos, category_stats['flights'], color='#14b8a6', alpha=0.75, height=0.6, label='Flights')
            ax5.set_yticks(y_pos)
            ax5.set_yticklabels(category_stats.index, fontsize=9)
            ax5.invert_yaxis()
            ax5.set_title(f'5. Flights by Aircraft Category vs Perceived Ground Noise (Total={total_flights:,})', fontsize=11, fontweight='bold', color='#38bdf8', pad=8)
            ax5.set_xlabel('Total Flights', color='#14b8a6', fontsize=9)
            ax5.tick_params(axis='x', labelcolor='#14b8a6')
            ax5.grid(True, alpha=0.15, linestyle='--')
            for i, w in enumerate(category_stats['flights']):
                pct = (w / total_flights * 100.0) if total_flights > 0 else 0.0
                ax5.text(w * 0.02 + 5, i, f"{int(w):,} flights ({pct:.1f}%)", va='center', fontsize=8, color='#042f2e', fontweight='bold')

            ax5_noise = ax5.twiny()
            ax5_noise.plot(category_stats['avg_noise'], y_pos, color='#fbbf24', marker='o', linewidth=2, linestyle='None', label='Avg Noise (dB(A))')
            ax5_noise.set_xlabel('Mean Noise (dB(A))', color='#fbbf24', fontsize=9)
            ax5_noise.tick_params(axis='x', labelcolor='#fbbf24')
            for i, avg_n in enumerate(category_stats['avg_noise']):
                ax5_noise.text(avg_n + 0.6, i, f'{avg_n:.1f} dB', va='center', fontsize=8, color='#fef08a', fontweight='semibold')
        else:
            ax5.text(0.5, 0.5, 'No categories in filter', ha='center', va='center', color='#94a3b8')

        # --- PANEL 6: Cross-Dimensional Matrix (Day of Week x Time of Day Noise Heatmap) ---
        ax6 = fig.add_subplot(gs[2, 1])
        pivot_noise = flights.pivot_table(
            index='day_of_week',
            columns='hour_of_day',
            values='max_noise',
            aggfunc='mean'
        ).reindex(index=day_order, columns=range(24))

        sns.heatmap(
            pivot_noise,
            vmin=30,
            vmax=85,
            cmap='plasma',
            ax=ax6,
            cbar_kws={'label': 'Mean Peak Noise dB(A)', 'ticks': [30, 40, 50, 60, 70, 80, 85]},
            linewidths=0.5,
            linecolor='#1e293b'
        )
        ax6.set_facecolor('#0b0f19')
        ax6.set_title('6. Acoustic Matrix: Day of Week × Hour of Day (30–85+ dB(A) Scale, Max Color ≥85 dB)', fontsize=11, fontweight='bold', color='#38bdf8', pad=8)
        ax6.set_xlabel('Hour of Day (UTC)', fontsize=9)
        ax6.set_ylabel('Day of Week', fontsize=9)

        # --- PANEL 7: Flight Distribution by Ground Noise Level (5 dB steps) ---
        ax7 = fig.add_subplot(gs[3, :])
        x_indices = np.arange(len(bin_labels))
        f_vals = [flight_counts.get(l, 0) for l in bin_labels]
        bars_f = ax7.bar(x_indices, f_vals, width=0.6, color='#0284c7', alpha=0.85, edgecolor='#38bdf8', linewidth=1.2, label=f'Flight Passes (Total={total_flights:,})')
        ax7.set_title(f'7. Flight Pass Distribution by Ground Noise Level (5 dB steps) — Total Count of Flights: {total_flights:,}', fontsize=11, fontweight='bold', color='#38bdf8', pad=8)
        ax7.set_xlabel('Perceived Ground Noise Level (5 dB steps)', fontsize=9)
        ax7.set_ylabel('Number of Flights', color='#38bdf8', fontsize=9)
        ax7.set_xticks(x_indices)
        ax7.set_xticklabels([l.replace(' dB(A)', '') for l in bin_labels], fontsize=8.5)
        ax7.tick_params(axis='y', labelcolor='#38bdf8')
        ax7.grid(True, alpha=0.15, linestyle='--')
        
        max_y = max(f_vals) if f_vals else 1
        for bar in bars_f:
            h = bar.get_height()
            if h > 0:
                pct = (h / total_flights * 100.0) if total_flights > 0 else 0.0
                ax7.text(bar.get_x() + bar.get_width()/2., h + max_y * 0.015, f'{int(h):,} ({pct:.1f}%)', ha='center', va='bottom', fontsize=7.5, color='#cbd5e1')

        for idx_l, lbl in enumerate(bin_labels):
            val_low = int(lbl.split('-')[0])
            if val_low == 65:
                ax7.axvline(idx_l - 0.5, color='#ef4444', linestyle='--', linewidth=1.5, alpha=0.8, label='Urban Noise Threshold (65 dB(A))')
            elif val_low == 75:
                ax7.axvline(idx_l - 0.5, color='#dc2626', linestyle=':', linewidth=1.8, alpha=0.9, label='High Annoyance Threshold (75 dB(A))')

        ax7.legend(loc='upper right', fontsize=8.5, framealpha=0.6)

        plt.subplots_adjust(top=0.95, bottom=0.04, left=0.06, right=0.94, hspace=0.38, wspace=0.28)
        plt.show()
        plt.close(fig)

    _render()

# ==============================================================================
# 🗺️ INTERACTIVE MAP, DRAW CONTROL & FILTER WIDGETS
# ==============================================================================

subregion_map = Map(
    center=[center_lat, center_lon],
    zoom=10,
    basemap=basemaps.OpenTopoMap,
    scroll_wheel_zoom=True,
    layout=Layout(height='380px', width='100%')
)

parent_boundary = Rectangle(
    bounds=((MAIN_LAT_MIN, MAIN_LON_MIN), (MAIN_LAT_MAX, MAIN_LON_MAX)),
    color='#f59e0b',
    dash_array='6, 6',
    weight=2,
    fill=False
)
subregion_map.add_layer(parent_boundary)

draw_control = DrawControl(
    polygon={'shapeOptions': {'color': '#38bdf8', 'fillColor': '#0284c7', 'fillOpacity': 0.35, 'weight': 3}},
    rectangle={'shapeOptions': {'color': '#38bdf8', 'fillColor': '#0284c7', 'fillOpacity': 0.35, 'weight': 3}},
    polyline={}, circle={}, circlemarker={}, marker={}
)
draw_control.data = [coords_to_geojson_feature(active_polygon_coords)]
subregion_map.add_control(draw_control)

subregion_info_html = HTML(value=generate_subregion_html(active_polygon_coords))

# --- FILTER 1: Multi-Select Altitude Buckets ---
alt_tier_selector = SelectMultiple(
    options=ALTITUDE_TIER_OPTIONS,
    value=tuple(ALTITUDE_TIER_OPTIONS),
    description='Altitudes:',
    layout=Layout(width='320px', height='115px')
)
btn_alt_all = Button(description="All", layout=Layout(width='50px', height='26px'))
btn_alt_low = Button(description="<2000m", layout=Layout(width='65px', height='26px'))
btn_alt_high = Button(description="≥2000m", layout=Layout(width='65px', height='26px'))
btn_alt_none = Button(description="Clear", layout=Layout(width='50px', height='26px'))
alt_btn_box = HBox([btn_alt_all, btn_alt_low, btn_alt_high, btn_alt_none], layout=Layout(margin='2px 0'))
alt_filter_col = VBox([Label("🏔️ Altitude Tiers (Multi-Select):"), alt_tier_selector, alt_btn_box], layout=Layout(width='330px'))

# --- FILTER 2: Multi-Select Aircraft Groups ---
group_selector = SelectMultiple(
    options=ALL_AIRCRAFT_GROUPS,
    value=tuple(ALL_AIRCRAFT_GROUPS),
    description='Groups:',
    layout=Layout(width='340px', height='115px')
)
btn_grp_all = Button(description="All", layout=Layout(width='45px', height='26px'))
btn_grp_airliners = Button(description="Airliners", layout=Layout(width='70px', height='26px'))
btn_grp_ga = Button(description="GA", layout=Layout(width='45px', height='26px'))
btn_grp_helis = Button(description="Helis", layout=Layout(width='55px', height='26px'))
btn_grp_none = Button(description="Clear", layout=Layout(width='50px', height='26px'))
grp_btn_box = HBox([btn_grp_all, btn_grp_airliners, btn_grp_ga, btn_grp_helis, btn_grp_none], layout=Layout(margin='2px 0'))
grp_filter_col = VBox([Label("🛫 Operational Groups (Multi-Select):"), group_selector, grp_btn_box], layout=Layout(width='350px'))

# --- FILTER 3: Time Range & Presets ---
use_full_history_chk = Checkbox(value=True, description="Full History", indent=False, layout=Layout(width='110px'))
start_dt_picker = DatetimePicker(value=dataset_min_dt, description="Start UTC:", disabled=True, layout=Layout(width='280px'))
end_dt_picker = DatetimePicker(value=dataset_max_dt, description="End UTC:", disabled=True, layout=Layout(width='280px'))

btn_t_all = Button(description="All", layout=Layout(width='42px', height='26px'))
btn_t_24h = Button(description="24h", layout=Layout(width='42px', height='26px'))
btn_t_12h = Button(description="12h", layout=Layout(width='42px', height='26px'))
btn_t_6h = Button(description="6h", layout=Layout(width='42px', height='26px'))
btn_t_3h = Button(description="3h", layout=Layout(width='42px', height='26px'))
btn_t_1h = Button(description="1h", layout=Layout(width='42px', height='26px'))
time_btn_box = HBox([btn_t_all, btn_t_24h, btn_t_12h, btn_t_6h, btn_t_3h, btn_t_1h], layout=Layout(margin='2px 0'))
time_filter_row = HBox([use_full_history_chk, start_dt_picker, end_dt_picker, time_btn_box], layout=Layout(align_items='center', margin='4px 0'))

# --- FILTER 4: Min Noise & Cascading Airframe Model Dropdown ---
noise_slider = IntSlider(value=30, min=30, max=85, step=5, description="Min Noise (dB):", continuous_update=False, layout=Layout(width='340px'))
model_dropdown = Dropdown(options=['All Specific Models'], value='All Specific Models', description="Model Filter:", layout=Layout(width='320px'))

btn_reset = Button(description="🔄 Reset to Defaults", button_style="warning", icon="undo", layout=Layout(width='180px', height='32px'))
btn_refresh = Button(description="📊 Refresh Stats", button_style="primary", icon="sync", layout=Layout(width='160px', height='32px'))
noise_model_row = HBox([noise_slider, model_dropdown, btn_reset, btn_refresh], layout=Layout(align_items='center', margin='4px 0'))

_is_recomputing = False
_last_params = None

def recompute_dashboard(force: bool = False):
    global _is_recomputing, _last_params
    cur_params = (
        tuple(active_polygon_coords),
        tuple(alt_tier_selector.value),
        bool(use_full_history_chk.value),
        start_dt_picker.value,
        end_dt_picker.value,
        float(noise_slider.value),
        tuple(group_selector.value),
        str(model_dropdown.value)
    )
    if not force and (cur_params == _last_params or _is_recomputing):
        return
    _is_recomputing = True
    try:
        _last_params = cur_params
        clear_output(wait=True)
        display(subregion_map)
        display(subregion_info_html)
        display(filter_panel_box)
        analyze_and_plot_polygon_stats(
            lat_lons=active_polygon_coords,
            selected_altitudes=alt_tier_selector.value,
            use_full_history=use_full_history_chk.value,
            start_dt=start_dt_picker.value,
            end_dt=end_dt_picker.value,
            min_noise=float(noise_slider.value),
            selected_groups=group_selector.value,
            model_filter=model_dropdown.value
        )
    finally:
        _is_recomputing = False

def on_filter_changed(change=None):
    recompute_dashboard()

def on_groups_changed(change=None):
    # Update cascading models
    active_grps = group_selector.value
    available_models = set()
    for g in active_grps:
        for m in group_models_dict.get(g, []):
            available_models.add(m)
    sorted_models = ['All Specific Models'] + sorted(list(available_models))
    cur_val = model_dropdown.value
    model_dropdown.options = sorted_models
    model_dropdown.value = cur_val if cur_val in sorted_models else 'All Specific Models'
    recompute_dashboard()

# Connect Observers for automatic recomputation on change
alt_tier_selector.observe(on_filter_changed, names='value')
noise_slider.observe(on_filter_changed, names='value')
group_selector.observe(on_groups_changed, names='value')
model_dropdown.observe(on_filter_changed, names='value')
use_full_history_chk.observe(lambda c: (
    setattr(start_dt_picker, 'disabled', c['new']),
    setattr(end_dt_picker, 'disabled', c['new']),
    on_filter_changed()
), names='value')
start_dt_picker.observe(on_filter_changed, names='value')
end_dt_picker.observe(on_filter_changed, names='value')

# Quick altitude presets
btn_alt_all.on_click(lambda b: setattr(alt_tier_selector, 'value', tuple(ALTITUDE_TIER_OPTIONS)))
btn_alt_low.on_click(lambda b: setattr(alt_tier_selector, 'value', ("0 - 500m (Red)", "500 - 1000m (Orange)", "1000 - 2000m (Amber)")))
btn_alt_high.on_click(lambda b: setattr(alt_tier_selector, 'value', ("2000 - 3000m (Yellow)", "3000 - 5000m (Teal)", "5000m+ (Green)")))
btn_alt_none.on_click(lambda b: setattr(alt_tier_selector, 'value', ()))

# Quick group presets
btn_grp_all.on_click(lambda b: setattr(group_selector, 'value', tuple(ALL_AIRCRAFT_GROUPS)))
btn_grp_airliners.on_click(lambda b: setattr(group_selector, 'value', ('Commercial Heavy Jet', 'Commercial Medium Jet')))
btn_grp_ga.on_click(lambda b: setattr(group_selector, 'value', ('Light Aircraft / Leisure GA', 'Glider / Sailplane')))
btn_grp_helis.on_click(lambda b: setattr(group_selector, 'value', ('Helicopter / Rotorcraft',)))
btn_grp_none.on_click(lambda b: setattr(group_selector, 'value', ()))

# Quick time presets
def set_time_preset(hours=None):
    if hours is None:
        use_full_history_chk.value = True
    else:
        use_full_history_chk.value = False
        start_dt_picker.disabled = False
        end_dt_picker.disabled = False
        end_dt_picker.value = dataset_max_dt
        start_dt_picker.value = dataset_max_dt - timedelta(hours=hours)

btn_t_all.on_click(lambda b: set_time_preset(None))
btn_t_24h.on_click(lambda b: set_time_preset(24))
btn_t_12h.on_click(lambda b: set_time_preset(12))
btn_t_6h.on_click(lambda b: set_time_preset(6))
btn_t_3h.on_click(lambda b: set_time_preset(3))
btn_t_1h.on_click(lambda b: set_time_preset(1))

# Reset all filters to default
def on_btn_reset_clicked(b):
    global active_polygon_coords
    active_polygon_coords = get_default_center_square()
    draw_control.clear()
    draw_control.data = [coords_to_geojson_feature(active_polygon_coords)]
    subregion_info_html.value = generate_subregion_html(active_polygon_coords)
    alt_tier_selector.value = tuple(ALTITUDE_TIER_OPTIONS)
    group_selector.value = tuple(ALL_AIRCRAFT_GROUPS)
    use_full_history_chk.value = True
    start_dt_picker.value = dataset_min_dt
    end_dt_picker.value = dataset_max_dt
    noise_slider.value = 30
    model_dropdown.value = 'All Specific Models'
    recompute_dashboard(force=True)

btn_reset.on_click(on_btn_reset_clicked)
btn_refresh.on_click(lambda b: recompute_dashboard(force=True))

# Draw control callback
def handle_draw(target, action, geo_json):
    global active_polygon_coords
    if action in ['created', 'edited'] and geo_json.get('geometry', {}).get('type') == 'Polygon':
        raw_coords = geo_json['geometry']['coordinates'][0]
        parsed = [(round(pt[1], 4), round(pt[0], 4)) for pt in raw_coords]
        if len(parsed) > 1 and parsed[0] == parsed[-1]:
            parsed = parsed[:-1]
        if len(parsed) >= 3 and parsed != active_polygon_coords:
            active_polygon_coords = parsed
            subregion_info_html.value = generate_subregion_html(active_polygon_coords)
            recompute_dashboard()

draw_control.on_draw(handle_draw)

# Assemble Filter Panel Box
filter_panel_box = VBox([
    HBox([alt_filter_col, grp_filter_col], layout=Layout(gap='16px', margin='4px 0')),
    time_filter_row,
    noise_model_row,
    HTML("<hr style='border: 1px solid #334155; margin: 8px 0;'>")
], layout=Layout(background='#0f172a', padding='8px 12px', border='1px solid #334155', border_radius='8px', margin='6px 0'))

# Trigger initial execution on center square (renders map, filter panel, stats, and plots)
recompute_dashboard(force=True)

## 8. 🛡️ Privacy-Preserving Public Density Map (Option 1: Raster Density Overlay)

Generates an ultra-lightweight (~1 MB), privacy-preserving raster density heatmap.
- **Zero Raw Point Redistribution**: Aggregates all trajectories into 2D continuous density rasters per altitude tier, fully compliant with OpenSky Network terms regarding bulk raw data distribution.
- **Mobile & Public Web Ready**: Instant rendering with no browser lagging or memory spikes.
- **Parallel Output**: Exported alongside the private interactive point map as .

In [ ]:
from generate_density_heatmap import build_public_density_map

public_html_path = os.path.join(WORKSPACE_DIR, "opensky_flight_density_map.html")
parquet_data_source = parquet_export_path if ('parquet_export_path' in globals() and os.path.exists(parquet_export_path)) else df_processed

# build_public_density_map reads directly from parquet in low-RAM float32 columnar mode
if (isinstance(parquet_data_source, str) and os.path.exists(parquet_data_source)) or not df_processed.empty:
    try:
        public_density_map = build_public_density_map(
            df=parquet_data_source,
            lat_min=LAT_MIN,
            lat_max=LAT_MAX,
            lon_min=LON_MIN,
            lon_max=LON_MAX,
            region_name=REGION_NAME,
            output_html_path=public_html_path
        )
    except Exception as e:
        print(f"⚠️ Direct Parquet read note ({e}). Falling back to in-memory preview dataframe...")
        public_density_map = build_public_density_map(
            df=df_processed,
            lat_min=LAT_MIN,
            lat_max=LAT_MAX,
            lon_min=LON_MIN,
            lon_max=LON_MAX,
            region_name=REGION_NAME,
            output_html_path=public_html_path
        )

## 9. 💾 Export Processed Trajectories, Groups & Acoustic Data

In [ ]:
csv_export_path = os.path.join(WORKSPACE_DIR, "processed_flight_paths.csv")
parquet_export_path = os.path.join(WORKSPACE_DIR, "processed_flight_paths.parquet")

# 1. Parquet dataset is already streamed and complete on disk from Step 4
# 2. Stream Parquet to CSV in chunks to avoid memory spikes
if os.path.exists(parquet_export_path):
    print("💾 Streaming Parquet dataset to CSV in chunks...")
    try:
        parquet_file = pq.ParquetFile(parquet_export_path)
        first_chunk = True
        with open(csv_export_path, "w", newline="", encoding="utf-8") as f_csv:
            for batch in parquet_file.iter_batches(batch_size=50000):
                chunk_df = batch.to_pandas()
                chunk_df.to_csv(f_csv, header=first_chunk, index=False)
                first_chunk = False
                del chunk_df
                gc.collect()
    except Exception as e:
        print(f"⚠️ Chunked CSV streaming note: {e}. Falling back to preview CSV.")
        if not df_processed.empty:
            df_processed.to_csv(csv_export_path, index=False)
elif not df_processed.empty:
    df_processed.to_parquet(parquet_export_path, index=False)
    df_processed.to_csv(csv_export_path, index=False)

total_pts_exported = interp_stats.get('interpolated_points', len(df_processed)) if 'interp_stats' in globals() else len(df_processed)
csv_size_str = f"{os.path.getsize(csv_export_path)/1024/1024:.2f} MB" if os.path.exists(csv_export_path) else "N/A"
parquet_size_str = f"{os.path.getsize(parquet_export_path)/1024/1024:.2f} MB" if os.path.exists(parquet_export_path) else "N/A"

print("=" * 60)
print("🎉 Analysis Pipeline Completed Successfully!")
print(f"  • Processed Points Exported : {total_pts_exported:,}")
print(f"  • CSV Export               : {csv_export_path} ({csv_size_str})")
print(f"  • Parquet Export           : {parquet_export_path} ({parquet_size_str})")
print(f"  • Standalone Interactive HTML (Private) : {html_export_path}")
print(f"  • Standalone Density HTML (Public Safe) : {public_html_path}")
print(f"  • Standalone Time-Lapse HTML : {os.path.join(WORKSPACE_DIR, 'timelapse_flight_heatmap.html')}")
print("=" * 60)